In [1]:
from pathlib import Path
import hashlib
import json

import numpy as np
from IPython.display import display, Markdown
from ai_edge_litert.interpreter import Interpreter, OpResolverType

project_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_DIR = next(
    (
        path for path in project_candidates
        if (
            path / "models/tinystories_3m_blocks_w8a8"
        ).is_dir()
    ),
    None,
)

if PROJECT_DIR is None:
    raise RuntimeError("Start this notebook inside the project directory.")

MODEL_DIR = PROJECT_DIR / (
    "models/tinystories_3m_blocks_w8a8/"
    "20261011T134954617091Z"
)

with (MODEL_DIR / "export_report.json").open() as file:
    export_report = json.load(file)

with np.load(
    MODEL_DIR / "validation_inputs.npz",
    allow_pickle=False,
) as saved:
    validation_inputs = {
        name: saved[name].copy()
        for name in saved.files
    }

print("Project:", PROJECT_DIR)
print("Models:", MODEL_DIR)
print("Saved benchmark block inputs:",
      sum(f"block{i}_input_fp32" in validation_inputs for i in range(8)))
print("Calibration token sequences:",
      validation_inputs["calibration_input_ids"].shape)

Project: /home/randy/Documents/ACompArc/model/project
Models: /home/randy/Documents/ACompArc/model/project/artifacts/models/tinystories_3m_blocks_w8a8/20261011T134954617091Z
Saved benchmark block inputs: 8
Calibration token sequences: (7, 32)


In [2]:
expected_hashes = {
    int(entry["block_index"]): entry["sha256"]
    for entry in export_report["blocks"]
}

block_models = {}

rows = [
    "| Block | Operations | INT8 FCs | Input shape / dtype | Output shape / dtype |",
    "|---:|---:|---:|---|---|",
]

for block_index in range(8):
    model_path = MODEL_DIR / f"block{block_index}_w8a8.tflite"
    digest = hashlib.sha256(model_path.read_bytes()).hexdigest()

    if digest != expected_hashes[block_index]:
        raise ValueError(f"Block {block_index}: model hash mismatch")

    interpreter = Interpreter(
        model_path=str(model_path),
        experimental_preserve_all_tensors=True,
        experimental_op_resolver_type=OpResolverType.BUILTIN_REF,
    )
    interpreter.allocate_tensors()

    tensors = {
        int(detail["index"]): detail
        for detail in interpreter.get_tensor_details()
    }
    ops = interpreter._get_ops_details()
    input_detail = interpreter.get_input_details()[0]
    output_detail = interpreter.get_output_details()[0]

    int8_fc_count = sum(
        op["op_name"] == "FULLY_CONNECTED"
        and tensors[int(op["inputs"][1])]["dtype"] == np.int8
        for op in ops
    )

    block_models[block_index] = {
        "interpreter": interpreter,
        "tensors": tensors,
        "ops": ops,
        "input": input_detail,
        "output": output_detail,
        "sha256": digest,
    }

    input_label = (
        f"{tuple(int(v) for v in input_detail['shape'])} / "
        f"{np.dtype(input_detail['dtype']).name}"
    )
    output_label = (
        f"{tuple(int(v) for v in output_detail['shape'])} / "
        f"{np.dtype(output_detail['dtype']).name}"
    )

    rows.append(
        f"| {block_index} | {len(ops)} | {int8_fc_count} | "
        f"{input_label} | {output_label} |"
    )

display(Markdown("\n".join(rows)))
print("PASS: all eight model hashes match the export report.")

| Block | Operations | INT8 FCs | Input shape / dtype | Output shape / dtype |
|---:|---:|---:|---|---|
| 0 | 74 | 6 | (1, 32, 128) / float32 | (1, 32, 128) / float32 |
| 1 | 74 | 6 | (1, 32, 128) / float32 | (1, 32, 128) / float32 |
| 2 | 74 | 6 | (1, 32, 128) / float32 | (1, 32, 128) / float32 |
| 3 | 74 | 6 | (1, 32, 128) / float32 | (1, 32, 128) / float32 |
| 4 | 74 | 6 | (1, 32, 128) / float32 | (1, 32, 128) / float32 |
| 5 | 74 | 6 | (1, 32, 128) / float32 | (1, 32, 128) / float32 |
| 6 | 74 | 6 | (1, 32, 128) / float32 | (1, 32, 128) / float32 |
| 7 | 74 | 6 | (1, 32, 128) / float32 | (1, 32, 128) / float32 |

PASS: all eight model hashes match the export report.


In [3]:
boundary_encoding_results = {}

rows = [
    "| Block | Boundary | Saved FP32 range | INT8 clipping count | "
    "INT8 max error | INT32 max error |",
    "|---:|---|---|---:|---:|---:|",
]

i8_limits = np.iinfo(np.int8)
i32_limits = np.iinfo(np.int32)

for block_index in range(8):
    for boundary in ("input", "output"):
        name = f"block{block_index}_{boundary}_fp32"
        values = validation_inputs[name].astype(np.float64)

        if values.shape != (1, 32, 128):
            raise ValueError(f"{name}: unexpected shape {values.shape}")
        if not np.all(np.isfinite(values)):
            raise ValueError(f"{name}: non-finite values")

        # Comparison format: INT8, scale 2^-8, zero point 0.
        rounded_i8 = np.rint(values * (1 << 8))
        clipping_count = int(np.count_nonzero(
            (rounded_i8 < i8_limits.min)
            | (rounded_i8 > i8_limits.max)
        ))
        encoded_i8 = np.clip(
            rounded_i8, i8_limits.min, i8_limits.max
        ).astype(np.int8)
        restored_i8 = encoded_i8.astype(np.float64) * (2.0 ** -8)

        # Candidate format: INT32, scale 2^-16, zero point 0.
        # Reject overflow rather than silently clipping.
        rounded_i32 = np.rint(values * (1 << 16))
        if np.any(
            (rounded_i32 < i32_limits.min)
            | (rounded_i32 > i32_limits.max)
        ):
            raise OverflowError(f"{name}: INT32 encoding overflow")

        encoded_i32 = rounded_i32.astype(np.int32)
        restored_i32 = encoded_i32.astype(np.float64) * (2.0 ** -16)

        i8_error = float(np.max(np.abs(restored_i8 - values)))
        i32_error = float(np.max(np.abs(restored_i32 - values)))

        boundary_encoding_results[(block_index, boundary)] = {
            "int8_clipping_count": clipping_count,
            "int8_max_abs_error": i8_error,
            "int32_max_abs_error": i32_error,
            "int32_values": encoded_i32,
        }

        rows.append(
            f"| {block_index} | {boundary} | "
            f"[{values.min():.6f}, {values.max():.6f}] | "
            f"{clipping_count} | {i8_error:.9g} | {i32_error:.9g} |"
        )

display(Markdown("\n".join(rows)))
print("PASS: all saved benchmark boundaries fit the candidate INT32 format.")

| Block | Boundary | Saved FP32 range | INT8 clipping count | INT8 max error | INT32 max error |
|---:|---|---|---:|---:|---:|
| 0 | input | [-0.328789, 0.307286] | 0 | 0.00195239484 | 7.62939453e-06 |
| 0 | output | [-0.648160, 0.535610] | 2 | 0.148160279 | 7.62939453e-06 |
| 1 | input | [-0.648160, 0.535610] | 2 | 0.148160279 | 7.62939453e-06 |
| 1 | output | [-0.389435, 0.389251] | 0 | 0.00195299089 | 7.62939453e-06 |
| 2 | input | [-0.389435, 0.389251] | 0 | 0.00195299089 | 7.62939453e-06 |
| 2 | output | [-0.317598, 0.348882] | 0 | 0.00195207447 | 7.62939453e-06 |
| 3 | input | [-0.317598, 0.348882] | 0 | 0.00195207447 | 7.62939453e-06 |
| 3 | output | [-0.284165, 0.264227] | 0 | 0.00195004605 | 7.62939453e-06 |
| 4 | input | [-0.284165, 0.264227] | 0 | 0.00195004605 | 7.62939453e-06 |
| 4 | output | [-0.274872, 0.279391] | 0 | 0.00195264816 | 7.62939453e-06 |
| 5 | input | [-0.274872, 0.279391] | 0 | 0.00195264816 | 7.62939453e-06 |
| 5 | output | [-0.257529, 0.287483] | 0 | 0.00195297599 | 7.62939453e-06 |
| 6 | input | [-0.257529, 0.287483] | 0 | 0.00195297599 | 7.62939453e-06 |
| 6 | output | [-0.229009, 0.389007] | 0 | 0.00195303559 | 7.6238066e-06 |
| 7 | input | [-0.229009, 0.389007] | 0 | 0.00195303559 | 7.6238066e-06 |
| 7 | output | [-0.490246, 0.242620] | 0 | 0.00195285678 | 7.62939453e-06 |

PASS: all saved benchmark boundaries fit the candidate INT32 format.


In [4]:
import torch
from transformers import AutoModelForCausalLM

FP32_DIR = PROJECT_DIR / "models/tinystories_3m_fp32"

reference_model = AutoModelForCausalLM.from_pretrained(
    FP32_DIR,
    local_files_only=True,
    dtype=torch.float32,
).cpu().eval()

reference_model.set_attn_implementation("eager")

assert len(reference_model.transformer.h) == 8

print("Checkpoint:", FP32_DIR)
print("Blocks:", len(reference_model.transformer.h))
print("Parameter dtype:", next(reference_model.parameters()).dtype)

/home/randy/Documents/ACompArc/model/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1024 10:34:39.681000 9224 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1024 10:34:39.714000 9224 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 108/108 [00:00<00:00, 9794.27it/s]


Checkpoint: /home/randy/Documents/ACompArc/model/project/artifacts/models/tinystories_3m_fp32
Blocks: 8
Parameter dtype: torch.float32


In [5]:
def collect_fp32_boundaries(input_ids):
    ids = torch.as_tensor(
        np.asarray(input_ids).copy(),
        dtype=torch.long,
    ).reshape(1, 32)

    captured = {}
    handles = []

    def make_hook(block_index):
        def hook(module, args, output):
            hidden_output = (
                output[0]
                if isinstance(output, (tuple, list))
                else output
            )

            captured[block_index] = {
                "input_fp32": args[0].detach().cpu().numpy().copy(),
                "output_fp32": hidden_output.detach().cpu().numpy().copy(),
            }

        return hook

    try:
        for block_index, block in enumerate(reference_model.transformer.h):
            handles.append(
                block.register_forward_hook(make_hook(block_index))
            )

        with torch.no_grad():
            reference_model.transformer(
                input_ids=ids,
                use_cache=False,
                return_dict=True,
            )
    finally:
        for handle in handles:
            handle.remove()

    if len(captured) != 8:
        raise RuntimeError("Did not capture all eight blocks.")

    for block_index in range(8):
        for boundary in ("input", "output"):
            values = captured[block_index][f"{boundary}_fp32"]
            if values.shape != (1, 32, 128):
                raise ValueError("Unexpected block boundary shape.")
            if not np.all(np.isfinite(values)):
                raise ValueError("Non-finite reference activation.")

    return [captured[i] for i in range(8)]


benchmark_blocks = collect_fp32_boundaries(
    validation_inputs["benchmark_input_ids"]
)

# Verify the reconstruction before collecting calibration cases.
for block_index in range(8):
    for boundary in ("input", "output"):
        np.testing.assert_allclose(
            benchmark_blocks[block_index][f"{boundary}_fp32"],
            validation_inputs[f"block{block_index}_{boundary}_fp32"],
            atol=1e-6,
            rtol=1e-6,
            err_msg=f"Block {block_index} {boundary} benchmark mismatch",
        )

print("PASS: reconstructed benchmark matches all saved block boundaries.")

reference_cases = [{
    "name": "benchmark",
    "blocks": benchmark_blocks,
}]

for case_index, ids in enumerate(
    validation_inputs["calibration_input_ids"]
):
    reference_cases.append({
        "name": f"calibration_{case_index}",
        "blocks": collect_fp32_boundaries(ids),
    })

print("Reference cases collected:", len(reference_cases))
print("Values per block boundary:",
      len(reference_cases) * 32 * 128)

PASS: reconstructed benchmark matches all saved block boundaries.
Reference cases collected: 8
Values per block boundary: 32768


In [6]:
rows = [
    "| Block | Boundary | FP32 range across 8 cases | "
    "INT8 clipping count | INT32 stored range | INT32 max encoding error |",
    "|---:|---|---|---:|---|---:|",
]

boundary_range_results = {}
i32_limits = np.iinfo(np.int32)

for block_index in range(8):
    for boundary in ("input", "output"):
        values = np.concatenate([
            case["blocks"][block_index][f"{boundary}_fp32"].reshape(-1)
            for case in reference_cases
        ]).astype(np.float64)

        rounded_i8 = np.rint(values * (1 << 8))
        i8_clipping_count = int(np.count_nonzero(
            (rounded_i8 < -128) | (rounded_i8 > 127)
        ))

        rounded_i32 = np.rint(values * (1 << 16))
        if np.any(
            (rounded_i32 < i32_limits.min)
            | (rounded_i32 > i32_limits.max)
        ):
            raise OverflowError(
                f"Block {block_index} {boundary}: INT32 encoding overflow"
            )

        encoded = rounded_i32.astype(np.int32)
        represented = encoded.astype(np.float64) * (2.0 ** -16)
        max_error = float(np.max(np.abs(represented - values)))

        boundary_range_results[(block_index, boundary)] = {
            "minimum": float(values.min()),
            "maximum": float(values.max()),
            "int8_clipping_count": i8_clipping_count,
            "int32_minimum": int(encoded.min()),
            "int32_maximum": int(encoded.max()),
            "int32_max_encoding_error": max_error,
        }

        rows.append(
            f"| {block_index} | {boundary} | "
            f"[{values.min():.6f}, {values.max():.6f}] | "
            f"{i8_clipping_count} | "
            f"[{encoded.min()}, {encoded.max()}] | "
            f"{max_error:.9g} |"
        )

display(Markdown("\n".join(rows)))
print("PASS: all eight reference cases fit the candidate boundary format.")

| Block | Boundary | FP32 range across 8 cases | INT8 clipping count | INT32 stored range | INT32 max encoding error |
|---:|---|---|---:|---|---:|
| 0 | input | [-0.469448, 0.455548] | 0 | [-30766, 29855] | 7.62939453e-06 |
| 0 | output | [-0.648160, 0.603830] | 14 | [-42478, 39573] | 7.62939453e-06 |
| 1 | input | [-0.648160, 0.603830] | 14 | [-42478, 39573] | 7.62939453e-06 |
| 1 | output | [-0.431264, 0.392886] | 0 | [-28263, 25748] | 7.62939453e-06 |
| 2 | input | [-0.431264, 0.392886] | 0 | [-28263, 25748] | 7.62939453e-06 |
| 2 | output | [-0.391948, 0.381730] | 0 | [-25687, 25017] | 7.62939453e-06 |
| 3 | input | [-0.391948, 0.381730] | 0 | [-25687, 25017] | 7.62939453e-06 |
| 3 | output | [-0.308395, 0.378448] | 0 | [-20211, 24802] | 7.62939453e-06 |
| 4 | input | [-0.308395, 0.378448] | 0 | [-20211, 24802] | 7.62939453e-06 |
| 4 | output | [-0.329801, 0.327732] | 0 | [-21614, 21478] | 7.62939453e-06 |
| 5 | input | [-0.329801, 0.327732] | 0 | [-21614, 21478] | 7.62939453e-06 |
| 5 | output | [-0.318439, 0.293674] | 0 | [-20869, 19246] | 7.62939453e-06 |
| 6 | input | [-0.318439, 0.293674] | 0 | [-20869, 19246] | 7.62939453e-06 |
| 6 | output | [-0.339694, 0.407415] | 0 | [-22262, 26700] | 7.62939453e-06 |
| 7 | input | [-0.339694, 0.407415] | 0 | [-22262, 26700] | 7.62939453e-06 |
| 7 | output | [-0.560879, 0.308717] | 3 | [-36758, 20232] | 7.62939453e-06 |

PASS: all eight reference cases fit the candidate boundary format.


In [7]:
calibration_cases = [
    case for case in reference_cases
    if case["name"].startswith("calibration_")
]
benchmark_cases = [
    case for case in reference_cases
    if case["name"] == "benchmark"
]

assert len(calibration_cases) == 7
assert len(benchmark_cases) == 1


def boundary_values(cases, edge):
    # Nine boundaries:
    # edge 0 = block-0 input
    # edge 1 = block-0 output / block-1 input
    # ...
    # edge 8 = block-7 output
    if edge == 0:
        block_index, key = 0, "input_fp32"
    else:
        block_index, key = edge - 1, "output_fp32"

    return np.concatenate([
        case["blocks"][block_index][key].reshape(-1)
        for case in cases
    ]).astype(np.float64)


def inspect_int8_encoding(values, scale):
    rounded = np.rint(values / scale)
    clipping_count = int(np.count_nonzero(
        (rounded < -128) | (rounded > 127)
    ))

    encoded = np.clip(rounded, -128, 127).astype(np.int8)
    represented = encoded.astype(np.float64) * scale
    errors = represented - values

    return {
        "clipping_count": clipping_count,
        "max_abs_error": float(np.max(np.abs(errors))),
        "rmse": float(np.sqrt(np.mean(errors * errors))),
    }


int8_boundary_specs = []

rows = [
    "| Boundary | Scale | Representable range | "
    "Calibration clips | Benchmark clips | Max encoding error: all cases |",
    "|---|---|---|---:|---:|---:|",
]

for edge in range(9):
    calibration = boundary_values(calibration_cases, edge)
    benchmark = boundary_values(benchmark_cases, edge)

    # Symmetric choice: reserve codes -127..127 for range selection.
    # Round the required scale upward to a power of two.
    maximum_magnitude = float(np.max(np.abs(calibration)))
    exponent = (
        int(np.ceil(np.log2(maximum_magnitude / 127.0)))
        if maximum_magnitude > 0
        else -8
    )
    scale = float(2.0 ** exponent)

    calibration_result = inspect_int8_encoding(calibration, scale)
    benchmark_result = inspect_int8_encoding(benchmark, scale)

    int8_boundary_specs.append({
        "edge": edge,
        "dtype": "int8",
        "scale": scale,
        "scale_exponent": exponent,
        "zero_point": 0,
        "calibration": calibration_result,
        "benchmark": benchmark_result,
    })

    label = "Block 0 input" if edge == 0 else f"Block {edge - 1} output"
    maximum_error = max(
        calibration_result["max_abs_error"],
        benchmark_result["max_abs_error"],
    )

    rows.append(
        f"| {label} | 2^({exponent}) | "
        f"[{-128 * scale:.6f}, {127 * scale:.6f}] | "
        f"{calibration_result['clipping_count']} | "
        f"{benchmark_result['clipping_count']} | "
        f"{maximum_error:.9g} |"
    )

display(Markdown("\n".join(rows)))

# Block i consumes edge i and produces edge i+1.
block_boundary_specs = [
    {
        "input": int8_boundary_specs[i],
        "output": int8_boundary_specs[i + 1],
    }
    for i in range(8)
]

print("Prepared candidate INT8 interfaces for all eight blocks.")

| Boundary | Scale | Representable range | Calibration clips | Benchmark clips | Max encoding error: all cases |
|---|---|---|---:|---:|---:|
| Block 0 input | 2^(-8) | [-0.500000, 0.496094] | 0 | 0 | 0.00195302069 |
| Block 0 output | 2^(-7) | [-1.000000, 0.992188] | 0 | 0 | 0.00390617549 |
| Block 1 output | 2^(-8) | [-0.500000, 0.496094] | 0 | 0 | 0.00195299089 |
| Block 2 output | 2^(-8) | [-0.500000, 0.496094] | 0 | 0 | 0.00195300579 |
| Block 3 output | 2^(-8) | [-0.500000, 0.496094] | 0 | 0 | 0.00195309892 |
| Block 4 output | 2^(-8) | [-0.500000, 0.496094] | 0 | 0 | 0.00195311755 |
| Block 5 output | 2^(-8) | [-0.500000, 0.496094] | 0 | 0 | 0.00195308775 |
| Block 6 output | 2^(-8) | [-0.500000, 0.496094] | 0 | 0 | 0.0019531101 |
| Block 7 output | 2^(-7) | [-1.000000, 0.992188] | 0 | 0 | 0.00390617549 |

Prepared candidate INT8 interfaces for all eight blocks.


In [8]:
def run_tflite_block(block_index, values):
    model = block_models[block_index]
    interpreter = model["interpreter"]

    values = np.ascontiguousarray(values, dtype=np.float32)
    expected_shape = tuple(int(v) for v in model["input"]["shape"])

    if values.shape != expected_shape:
        raise ValueError(f"Block {block_index}: unexpected input shape")

    interpreter.set_tensor(int(model["input"]["index"]), values)
    interpreter.invoke()
    output = interpreter.get_tensor(
        int(model["output"]["index"])
    ).copy()

    if not np.all(np.isfinite(output)):
        raise ValueError(f"Block {block_index}: non-finite output")

    return output


def encode_boundary(values, edge):
    spec = int8_boundary_specs[edge]
    scale = spec["scale"]
    zero_point = spec["zero_point"]

    # Same nearest-even encoding rule used in the boundary inspection.
    rounded = np.rint(
        values.astype(np.float64) / scale
    ) + zero_point

    clipping_count = int(np.count_nonzero(
        (rounded < -128) | (rounded > 127)
    ))

    codes = np.clip(rounded, -128, 127).astype(np.int8)
    represented = (
        (codes.astype(np.float64) - zero_point) * scale
    ).astype(np.float32)

    return codes, represented, clipping_count


boundary_chain_results = {}

for case in reference_cases:
    initial_input = case["blocks"][0]["input_fp32"].copy()

    direct_input = initial_input.copy()
    _, int8_input, initial_clips = encode_boundary(initial_input, 0)

    result = {
        "direct_outputs": [],
        "int8_outputs": [],
        "int8_codes": [],
        "output_clips": [],
        "initial_input_clips": initial_clips,
    }

    for block_index in range(8):
        direct_output = run_tflite_block(block_index, direct_input)

        int8_raw_output = run_tflite_block(block_index, int8_input)
        codes, int8_output, clips = encode_boundary(
            int8_raw_output, block_index + 1
        )

        result["direct_outputs"].append(direct_output.copy())
        result["int8_outputs"].append(int8_output.copy())
        result["int8_codes"].append(codes.copy())
        result["output_clips"].append(clips)

        # Feed each path's actual output into its next block.
        direct_input = direct_output
        int8_input = int8_output

    boundary_chain_results[case["name"]] = result
    print(f"Completed both chains: {case['name']}", flush=True)

Completed both chains: benchmark
Completed both chains: calibration_0
Completed both chains: calibration_1
Completed both chains: calibration_2
Completed both chains: calibration_3
Completed both chains: calibration_4
Completed both chains: calibration_5
Completed both chains: calibration_6


In [9]:
def error_metrics(candidate, reference):
    candidate = np.asarray(candidate, dtype=np.float64).reshape(-1)
    reference = np.asarray(reference, dtype=np.float64).reshape(-1)
    difference = candidate - reference

    reference_norm = np.linalg.norm(reference)
    error_norm = np.linalg.norm(difference)
    relative_l2 = (
        100.0 * error_norm / reference_norm
        if reference_norm > 0
        else (0.0 if error_norm == 0 else float("inf"))
    )

    return {
        "max_abs": float(np.max(np.abs(difference))),
        "rmse": float(np.sqrt(np.mean(difference * difference))),
        "relative_l2_percent": float(relative_l2),
    }


chain_error_metrics = {}

rows = [
    "| Cases | After block | Direct TFLite RMSE vs FP32 | "
    "Direct relative L2 | INT8-boundary RMSE vs FP32 | "
    "INT8-boundary relative L2 | INT8 vs direct RMSE | Output clips |",
    "|---|---:|---:|---:|---:|---:|---:|---:|",
]

groups = [
    ("Benchmark", benchmark_cases),
    ("Calibration", calibration_cases),
]

for group_name, cases in groups:
    for block_index in range(8):
        reference = np.concatenate([
            case["blocks"][block_index]["output_fp32"].reshape(-1)
            for case in cases
        ])

        direct = np.concatenate([
            boundary_chain_results[case["name"]]
            ["direct_outputs"][block_index].reshape(-1)
            for case in cases
        ])

        int8_boundary = np.concatenate([
            boundary_chain_results[case["name"]]
            ["int8_outputs"][block_index].reshape(-1)
            for case in cases
        ])

        direct_metrics = error_metrics(direct, reference)
        int8_metrics = error_metrics(int8_boundary, reference)
        interface_metrics = error_metrics(int8_boundary, direct)

        clips = sum(
            boundary_chain_results[case["name"]]
            ["output_clips"][block_index]
            for case in cases
        )

        chain_error_metrics[(group_name, block_index)] = {
            "direct_vs_fp32": direct_metrics,
            "int8_vs_fp32": int8_metrics,
            "int8_vs_direct": interface_metrics,
            "output_clips": clips,
        }

        rows.append(
            f"| {group_name} | {block_index} | "
            f"{direct_metrics['rmse']:.6g} | "
            f"{direct_metrics['relative_l2_percent']:.4f}% | "
            f"{int8_metrics['rmse']:.6g} | "
            f"{int8_metrics['relative_l2_percent']:.4f}% | "
            f"{interface_metrics['rmse']:.6g} | {clips} |"
        )

display(Markdown("\n".join(rows)))

print(
    "Initial INT8 input clipping:",
    sum(
        result["initial_input_clips"]
        for result in boundary_chain_results.values()
    ),
)

| Cases | After block | Direct TFLite RMSE vs FP32 | Direct relative L2 | INT8-boundary RMSE vs FP32 | INT8-boundary relative L2 | INT8 vs direct RMSE | Output clips |
|---|---:|---:|---:|---:|---:|---:|---:|
| Benchmark | 0 | 0.0018817 | 1.5867% | 0.00309522 | 2.6100% | 0.00320082 | 0 |
| Benchmark | 1 | 0.00484003 | 4.9660% | 0.00484735 | 4.9735% | 0.00592527 | 0 |
| Benchmark | 2 | 0.00474118 | 5.6571% | 0.00521582 | 6.2235% | 0.00631481 | 0 |
| Benchmark | 3 | 0.00488948 | 6.6737% | 0.00547009 | 7.4662% | 0.00658619 | 0 |
| Benchmark | 4 | 0.00545543 | 7.4423% | 0.00621659 | 8.4806% | 0.00739592 | 0 |
| Benchmark | 5 | 0.00569174 | 8.2660% | 0.00641918 | 9.3225% | 0.00751269 | 0 |
| Benchmark | 6 | 0.00577036 | 8.0716% | 0.00656026 | 9.1765% | 0.00753 | 0 |
| Benchmark | 7 | 0.00900613 | 12.5281% | 0.0106861 | 14.8650% | 0.0122519 | 0 |
| Calibration | 0 | 0.00182024 | 1.5328% | 0.00304631 | 2.5652% | 0.0031935 | 0 |
| Calibration | 1 | 0.00467384 | 4.8188% | 0.00577443 | 5.9535% | 0.00629923 | 0 |
| Calibration | 2 | 0.00476134 | 5.6921% | 0.00599324 | 7.1648% | 0.00658651 | 0 |
| Calibration | 3 | 0.00485756 | 6.6239% | 0.00616973 | 8.4132% | 0.006818 | 0 |
| Calibration | 4 | 0.00546803 | 7.3664% | 0.00699334 | 9.4212% | 0.00768934 | 0 |
| Calibration | 5 | 0.005611 | 8.0631% | 0.00712773 | 10.2426% | 0.0078394 | 0 |
| Calibration | 6 | 0.00570392 | 7.7805% | 0.00721002 | 9.8350% | 0.00791179 | 0 |
| Calibration | 7 | 0.00888746 | 11.4409% | 0.01104 | 14.2119% | 0.0120893 | 0 |

Initial INT8 input clipping: 0


In [10]:
def predict_from_final_hidden(hidden_states):
    hidden = torch.from_numpy(
        np.ascontiguousarray(hidden_states, dtype=np.float32)
    )

    with torch.no_grad():
        normalized = reference_model.transformer.ln_f(hidden)
        logits = reference_model.get_output_embeddings()(normalized)
        predictions = logits.argmax(dim=-1)

    return predictions.cpu().numpy().reshape(-1)


prediction_results = {}

rows = [
    "| Case | Direct matches FP32 / 32 | INT8 matches FP32 / 32 | "
    "Final-position IDs: FP32 / direct / INT8 |",
    "|---|---:|---:|---|",
]

for case in reference_cases:
    result = boundary_chain_results[case["name"]]

    fp32_predictions = predict_from_final_hidden(
        case["blocks"][7]["output_fp32"]
    )
    direct_predictions = predict_from_final_hidden(
        result["direct_outputs"][7]
    )
    int8_predictions = predict_from_final_hidden(
        result["int8_outputs"][7]
    )

    direct_matches = int(np.count_nonzero(
        direct_predictions == fp32_predictions
    ))
    int8_matches = int(np.count_nonzero(
        int8_predictions == fp32_predictions
    ))

    prediction_results[case["name"]] = {
        "fp32": fp32_predictions,
        "direct": direct_predictions,
        "int8": int8_predictions,
    }

    rows.append(
        f"| {case['name']} | {direct_matches}/32 | "
        f"{int8_matches}/32 | "
        f"{fp32_predictions[-1]} / "
        f"{direct_predictions[-1]} / "
        f"{int8_predictions[-1]} |"
    )

display(Markdown("\n".join(rows)))

| Case | Direct matches FP32 / 32 | INT8 matches FP32 / 32 | Final-position IDs: FP32 / direct / INT8 |
|---|---:|---:|---|
| benchmark | 30/32 | 32/32 | 1965 / 1965 / 1965 |
| calibration_0 | 30/32 | 31/32 | 13 / 13 / 13 |
| calibration_1 | 30/32 | 28/32 | 2933 / 2933 / 2933 |
| calibration_2 | 30/32 | 30/32 | 287 / 287 / 287 |
| calibration_3 | 27/32 | 25/32 | 673 / 262 / 262 |
| calibration_4 | 28/32 | 29/32 | 290 / 290 / 290 |
| calibration_5 | 32/32 | 28/32 | 351 / 351 / 351 |
| calibration_6 | 24/32 | 27/32 | 198 / 198 / 198 |

# TEST STANDARDIZED BLOCK DEFINITION

## Part 1 — Integer LN1, shared across blocks 0–7

### Input, output, and parameters

- Input: INT8 `[1,32,128]`, scale `2^-F`, zero point `0`.
- Output: INT8 `[1,32,128]`, using that block’s saved TFLite LN1 output scale and zero point.
- Gamma and beta: encoded offline as INT32 at scale `2^-16`.
- Intermediate computations: checked INT32/INT64 arithmetic.
- Feature count: `N = 128`.

| Blocks | F | Square/normalize shift | Variance scale | Integer epsilon | RSQRT numerator |
|---|---:|---:|---|---:|---|
| 0, 2–7 | 8 | 15 | `2^-22` | 42 | `2^54` |
| 1 | 7 | 14 | `2^-21` | 21 | `2^53` |

The algorithm is identical across blocks; parameters depend on the input scale and saved model metadata.

### Rounding rule

For centered integer `a`:

$$
RQ(a,M,r)=(aM+2^{r-1})\gg r
$$

The shift is arithmetic. Runtime rounding is nearest, with ties toward positive infinity, matching XREQUANT.

Offline parameter encoding uses nearest rounding with ties to even.

### 1. Mean and centered differences

For each token:

$$
m=\sum_{k=0}^{127}q_x[k]
$$

Store `m` as INT32 at scale `2^-(F+7)`. The feature sum represents the mean through this scale; no division is required.

Align the input before subtraction:

$$
d[k]=128q_x[k]-m
$$

Compute `d` using INT64 at scale `2^-(F+7)`.

### 2. Square and variance

Compute:

$$
u[k]=d[k]^2
$$

`u` uses INT64 at scale `2^-2(F+7)`.

**Requantization 1 — after squaring:**

$$
v[k]=RQ(u[k],1,F+7)
$$

Store `v` as INT32 at scale `2^-(F+7)`.

Sum the features:

$$
V=\sum_k v[k]
$$

Store `V` as INT32 at scale `2^-(F+14)`, representing the variance.

Encode epsilon at that same scale:

$$
\epsilon_q=\operatorname{round}_{even}(\epsilon\,2^{F+14})
$$

### 3. Inverse standard deviation

$$
I=\operatorname{isqrt}
\left(
\left\lfloor\frac{2^{F+46}}{V+\epsilon_q}\right\rfloor
\right)
$$

`isqrt` returns the floor integer square root.

Store `I` as INT32 at scale `2^-16`. The denominator must be positive.

### 4. Normalize

Compute the INT64 product:

$$
n_{\mathrm{fine}}[k]=d[k]I
$$

Its scale is `2^-(F+23)`.

**Requantization 2 — after normalization:**

$$
n[k]=RQ(n_{\mathrm{fine}}[k],1,F+7)
$$

Store `n` as INT32 at scale `2^-16`.

### 5. Apply gamma and beta

Offline parameter encoding:

$$
\gamma_q=\operatorname{round}_{even}(\gamma\,2^{16}),
\qquad
\beta_q=\operatorname{round}_{even}(\beta\,2^{16})
$$

Runtime affine calculation:

$$
a[k]=n[k]\gamma_q[k]+\beta_q[k]\,2^{16}
$$

Store `a` as INT64 at scale `2^-32`.

Beta alignment is exact; it does not require rounded requantization.

### 6. Convert into the projection input format

Let `s_out` and `z_out` be this block’s saved TFLite LN1 output scale and zero point.

Prepare the multiplier offline:

$$
M=\operatorname{round}_{even}
\left(
\frac{2^{-32}}{s_{\mathrm{out}}}\,2^{47}
\right)
$$

**Requantization 3 — after the affine calculation:**

$$
q_{\mathrm{out}}[k]
=
\operatorname{clip}_{[-128,127]}
\left(RQ(a[k],M,47)+z_{\mathrm{out}}\right)
$$

This INT8 output feeds all three Q/K/V projections.

### Validation

- Check every intended INT32/INT64 result before storing it.
- Compare integer LN1 with TFLite LN1 using the same represented INT8 input.
- Current tests: eight cases per block; maximum output difference is one INT8 code.
- This is local LN1 validation, not full-chain inference validation.

In [11]:
import math


def checked_integer(values, dtype, label):
    values = np.asarray(values, dtype=object)
    limits = np.iinfo(dtype)

    if int(values.min()) < limits.min or int(values.max()) > limits.max:
        raise OverflowError(
            f"{label}: outside {np.dtype(dtype).name}"
        )

    return values.astype(dtype)


def rq_checked(values, multiplier, shift, label):
    """Match XREQUANT: nearest rounding, ties toward +infinity."""
    if not 1 <= shift <= 63:
        raise ValueError(f"{label}: unsupported shift")

    product = checked_integer(
        np.asarray(values, dtype=object) * int(multiplier),
        np.int64,
        label + " product",
    )
    rounded = checked_integer(
        product.astype(object) + (1 << (shift - 1)),
        np.int64,
        label + " rounding addition",
    )

    return rounded >> shift

In [12]:
ln1_params = {}

rows = [
    "| Block | Input scale | Square / normalize shift | "
    "Integer epsilon | RSQRT numerator | Output multiplier / shift | Output ZP |",
    "|---:|---|---:|---:|---|---|---:|",
]

for block_index in range(8):
    model = block_models[block_index]
    interpreter = model["interpreter"]
    ops = model["ops"]

    boundary = block_boundary_specs[block_index]["input"]
    assert boundary["zero_point"] == 0
    input_frac_bits = -int(boundary["scale_exponent"])
    assert input_frac_bits in (7, 8)

    gamma_id = int(ops[12]["inputs"][1])
    beta_id = int(ops[13]["inputs"][1])
    epsilon_id = int(ops[4]["inputs"][1])
    output_id = int(ops[14]["outputs"][0])

    gamma = interpreter.get_tensor(gamma_id).astype(np.float64).reshape(128)
    beta = interpreter.get_tensor(beta_id).astype(np.float64).reshape(128)
    epsilon = float(interpreter.get_tensor(epsilon_id).item())

    if not (
        np.all(np.isfinite(gamma))
        and np.all(np.isfinite(beta))
        and math.isfinite(epsilon)
        and epsilon > 0
    ):
        raise ValueError("Invalid LN1 constants")

    quant = model["tensors"][output_id]["quantization_parameters"]
    assert len(quant["scales"]) == len(quant["zero_points"]) == 1

    output_scale = float(quant["scales"][0])
    output_zp = int(quant["zero_points"][0])
    assert math.isfinite(output_scale) and output_scale > 0

    mean_frac_bits = input_frac_bits + 7
    variance_frac_bits = input_frac_bits + 14
    numerator_exponent = variance_frac_bits + 32

    params = {
        "input_frac_bits": input_frac_bits,
        "square_shift": mean_frac_bits,
        "normalize_shift": mean_frac_bits,
        "epsilon_q": int(np.rint(epsilon * (1 << variance_frac_bits))),
        "rsqrt_numerator": 1 << numerator_exponent,
        "gamma": checked_integer(
            np.rint(gamma * (1 << 16)), np.int32, "gamma"
        ),
        "beta": checked_integer(
            np.rint(beta * (1 << 16)), np.int32, "beta"
        ),
        "output_scale": output_scale,
        "output_zero_point": output_zp,
        "output_multiplier": int(np.rint(
            (2.0 ** -32 / output_scale) * (1 << 47)
        )),
        "output_shift": 47,
        "output_tensor": output_id,
    }

    assert 0 < params["epsilon_q"] <= np.iinfo(np.int32).max
    assert 0 < params["output_multiplier"] <= np.iinfo(np.int64).max

    ln1_params[block_index] = params

    rows.append(
        f"| {block_index} | 2^(-{input_frac_bits}) | "
        f"{mean_frac_bits} | {params['epsilon_q']} | "
        f"2^{numerator_exponent} | "
        f"{params['output_multiplier']} / 47 | {output_zp} |"
    )

display(Markdown("\n".join(rows)))

| Block | Input scale | Square / normalize shift | Integer epsilon | RSQRT numerator | Output multiplier / shift | Output ZP |
|---:|---|---:|---:|---|---|---:|
| 0 | 2^(-8) | 15 | 42 | 2^54 | 974600 / 47 | -17 |
| 1 | 2^(-7) | 14 | 21 | 2^53 | 743369 / 47 | 0 |
| 2 | 2^(-8) | 15 | 42 | 2^54 | 741987 / 47 | 23 |
| 3 | 2^(-8) | 15 | 42 | 2^54 | 789064 / 47 | 6 |
| 4 | 2^(-8) | 15 | 42 | 2^54 | 875470 / 47 | 10 |
| 5 | 2^(-8) | 15 | 42 | 2^54 | 890697 / 47 | -2 |
| 6 | 2^(-8) | 15 | 42 | 2^54 | 943455 / 47 | -5 |
| 7 | 2^(-8) | 15 | 42 | 2^54 | 763419 / 47 | -3 |

In [34]:
def integer_layernorm(input_codes, params):
    if (
        input_codes.dtype not in (np.dtype(np.int8), np.dtype(np.int32))
        or input_codes.shape != (1, 32, 128)
    ):
        raise ValueError("LayerNorm expects INT8 or INT32 [1,32,128]")

    q = input_codes.astype(object)

    # Mean represented by the feature sum.
    mean = checked_integer(
        q.sum(axis=-1, keepdims=True), np.int32, "mean"
    )

    difference = checked_integer(
        128 * q - mean.astype(object), np.int64, "difference"
    )
    square_fine = checked_integer(
        difference.astype(object) ** 2, np.int64, "square"
    )

    # Requantization 1: squared differences -> stored INT32.
    square = checked_integer(
        rq_checked(
            square_fine, 1, params["square_shift"], "square conversion"
        ),
        np.int32, "stored square",
    )

    variance = checked_integer(
        square.astype(object).sum(axis=-1, keepdims=True),
        np.int32, "variance",
    )
    variance_eps = checked_integer(
        variance.astype(object) + params["epsilon_q"],
        np.int32, "variance plus epsilon",
    )

    if np.any(variance_eps <= 0):
        raise ValueError("Nonpositive reciprocal-square-root input")

    inv_std = checked_integer(
        [
            math.isqrt(params["rsqrt_numerator"] // int(v))
            for v in variance_eps.flat
        ],
        np.int32, "inverse standard deviation",
    ).reshape(1, 32, 1)

    normalized_fine = checked_integer(
        difference.astype(object) * inv_std.astype(object),
        np.int64, "normalization",
    )

    # Requantization 2: normalized features -> INT32 at scale 2^-16.
    normalized = checked_integer(
        rq_checked(
            normalized_fine, 1,
            params["normalize_shift"], "normalization conversion",
        ),
        np.int32, "stored normalized features",
    )

    affine_product = checked_integer(
        normalized.astype(object) * params["gamma"].astype(object),
        np.int64, "affine product",
    )
    aligned_beta = checked_integer(
        params["beta"].astype(object) * (1 << 16),
        np.int64, "aligned beta",
    )
    affine = checked_integer(
        affine_product.astype(object) + aligned_beta.astype(object),
        np.int64, "affine sum",
    )

    # Requantization 3: affine INT64 at scale 2^-32 -> INT8.
    shifted = rq_checked(
        affine, params["output_multiplier"],
        params["output_shift"], "LN1 output",
    )
    unbounded = checked_integer(
        shifted.astype(object) + params["output_zero_point"],
        np.int64, "output zero point",
    )
    output = np.clip(unbounded, -128, 127).astype(np.int8)

    return {
        "mean": mean,
        "square_fine": square_fine,
        "square": square,
        "variance": variance,
        "variance_eps": variance_eps,
        "inv_std": inv_std,
        "normalized_fine": normalized_fine,
        "normalized": normalized,
        "affine": affine,
        "output": output,
        "saturation_count": int(np.count_nonzero(
            (unbounded < -128) | (unbounded > 127)
        )),
    }


def integer_ln1(input_codes, params):
    if input_codes.dtype != np.int8:
        raise ValueError("LN1 input must be INT8")
    return integer_layernorm(input_codes, params)


def integer_ln2(input_codes, params):
    if input_codes.dtype != np.int32:
        raise ValueError("LN2 input must be INT32")
    return integer_layernorm(input_codes, params)

In [14]:
ln1_trials = {i: [] for i in range(8)}

for block_index in range(8):
    params = ln1_params[block_index]
    interpreter = block_models[block_index]["interpreter"]

    for case in reference_cases:
        source = case["blocks"][block_index]["input_fp32"]

        codes, represented, input_clips = encode_boundary(
            source, block_index
        )

        # Run TFLite and capture its preserved LN1 output.
        run_tflite_block(block_index, represented)
        reference_output = interpreter.get_tensor(
            params["output_tensor"]
        ).copy()

        trial = integer_ln1(codes, params)

        ln1_trials[block_index].append({
            "name": case["name"],
            "input_codes": codes,
            "input_clips": input_clips,
            "reference_output": reference_output,
            "integer": trial,
        })


rows = [
    "| Block | Cases | Matching INT8 outputs | Agreement | "
    "Max code difference | Integer saturation count |",
    "|---:|---|---:|---:|---:|---:|",
]

for block_index in range(8):
    for group_name, names in (
        ("Benchmark", {"benchmark"}),
        ("Calibration", {case["name"] for case in calibration_cases}),
    ):
        selected = [
            trial for trial in ln1_trials[block_index]
            if trial["name"] in names
        ]

        actual = np.concatenate([
            trial["integer"]["output"].reshape(-1)
            for trial in selected
        ])
        expected = np.concatenate([
            trial["reference_output"].reshape(-1)
            for trial in selected
        ])

        differences = actual.astype(np.int32) - expected.astype(np.int32)
        matches = int(np.count_nonzero(differences == 0))
        max_difference = int(np.max(np.abs(differences)))
        saturations = sum(
            trial["integer"]["saturation_count"]
            for trial in selected
        )

        rows.append(
            f"| {block_index} | {group_name} | "
            f"{matches}/{actual.size} | "
            f"{100.0 * matches / actual.size:.3f}% | "
            f"{max_difference} | {saturations} |"
        )

display(Markdown("\n".join(rows)))
print("PASS: LN1 trials completed without checked arithmetic overflow.")

| Block | Cases | Matching INT8 outputs | Agreement | Max code difference | Integer saturation count |
|---:|---|---:|---:|---:|---:|
| 0 | Benchmark | 4090/4096 | 99.854% | 1 | 0 |
| 0 | Calibration | 28634/28672 | 99.867% | 1 | 1 |
| 1 | Benchmark | 4090/4096 | 99.854% | 1 | 1 |
| 1 | Calibration | 28628/28672 | 99.847% | 1 | 1 |
| 2 | Benchmark | 4094/4096 | 99.951% | 1 | 0 |
| 2 | Calibration | 28637/28672 | 99.878% | 1 | 2 |
| 3 | Benchmark | 4090/4096 | 99.854% | 1 | 0 |
| 3 | Calibration | 28644/28672 | 99.902% | 1 | 1 |
| 4 | Benchmark | 4087/4096 | 99.780% | 1 | 0 |
| 4 | Calibration | 28619/28672 | 99.815% | 1 | 2 |
| 5 | Benchmark | 4089/4096 | 99.829% | 1 | 1 |
| 5 | Calibration | 28610/28672 | 99.784% | 1 | 2 |
| 6 | Benchmark | 4073/4096 | 99.438% | 1 | 1 |
| 6 | Calibration | 28607/28672 | 99.773% | 1 | 3 |
| 7 | Benchmark | 4091/4096 | 99.878% | 1 | 1 |
| 7 | Calibration | 28628/28672 | 99.847% | 1 | 2 |

PASS: LN1 trials completed without checked arithmetic overflow.


## Part 2 — Integer Q/K/V projections, shared across blocks 0–7

### Input, output, and parameters

- Input: LN1 output, INT8 `[1,32,128]`.
- Weights: three INT8 matrices `[128,128]`, stored as `[output,input]`.
- Weight zero points: `0`.
- Bias: none for Q/K/V in the current exports.
- Accumulators: INT32 `[1,32,128]`.
- Outputs: three INT8 tensors `[1,32,128]`.
- Each projection uses its own weight scale, output scale, and output zero point.

Weights and quantization metadata come directly from each block’s saved TFLite.

### 1. Center the input

$$
x_c[t,k]=q_x[t,k]-z_x
$$

Promote before subtraction. Centered values may exceed INT8 range.

### 2. Accumulate the dot products

For each projection $p\in\{Q,K,V\}$:

$$
A_p[t,o]=\sum_{k=0}^{127}x_c[t,k]q_{W_p}[o,k]
$$

The accumulator scale is:

$$
s_{A_p}=s_xs_{W_p}
$$

- Store accumulators as INT32.
- Every product and partial sum must fit INT32.
- No requantization occurs inside the dot product.
- The C implementation uses XTILE for accumulation.

### 3. Prepare the conversion multiplier offline

Use the common right shift `r = 39`:

$$
M_p=\operatorname{round}_{even}
\left(
\frac{s_xs_{W_p}}{s_{\mathrm{out},p}}\,2^{39}
\right)
$$

Store the multiplier as INT64. It may exceed positive INT32.

Check that accumulator multiplication and rounding-offset addition fit INT64.

### 4. Requantize immediately after accumulation

Runtime rounding:

$$
RQ(A,M,39)=(AM+2^{38})\gg39
$$

This is nearest rounding, with ties toward positive infinity.

**Requantization point — after each completed projection:**

$$
q_p[t,o]=
\operatorname{clip}_{[-128,127]}
\left(
RQ(A_p[t,o],M_p,39)+z_{\mathrm{out},p}
\right)
$$

The C implementation uses XREQUANT mode 2: INT32 → INT8.

Q/K/V are converted before attention head layout operations.
Reshape and transpose preserve their codes, scales, and zero points.

### Validation

Two paths are checked against TFLite:

1. Projection only: use TFLite LN1 output to isolate projection arithmetic.
2. LN1 → Q/K/V: use integer LN1 output to measure propagated differences.

Current results across eight blocks and eight cases per block:

- Projection only: 786,432 INT8 outputs, zero mismatches.
- LN1 → Q/K/V agreement: 98.083%–99.756%.
- Maximum chained difference: one INT8 code.
- Local and chained saturation counts are identical.
- Parameter and arithmetic-bound checks passed.

“Chained” here means LN1 → Q/K/V within each block, not the complete multi-block integer inference path.

In [15]:
def tensor_int8_quant(detail):
    if np.dtype(detail["dtype"]) != np.dtype(np.int8):
        raise ValueError("Expected an INT8 tensor")

    quant = detail["quantization_parameters"]
    if len(quant["scales"]) != 1 or len(quant["zero_points"]) != 1:
        raise ValueError("Expected per-tensor activation quantization")

    scale = float(quant["scales"][0])
    zero_point = int(quant["zero_points"][0])

    if not math.isfinite(scale) or scale <= 0:
        raise ValueError("Invalid scale")
    if not -128 <= zero_point <= 127:
        raise ValueError("Invalid INT8 zero point")

    return scale, zero_point


qkv_params = {}

rows = [
    "| Block | Projection | Weight scale | Output scale | "
    "Output ZP | Multiplier / shift |",
    "|---:|---|---:|---:|---:|---|",
]

for block_index in range(8):
    model = block_models[block_index]
    interpreter = model["interpreter"]
    tensors = model["tensors"]
    qkv_params[block_index] = {}

    for name, op_index in (("Q", 15), ("K", 16), ("V", 17)):
        op = model["ops"][op_index]
        assert op["op_name"] == "FULLY_CONNECTED"

        inputs = [int(v) for v in op["inputs"] if int(v) >= 0]
        assert len(inputs) == 2, "This Q/K/V path expects no bias"

        input_id, weight_id = inputs
        output_id = int(op["outputs"][0])
        assert input_id == ln1_params[block_index]["output_tensor"]

        input_scale, input_zp = tensor_int8_quant(tensors[input_id])
        output_scale, output_zp = tensor_int8_quant(tensors[output_id])

        weights = interpreter.get_tensor(weight_id).copy()
        assert weights.dtype == np.int8
        assert weights.shape == (128, 128)

        weight_quant = tensors[weight_id]["quantization_parameters"]
        weight_scales = np.asarray(weight_quant["scales"], dtype=np.float64)
        weight_zps = np.asarray(weight_quant["zero_points"], dtype=np.int64)

        assert weight_scales.size in (1, 128)
        assert weight_zps.size in (1, 128)
        assert np.all(np.isfinite(weight_scales))
        assert np.all(weight_scales > 0)
        assert np.all(weight_zps == 0)
        assert np.all(weight_scales == weight_scales[0]), (
            "Nonuniform per-channel scales need a different conversion path"
        )
        if weight_scales.size > 1 or weight_zps.size > 1:
            assert int(weight_quant["quantized_dimension"]) == 0

        weight_scale = float(weight_scales[0])
        shift = 39
        multiplier = int(np.rint(
            input_scale * weight_scale / output_scale * (1 << shift)
        ))

        # Bound every accumulator and partial sum over all INT8 inputs.
        input_bound = max(abs(-128 - input_zp), abs(127 - input_zp))
        accumulator_bounds = (
            input_bound * np.abs(weights.astype(np.int64)).sum(axis=1)
        )
        assert np.all(accumulator_bounds <= np.iinfo(np.int32).max)

        # Allow INT64 multipliers, retaining the product/rounding check.
        assert 0 < multiplier <= np.iinfo(np.int64).max
        assert (
            (1 << 31) * multiplier + (1 << (shift - 1))
            <= np.iinfo(np.int64).max
        ), "INT64 product/rounding bound exceeded"

        qkv_params[block_index][name] = {
            "input_tensor": input_id,
            "output_tensor": output_id,
            "input_scale": input_scale,
            "input_zero_point": input_zp,
            "weights": weights,
            "weight_scale": weight_scale,
            "output_scale": output_scale,
            "output_zero_point": output_zp,
            "multiplier": multiplier,
            "shift": shift,
        }

        rows.append(
            f"| {block_index} | {name} | "
            f"{weight_scale:.9g} | {output_scale:.9g} | "
            f"{output_zp} | {multiplier} / {shift} |"
        )

display(Markdown("\n".join(rows)))
print("PASS: all 24 projections passed parameter and arithmetic-bound checks.")

| Block | Projection | Weight scale | Output scale | Output ZP | Multiplier / shift |
|---:|---|---:|---:|---:|---|
| 0 | Q | 0.00173675152 | 0.0250611212 | -12 | 1280944693 / 39 |
| 0 | K | 0.00173511321 | 0.0182555057 | -30 | 1756819468 / 39 |
| 0 | V | 0.000636710436 | 0.00693884166 | 4 | 1696087205 / 39 |
| 1 | Q | 0.00441292208 | 0.110937938 | 21 | 963966046 / 39 |
| 1 | K | 0.00226015411 | 0.0431736633 | -2 | 1268629251 / 39 |
| 1 | V | 0.00100487308 | 0.0100876652 | -15 | 2413993628 / 39 |
| 2 | Q | 0.00336248567 | 0.0901082829 | 28 | 905981647 / 39 |
| 2 | K | 0.00196137466 | 0.039457079 | 25 | 1206866683 / 39 |
| 2 | V | 0.000817320833 | 0.00673493976 | -7 | 2946337712 / 39 |
| 3 | Q | 0.00338659482 | 0.0869133919 | -5 | 889578276 / 39 |
| 3 | K | 0.0018085168 | 0.0314015336 | 5 | 1314859401 / 39 |
| 3 | V | 0.000638312253 | 0.00651221676 | -20 | 2237752039 / 39 |
| 4 | Q | 0.00271653617 | 0.0617195852 | 3 | 905672561 / 39 |
| 4 | K | 0.00172414782 | 0.0342418104 | 8 | 1036087647 / 39 |
| 4 | V | 0.000636841869 | 0.00632951129 | 11 | 2070332722 / 39 |
| 5 | Q | 0.00259025162 | 0.0458002873 | 24 | 1143835664 / 39 |
| 5 | K | 0.0013373656 | 0.027128147 | 15 | 997056797 / 39 |
| 5 | V | 0.000563783105 | 0.00573980762 | 8 | 1986573338 / 39 |
| 6 | Q | 0.0018354893 | 0.0327574573 | 8 | 1069892899 / 39 |
| 6 | K | 0.00157897337 | 0.0256558228 | 4 | 1175134270 / 39 |
| 6 | V | 0.000481187191 | 0.00550880423 | 5 | 1667843715 / 39 |
| 7 | Q | 0.00206250092 | 0.0450716615 | 13 | 1079810296 / 39 |
| 7 | K | 0.00150548341 | 0.0271682031 | 0 | 1307591047 / 39 |
| 7 | V | 0.000723360921 | 0.00625059241 | 11 | 2730805338 / 39 |

PASS: all 24 projections passed parameter and arithmetic-bound checks.


In [16]:
def integer_projection(input_codes, params):
    if input_codes.dtype != np.int8 or input_codes.shape != (1, 32, 128):
        raise ValueError("Projection expects INT8 [1,32,128]")

    centered = (
        input_codes.reshape(32, 128).astype(np.int64)
        - params["input_zero_point"]
    )

    accumulator = centered @ params["weights"].astype(np.int64).T
    accumulator = checked_integer(
        accumulator, np.int32, "projection accumulator"
    )

    # Requantize after accumulation, before attention/layout operations.
    converted = rq_checked(
        accumulator,
        params["multiplier"],
        params["shift"],
        "projection requantization",
    )
    unbounded = checked_integer(
        converted.astype(object) + params["output_zero_point"],
        np.int64,
        "projection zero point",
    )

    output = np.clip(unbounded, -128, 127).astype(np.int8)

    return {
        "accumulator": accumulator.reshape(1, 32, 128),
        "output": output.reshape(1, 32, 128),
        "saturation_count": int(np.count_nonzero(
            (unbounded < -128) | (unbounded > 127)
        )),
    }

In [17]:
qkv_trials = {i: [] for i in range(8)}

for block_index in range(8):
    model = block_models[block_index]
    interpreter = model["interpreter"]
    input_scale = block_boundary_specs[block_index]["input"]["scale"]

    for ln1_trial in ln1_trials[block_index]:
        represented = (
            ln1_trial["input_codes"].astype(np.float64) * input_scale
        ).astype(np.float32)

        run_tflite_block(block_index, represented)

        # Confirm this rerun reproduces the previous LN1 reference.
        np.testing.assert_array_equal(
            interpreter.get_tensor(
                ln1_params[block_index]["output_tensor"]
            ),
            ln1_trial["reference_output"],
        )

        result = {
            "name": ln1_trial["name"],
            "projections": {},
        }

        for name in ("Q", "K", "V"):
            params = qkv_params[block_index][name]
            reference = interpreter.get_tensor(
                params["output_tensor"]
            ).copy()

            result["projections"][name] = {
                "reference": reference,
                "local": integer_projection(
                    ln1_trial["reference_output"], params
                ),
                "ln1_chained": integer_projection(
                    ln1_trial["integer"]["output"], params
                ),
            }

        qkv_trials[block_index].append(result)

    print(f"Completed Q/K/V trials: block {block_index}", flush=True)

Completed Q/K/V trials: block 0
Completed Q/K/V trials: block 1
Completed Q/K/V trials: block 2
Completed Q/K/V trials: block 3
Completed Q/K/V trials: block 4
Completed Q/K/V trials: block 5
Completed Q/K/V trials: block 6
Completed Q/K/V trials: block 7


In [18]:
rows = [
    "| Block | Projection | Projection-only agreement | Max code diff | "
    "LN1 → QKV agreement | Max code diff | Saturations: local / chained |",
    "|---:|---|---:|---:|---:|---:|---|",
]

for block_index in range(8):
    for name in ("Q", "K", "V"):
        selected = [
            trial["projections"][name]
            for trial in qkv_trials[block_index]
        ]

        reference = np.concatenate([
            trial["reference"].reshape(-1) for trial in selected
        ]).astype(np.int32)

        metrics = {}
        for path in ("local", "ln1_chained"):
            actual = np.concatenate([
                trial[path]["output"].reshape(-1)
                for trial in selected
            ]).astype(np.int32)

            difference = actual - reference
            matches = int(np.count_nonzero(difference == 0))

            metrics[path] = {
                "agreement": 100.0 * matches / reference.size,
                "max_difference": int(np.max(np.abs(difference))),
                "saturations": sum(
                    trial[path]["saturation_count"]
                    for trial in selected
                ),
            }

        local = metrics["local"]
        chained = metrics["ln1_chained"]

        rows.append(
            f"| {block_index} | {name} | "
            f"{local['agreement']:.3f}% | {local['max_difference']} | "
            f"{chained['agreement']:.3f}% | {chained['max_difference']} | "
            f"{local['saturations']} / {chained['saturations']} |"
        )

display(Markdown("\n".join(rows)))
print("PASS: Q/K/V trials completed without checked arithmetic overflow.")

| Block | Projection | Projection-only agreement | Max code diff | LN1 → QKV agreement | Max code diff | Saturations: local / chained |
|---:|---|---:|---:|---:|---:|---|
| 0 | Q | 100.000% | 0 | 99.481% | 1 | 3 / 3 |
| 0 | K | 100.000% | 0 | 99.399% | 1 | 9 / 9 |
| 0 | V | 100.000% | 0 | 99.200% | 1 | 0 / 0 |
| 1 | Q | 100.000% | 0 | 99.570% | 1 | 3 / 3 |
| 1 | K | 100.000% | 0 | 99.332% | 1 | 4 / 4 |
| 1 | V | 100.000% | 0 | 98.749% | 1 | 3 / 3 |
| 2 | Q | 100.000% | 0 | 99.686% | 1 | 2 / 2 |
| 2 | K | 100.000% | 0 | 99.475% | 1 | 6 / 6 |
| 2 | V | 100.000% | 0 | 98.874% | 1 | 2 / 2 |
| 3 | Q | 100.000% | 0 | 99.756% | 1 | 3 / 3 |
| 3 | K | 100.000% | 0 | 99.429% | 1 | 2 / 2 |
| 3 | V | 100.000% | 0 | 99.020% | 1 | 3 / 3 |
| 4 | Q | 100.000% | 0 | 99.277% | 1 | 0 / 0 |
| 4 | K | 100.000% | 0 | 99.200% | 1 | 2 / 2 |
| 4 | V | 100.000% | 0 | 98.276% | 1 | 1 / 1 |
| 5 | Q | 100.000% | 0 | 99.112% | 1 | 4 / 4 |
| 5 | K | 100.000% | 0 | 99.030% | 1 | 3 / 3 |
| 5 | V | 100.000% | 0 | 98.294% | 1 | 4 / 4 |
| 6 | Q | 100.000% | 0 | 98.953% | 1 | 4 / 4 |
| 6 | K | 100.000% | 0 | 98.575% | 1 | 6 / 6 |
| 6 | V | 100.000% | 0 | 98.083% | 1 | 1 / 1 |
| 7 | Q | 100.000% | 0 | 99.323% | 1 | 0 / 0 |
| 7 | K | 100.000% | 0 | 99.097% | 1 | 5 / 5 |
| 7 | V | 100.000% | 0 | 98.453% | 1 | 2 / 2 |

PASS: Q/K/V trials completed without checked arithmetic overflow.


In [19]:
attention_params = {}

rows = [
    "| Block | Score scale | Q zero point | K zero point | "
    "Maximum absolute score bound |",
    "|---:|---:|---:|---:|---:|",
]

for block_index in range(8):
    model = block_models[block_index]
    interpreter = model["interpreter"]
    ops = model["ops"]

    assert ops[28]["op_name"] == "BATCH_MATMUL"
    assert ops[31]["op_name"] == "SOFTMAX"
    assert ops[35]["op_name"] == "BATCH_MATMUL"

    # Verify the mask required by the existing attention method.
    mask_id = int(ops[30]["inputs"][0])
    mask = interpreter.get_tensor(mask_id)
    np.testing.assert_array_equal(
        mask, np.tri(32, dtype=bool).reshape(1, 1, 32, 32)
    )

    query = qkv_params[block_index]["Q"]
    key = qkv_params[block_index]["K"]

    query_zp = query["output_zero_point"]
    key_zp = key["output_zero_point"]

    query_bound = max(abs(-128 - query_zp), abs(127 - query_zp))
    key_bound = max(abs(-128 - key_zp), abs(127 - key_zp))
    score_bound = 8 * query_bound * key_bound

    assert score_bound <= np.iinfo(np.int32).max
    assert 2 * score_bound <= np.iinfo(np.int32).max

    attention_params[block_index] = {
        "query_zero_point": query_zp,
        "key_zero_point": key_zp,
        "score_scale": query["output_scale"] * key["output_scale"],
        "score_bound": score_bound,
        "score_tensor": int(ops[28]["outputs"][0]),
        "probability_tensor": int(ops[31]["outputs"][0]),
        "context_tensor": int(ops[35]["outputs"][0]),
        "output_tensor": int(ops[39]["outputs"][0]),
    }

    rows.append(
        f"| {block_index} | "
        f"{attention_params[block_index]['score_scale']:.12g} | "
        f"{query_zp} | {key_zp} | {score_bound} |"
    )

display(Markdown("\n".join(rows)))


def integer_attention_scores(query, key, params):
    for codes in (query, key):
        if codes.dtype != np.int8 or codes.shape != (1, 32, 128):
            raise ValueError("Q/K must be INT8 [1,32,128]")

    # Token-major [32,128] -> heads [16,32,8].
    query_heads = (
        query.reshape(32, 16, 8)
        .transpose(1, 0, 2)
        .astype(np.int64)
    )
    key_heads = (
        key.reshape(32, 16, 8)
        .transpose(1, 0, 2)
        .astype(np.int64)
    )

    query_heads -= params["query_zero_point"]
    key_heads -= params["key_zero_point"]

    # Each head: [32,8] @ [8,32] -> [32,32].
    scores = query_heads @ key_heads.transpose(0, 2, 1)

    return checked_integer(scores, np.int32, "attention scores")

| Block | Score scale | Q zero point | K zero point | Maximum absolute score bound |
|---:|---:|---:|---:|---:|
| 0 | 0.000457503441378 | -12 | -30 | 174584 |
| 1 | 0.00478959718849 | 21 | -2 | 153768 |
| 2 | 0.00355540963761 | 28 | 25 | 190944 |
| 3 | 0.00272921380045 | -5 | 5 | 140448 |
| 4 | 0.00211339033726 | 3 | 8 | 142528 |
| 5 | 0.00124247692411 | 24 | 15 | 173888 |
| 6 | 0.000840419521971 | 8 | 4 | 143616 |
| 7 | 0.00122451605497 | 13 | 0 | 144384 |

In [20]:
attention_score_trials = {i: [] for i in range(8)}

for block_index in range(8):
    model = block_models[block_index]
    interpreter = model["interpreter"]
    params = attention_params[block_index]
    input_scale = block_boundary_specs[block_index]["input"]["scale"]

    assert len(qkv_trials[block_index]) == len(ln1_trials[block_index])

    for projection_trial, ln1_trial in zip(
        qkv_trials[block_index], ln1_trials[block_index]
    ):
        assert projection_trial["name"] == ln1_trial["name"]

        represented = (
            ln1_trial["input_codes"].astype(np.float64) * input_scale
        ).astype(np.float32)

        run_tflite_block(block_index, represented)

        # Confirm that the references correspond to this same input.
        for name in ("Q", "K", "V"):
            np.testing.assert_array_equal(
                interpreter.get_tensor(
                    qkv_params[block_index][name]["output_tensor"]
                ),
                projection_trial["projections"][name]["reference"],
            )

        references = {
            "scores": interpreter.get_tensor(
                params["score_tensor"]
            ).copy().reshape(16, 32, 32),
            "probabilities": interpreter.get_tensor(
                params["probability_tensor"]
            ).copy().reshape(16, 32, 32),
            "context": interpreter.get_tensor(
                params["context_tensor"]
            ).copy().reshape(16, 32, 8),
            "output": interpreter.get_tensor(
                params["output_tensor"]
            ).copy(),
        }

        projections = projection_trial["projections"]

        local_scores = integer_attention_scores(
            projections["Q"]["reference"],
            projections["K"]["reference"],
            params,
        )
        chained_scores = integer_attention_scores(
            projections["Q"]["ln1_chained"]["output"],
            projections["K"]["ln1_chained"]["output"],
            params,
        )

        attention_score_trials[block_index].append({
            "name": projection_trial["name"],
            "reference": references,
            "projection_trial": projection_trial,
            "local_scores": local_scores,
            "ln1_chained_scores": chained_scores,
        })

    print(f"Completed QK score trials: block {block_index}", flush=True)

Completed QK score trials: block 0
Completed QK score trials: block 1
Completed QK score trials: block 2
Completed QK score trials: block 3
Completed QK score trials: block 4
Completed QK score trials: block 5
Completed QK score trials: block 6
Completed QK score trials: block 7


In [21]:
attention_score_metrics = {}

rows = [
    "| Block | Cases | Local max absolute error | Local relative L2 | "
    "LN1 → QKV → scores max absolute error | Chained relative L2 |",
    "|---:|---|---:|---:|---:|---:|",
]

for block_index in range(8):
    score_scale = attention_params[block_index]["score_scale"]

    for group_name, names in (
        ("Benchmark", {"benchmark"}),
        ("Calibration", {case["name"] for case in calibration_cases}),
    ):
        selected = [
            trial for trial in attention_score_trials[block_index]
            if trial["name"] in names
        ]

        reference = np.concatenate([
            trial["reference"]["scores"].reshape(-1)
            for trial in selected
        ]).astype(np.float64)

        metrics = {}

        for path in ("local_scores", "ln1_chained_scores"):
            represented = np.concatenate([
                trial[path].reshape(-1) for trial in selected
            ]).astype(np.float64) * score_scale

            metrics[path] = error_metrics(represented, reference)

        attention_score_metrics[(block_index, group_name)] = metrics
        local = metrics["local_scores"]
        chained = metrics["ln1_chained_scores"]

        rows.append(
            f"| {block_index} | {group_name} | "
            f"{local['max_abs']:.6g} | "
            f"{local['relative_l2_percent']:.6g}% | "
            f"{chained['max_abs']:.6g} | "
            f"{chained['relative_l2_percent']:.6g}% |"
        )

display(Markdown("\n".join(rows)))
print("PASS: QK score trials completed without checked arithmetic overflow.")

| Block | Cases | Local max absolute error | Local relative L2 | LN1 → QKV → scores max absolute error | Chained relative L2 |
|---:|---|---:|---:|---:|---:|
| 0 | Benchmark | 8.72252e-07 | 6.70653e-06% | 0.0695404 | 0.441346% |
| 0 | Calibration | 8.91919e-07 | 6.45366e-06% | 0.0741157 | 0.391169% |
| 1 | Benchmark | 2.70969e-05 | 4.95194e-06% | 0.991444 | 0.0993127% |
| 1 | Calibration | 3.1505e-05 | 5.00579e-06% | 1.08245 | 0.109242% |
| 2 | Benchmark | 1.57049e-05 | 4.96668e-06% | 0.355535 | 0.068941% |
| 2 | Calibration | 2.00637e-05 | 5.00904e-06% | 0.69686 | 0.103604% |
| 3 | Benchmark | 1.01561e-05 | 5.35531e-06% | 0.368441 | 0.0942997% |
| 3 | Calibration | 1.4484e-05 | 5.40209e-06% | 0.766906 | 0.102149% |
| 4 | Benchmark | 6.19951e-06 | 5.37864e-06% | 0.340257 | 0.192077% |
| 4 | Calibration | 7.08051e-06 | 5.32568e-06% | 0.429018 | 0.166286% |
| 5 | Benchmark | 3.22451e-06 | 5.5506e-06% | 0.244767 | 0.197858% |
| 5 | Calibration | 4.50654e-06 | 5.47638e-06% | 0.298194 | 0.203939% |
| 6 | Benchmark | 2.07446e-06 | 5.31637e-06% | 0.152956 | 0.279875% |
| 6 | Calibration | 2.34777e-06 | 5.34267e-06% | 0.189935 | 0.217049% |
| 7 | Benchmark | 2.19652e-06 | 6.10807e-06% | 0.29021 | 0.309168% |
| 7 | Calibration | 4.01838e-06 | 5.84233e-06% | 0.21429 | 0.261018% |

PASS: QK score trials completed without checked arithmetic overflow.


## Part 3 — Integer attention, shared across blocks 0–7

### Inputs, output, and layout

- Inputs: Q/K/V, each INT8 `[1,32,128]`, with independent scales and zero points.
- Heads: 16; head dimension: 8; sequence length: 32.
- Output: merged context, INT8 `[1,32,128]`.
- The output scale and zero point come from each block’s saved TFLite.
- Head layout changes preserve integer codes and quantization.
- Retain the existing software attention algorithm.
- Use XREQUANT mode 2 for the final context conversion.

### 1. Compute QK scores

For each head:

$$
S[h,t,j]=\sum_{d=0}^{7}
(q_Q[h,t,d]-z_Q)(q_K[h,j,d]-z_K)
$$

Store scores as INT32 `[16,32,32]` at scale:

$$
s_S=s_Qs_K
$$

Promote before zero-point subtraction.
Check product and partial-sum bounds.

No requantization occurs inside this dot product.
No additional attention scaling factor is inserted.

### 2. Apply causal masking and subtract the row maximum

Only keys `j <= t` participate:

$$
m[h,t]=\max_{j\le t}S[h,t,j]
$$

$$
D[h,t,j]=S[h,t,j]-m[h,t]
$$

Allowed shifted scores are nonpositive and retain scale `s_S`.
Masked entries must contribute zero probability.

### 3. Convert scores into exponential-table coordinates

Prepare the multiplier offline:

$$
M_{\mathrm{coord}}
=\operatorname{round}_{even}(s_S\,2^{16}\,2^{24})
$$

**Rounded coordinate conversion:**

$$
c=RQ(-D,M_{\mathrm{coord}},24)
$$

Coordinates use scale `2^-16`.
The multiplier and product use checked INT64 arithmetic.

### 4. Approximate exponentials

Shared offline table:

$$
L[i]=\operatorname{round}_{even}
\left(e^{-i/256}2^{30}\right),
\qquad i=0,\ldots,4096
$$

For allowed entries:

- If `c >= 16 × 65536`, set the exponential to zero.
- Otherwise, calculate:

$$
i=c\gg8,\qquad f=c\mathbin{\&}255
$$

$$
E=L[i]+RQ((L[i+1]-L[i])f,1,8)
$$

Store exponentials as INT32 at scale `2^-30`.
Interpolation rounding preserves that output scale.

Set masked exponentials to zero.

### 5. Normalize probabilities

Sum exponentials using INT64:

$$
Z[h,t]=\sum_{j\le t}E[h,t,j]
$$

The denominator must be positive.

Compute independently rounded probabilities:

$$
P_q[h,t,j]
=
\left\lfloor
\frac{E[h,t,j]2^{16}+\lfloor Z[h,t]/2\rfloor}
{Z[h,t]}
\right\rfloor
$$

Store probabilities as INT32 at scale `2^-16`.

- Probability code `65536` represents `1`.
- Masked probabilities are zero.
- Do not force independently rounded row sums to equal `65536`.

### 6. Compute probability × V

$$
C[h,t,d]
=\sum_{j=0}^{31}
P_q[h,t,j](q_V[h,j,d]-z_V)
$$

Store context accumulators as INT32 at scale:

$$
s_C=2^{-16}s_V
$$

Check product and partial-sum bounds.
No requantization occurs inside the dot product.

Merge heads into token-major `[1,32,128]`.
This layout change preserves values and scale.

### 7. Requantize the context into INT8

Let `s_out` and `z_out` be the saved context-output quantization.

Prepare offline:

$$
M_C=\operatorname{round}_{even}
\left(\frac{2^{-16}s_V}{s_{\mathrm{out}}}2^{46}\right)
$$

**Final context requantization:**

$$
q_{\mathrm{out}}
=
\operatorname{clip}_{[-128,127]}
\left(RQ(C,M_C,46)+z_{\mathrm{out}}\right)
$$

Use an INT64 multiplier and checked INT64 product/rounding addition.
The output feeds Part 4’s attention output projection.

### Validation

Local path:
reference Q/K/V → integer attention.

Chained path:
integer LN1 → integer Q/K/V → integer attention.

Current eight-case results across all eight blocks:

- Local merged-output agreement: 99.927%–100%.
- Local maximum difference: one INT8 code.
- Local maximum probability error: approximately 1.31e-5.
- Maximum probability row-sum deviation: six integer steps at scale 2^-16.
- Chained context relative L2: approximately 0.169%–1.403%.
- Chained maximum merged-output difference: 14 INT8 codes.
- Local and chained saturation counts are identical.
- Arithmetic checks completed without overflow.

Retain the block-1 chained outlier for full-chain evaluation.
These results validate the current stage candidate on the tested cases;
they do not establish complete-model accuracy.

In [22]:
softmax_exp_lut = checked_integer(
    np.rint(
        np.exp(-np.arange(4097, dtype=np.float64) / 256.0)
        * (1 << 30)
    ),
    np.int32,
    "exponential table",
)

assert softmax_exp_lut[0] == (1 << 30)
assert np.all(np.diff(softmax_exp_lut.astype(np.int64)) <= 0)

rows = [
    "| Block | Coordinate multiplier / shift | "
    "Exponential scale | Probability scale |",
    "|---:|---|---|---|",
]

for block_index in range(8):
    params = attention_params[block_index]

    # Convert score magnitudes into coordinates at scale 2^-16.
    multiplier = int(np.rint(
        params["score_scale"] * (1 << 16) * (1 << 24)
    ))

    assert 0 < multiplier <= np.iinfo(np.int64).max

    # A shifted score magnitude is bounded by twice the raw score bound.
    assert (
        2 * params["score_bound"] * multiplier + (1 << 23)
        <= np.iinfo(np.int64).max
    ), "Softmax coordinate product/rounding exceeds INT64"

    params.update({
        "coordinate_multiplier": multiplier,
        "coordinate_shift": 24,
        "exp_lut": softmax_exp_lut,
    })

    rows.append(
        f"| {block_index} | {multiplier} / 24 | "
        "2^-30 | 2^-16 |"
    )

display(Markdown("\n".join(rows)))

| Block | Coordinate multiplier / shift | Exponential scale | Probability scale |
|---:|---|---|---|
| 0 | 503030354 / 24 | 2^-30 | 2^-16 |
| 1 | 5266217801 / 24 | 2^-30 | 2^-16 |
| 2 | 3909214238 / 24 | 2^-30 | 2^-16 |
| 3 | 3000802308 / 24 | 2^-30 | 2^-16 |
| 4 | 2323697250 / 24 | 2^-30 | 2^-16 |
| 5 | 1366117825 / 24 | 2^-30 | 2^-16 |
| 6 | 924051037 / 24 | 2^-30 | 2^-16 |
| 7 | 1346369641 / 24 | 2^-30 | 2^-16 |

In [23]:
def integer_causal_softmax(scores, params):
    if scores.dtype != np.int32 or scores.shape != (16, 32, 32):
        raise ValueError("Softmax expects INT32 [16,32,32]")

    allowed = np.tri(32, dtype=bool)[None, :, :]

    # Maximum over allowed keys only.
    maximum = np.where(
        allowed, scores, np.iinfo(np.int32).min
    ).max(axis=-1, keepdims=True)

    shifted = checked_integer(
        scores.astype(object) - maximum.astype(object),
        np.int32, "shifted scores",
    )
    shifted = np.where(allowed, shifted, 0).astype(np.int32)
    assert np.all(shifted <= 0)

    # Rounded conversion to coordinates at scale 2^-16.
    coordinates = rq_checked(
        -shifted.astype(object),
        params["coordinate_multiplier"],
        params["coordinate_shift"],
        "softmax coordinate",
    )

    # Exponentials below the existing cutoff are discarded.
    active = allowed & (coordinates < (16 << 16))
    safe_coordinates = np.where(active, coordinates, 0)

    # LUT spacing: 1/256; remaining eight bits interpolate.
    indices = safe_coordinates >> 8
    fractions = safe_coordinates & 255

    table = params["exp_lut"]
    lower = table[indices].astype(np.int64)
    upper = table[indices + 1].astype(np.int64)

    interpolation_product = checked_integer(
        (upper.astype(object) - lower.astype(object))
        * fractions.astype(object),
        np.int64, "exponential interpolation product",
    )
    adjustment = rq_checked(
        interpolation_product, 1, 8, "exponential interpolation"
    )
    interpolated = checked_integer(
        lower.astype(object) + adjustment.astype(object),
        np.int32, "interpolated exponential",
    )
    exponentials = np.where(active, interpolated, 0).astype(np.int32)

    denominator = checked_integer(
        exponentials.astype(object).sum(axis=-1, keepdims=True),
        np.int64, "softmax denominator",
    )
    if np.any(denominator <= 0):
        raise ValueError("Nonpositive softmax denominator")

    # Rounded positive division, producing probability codes at 2^-16.
    numerator = checked_integer(
        exponentials.astype(object) * (1 << 16)
        + denominator.astype(object) // 2,
        np.int64, "probability numerator",
    )
    probabilities = checked_integer(
        numerator // denominator,
        np.int32, "probabilities",
    )

    assert np.all(
        (probabilities >= 0) & (probabilities <= (1 << 16))
    )
    assert np.all(np.where(allowed, 0, probabilities) == 0)

    row_sums = probabilities.astype(np.int64).sum(axis=-1)

    return {
        "shifted_scores": shifted,
        "coordinates": coordinates,
        "exponentials": exponentials,
        "denominator": denominator,
        "probabilities": probabilities,
        "max_row_sum_error": int(np.max(
            np.abs(row_sums - (1 << 16))
        )),
        "cutoff_count": int(np.count_nonzero(allowed & ~active)),
    }

In [24]:
softmax_trials = {i: [] for i in range(8)}

for block_index in range(8):
    params = attention_params[block_index]

    for score_trial in attention_score_trials[block_index]:
        softmax_trials[block_index].append({
            "name": score_trial["name"],
            "source": score_trial,
            "local": integer_causal_softmax(
                score_trial["local_scores"], params
            ),
            "ln1_chained": integer_causal_softmax(
                score_trial["ln1_chained_scores"], params
            ),
        })

    print(f"Completed softmax trials: block {block_index}", flush=True)

Completed softmax trials: block 0
Completed softmax trials: block 1
Completed softmax trials: block 2
Completed softmax trials: block 3
Completed softmax trials: block 4
Completed softmax trials: block 5
Completed softmax trials: block 6
Completed softmax trials: block 7


In [25]:
softmax_metrics = {}

rows = [
    "| Block | Cases | Local max probability error | Local relative L2 | "
    "Chained max probability error | Chained relative L2 | "
    "Max row-sum error: local / chained, integer steps |",
    "|---:|---|---:|---:|---:|---:|---|",
]

for block_index in range(8):
    for group_name, names in (
        ("Benchmark", {"benchmark"}),
        ("Calibration", {case["name"] for case in calibration_cases}),
    ):
        selected = [
            trial for trial in softmax_trials[block_index]
            if trial["name"] in names
        ]

        reference = np.concatenate([
            trial["source"]["reference"]["probabilities"].reshape(-1)
            for trial in selected
        ]).astype(np.float64)

        metrics = {}

        for path in ("local", "ln1_chained"):
            represented = np.concatenate([
                trial[path]["probabilities"].reshape(-1)
                for trial in selected
            ]).astype(np.float64) * (2.0 ** -16)

            metrics[path] = error_metrics(represented, reference)
            metrics[path]["max_row_sum_error"] = max(
                trial[path]["max_row_sum_error"]
                for trial in selected
            )

        softmax_metrics[(block_index, group_name)] = metrics
        local = metrics["local"]
        chained = metrics["ln1_chained"]

        rows.append(
            f"| {block_index} | {group_name} | "
            f"{local['max_abs']:.6g} | "
            f"{local['relative_l2_percent']:.6g}% | "
            f"{chained['max_abs']:.6g} | "
            f"{chained['relative_l2_percent']:.6g}% | "
            f"{local['max_row_sum_error']} / "
            f"{chained['max_row_sum_error']} |"
        )

display(Markdown("\n".join(rows)))
print("PASS: softmax trials completed without checked arithmetic overflow.")

| Block | Cases | Local max probability error | Local relative L2 | Chained max probability error | Chained relative L2 | Max row-sum error: local / chained, integer steps |
|---:|---|---:|---:|---:|---:|---|
| 0 | Benchmark | 8.86619e-06 | 0.00468552% | 0.0111794 | 0.248553% | 5 / 5 |
| 0 | Calibration | 9.74536e-06 | 0.0045944% | 0.00937718 | 0.193552% | 5 / 5 |
| 1 | Benchmark | 1.00732e-05 | 0.00297319% | 0.0868923 | 1.42827% | 4 / 4 |
| 1 | Calibration | 1.15633e-05 | 0.00282908% | 0.14385 | 2.30025% | 4 / 4 |
| 2 | Benchmark | 9.50694e-06 | 0.00329488% | 0.0202907 | 0.46682% | 4 / 4 |
| 2 | Calibration | 1.3113e-05 | 0.00290969% | 0.108712 | 1.82121% | 6 / 5 |
| 3 | Benchmark | 8.9258e-06 | 0.00348488% | 0.023249 | 0.890964% | 4 / 4 |
| 3 | Calibration | 9.41753e-06 | 0.00318984% | 0.0705001 | 1.24939% | 5 / 5 |
| 4 | Benchmark | 8.67248e-06 | 0.00371947% | 0.0421444 | 1.42175% | 5 / 5 |
| 4 | Calibration | 9.05991e-06 | 0.00337891% | 0.0535122 | 0.955736% | 5 / 6 |
| 5 | Benchmark | 8.56817e-06 | 0.00395363% | 0.0181693 | 0.672278% | 4 / 4 |
| 5 | Calibration | 9.38773e-06 | 0.00369892% | 0.0346505 | 1.02261% | 5 / 5 |
| 6 | Benchmark | 8.49366e-06 | 0.00405953% | 0.021206 | 0.803495% | 5 / 5 |
| 6 | Calibration | 9.26852e-06 | 0.00370877% | 0.0239464 | 0.580091% | 5 / 5 |
| 7 | Benchmark | 9.20892e-06 | 0.00389887% | 0.0183477 | 0.679137% | 5 / 5 |
| 7 | Calibration | 9.50694e-06 | 0.00364036% | 0.0327172 | 0.596601% | 5 / 5 |

PASS: softmax trials completed without checked arithmetic overflow.


In [26]:
rows = [
    "| Block | Context accumulator scale | INT8 output scale | "
    "Output ZP | Multiplier / shift |",
    "|---:|---:|---:|---:|---|",
]

for block_index in range(8):
    model = block_models[block_index]
    params = attention_params[block_index]
    value = qkv_params[block_index]["V"]

    output_scale, output_zp = tensor_int8_quant(
        model["tensors"][params["output_tensor"]]
    )

    # Head merging must preserve the context quantization.
    quantized_context_id = int(model["ops"][36]["outputs"][0])
    assert tensor_int8_quant(
        model["tensors"][quantized_context_id]
    ) == (output_scale, output_zp)

    context_scale = (2.0 ** -16) * value["output_scale"]
    multiplier = int(np.rint(
        context_scale / output_scale * (1 << 46)
    ))

    value_zp = value["output_zero_point"]
    value_bound = max(abs(-128 - value_zp), abs(127 - value_zp))

    # Conservative bound: 32 terms, each probability code <= 65536.
    context_bound = 32 * (1 << 16) * value_bound

    assert context_bound <= np.iinfo(np.int32).max
    assert 0 < multiplier <= np.iinfo(np.int64).max
    assert (
        context_bound * multiplier + (1 << 45)
        <= np.iinfo(np.int64).max
    ), "Context product/rounding exceeds INT64"

    params.update({
        "value_zero_point": value_zp,
        "context_scale": context_scale,
        "context_multiplier": multiplier,
        "context_shift": 46,
        "context_output_scale": output_scale,
        "context_output_zero_point": output_zp,
    })

    rows.append(
        f"| {block_index} | {context_scale:.12g} | "
        f"{output_scale:.12g} | {output_zp} | "
        f"{multiplier} / 46 |"
    )

display(Markdown("\n".join(rows)))

| Block | Context accumulator scale | INT8 output scale | Output ZP | Multiplier / shift |
|---:|---:|---:|---:|---|
| 0 | 1.05878321222e-07 | 0.00669277552515 | 9 | 1113218944 / 46 |
| 1 | 1.53925554969e-07 | 0.00902162585407 | -30 | 1200620395 / 46 |
| 2 | 1.02767025112e-07 | 0.00588968163356 | 9 | 1227839967 / 46 |
| 3 | 9.93685418393e-08 | 0.00563115533441 | -28 | 1241741541 / 46 |
| 4 | 9.65806776776e-08 | 0.00563085824251 | 21 | 1206967163 / 46 |
| 5 | 8.75825136859e-08 | 0.00389645481482 | -10 | 1581712555 / 46 |
| 6 | 8.40576817041e-08 | 0.00371876568533 | -13 | 1590590535 / 46 |
| 7 | 9.53764711653e-08 | 0.00395004777238 | 11 | 1699099071 / 46 |

In [27]:
def integer_attention_context(probabilities, value, params):
    if (
        probabilities.dtype != np.int32
        or probabilities.shape != (16, 32, 32)
    ):
        raise ValueError("Probabilities must be INT32 [16,32,32]")

    if value.dtype != np.int8 or value.shape != (1, 32, 128):
        raise ValueError("V must be INT8 [1,32,128]")

    if np.any(probabilities < 0) or np.any(probabilities > (1 << 16)):
        raise ValueError("Probability code outside [0,65536]")

    value_heads = (
        value.reshape(32, 16, 8)
        .transpose(1, 0, 2)
        .astype(np.int64)
    )
    value_heads -= params["value_zero_point"]

    # [16,32,32] @ [16,32,8] -> [16,32,8].
    context_heads = checked_integer(
        probabilities.astype(np.int64) @ value_heads,
        np.int32, "attention context",
    )

    # [head,token,dimension] -> [1,token,hidden].
    context_merged = (
        context_heads.transpose(1, 0, 2).reshape(1, 32, 128)
    )

    # Existing XREQUANT mode-2 conversion in the C implementation.
    converted = rq_checked(
        context_merged,
        params["context_multiplier"],
        params["context_shift"],
        "context requantization",
    )
    unbounded = checked_integer(
        converted.astype(object) + params["context_output_zero_point"],
        np.int64, "context output zero point",
    )

    return {
        "context_heads": context_heads,
        "context_merged": context_merged,
        "output": np.clip(unbounded, -128, 127).astype(np.int8),
        "saturation_count": int(np.count_nonzero(
            (unbounded < -128) | (unbounded > 127)
        )),
    }

In [28]:
attention_trials = {i: [] for i in range(8)}

for block_index in range(8):
    params = attention_params[block_index]

    for softmax_trial in softmax_trials[block_index]:
        score_trial = softmax_trial["source"]
        projections = score_trial["projection_trial"]["projections"]

        local = integer_attention_context(
            softmax_trial["local"]["probabilities"],
            projections["V"]["reference"],
            params,
        )
        chained = integer_attention_context(
            softmax_trial["ln1_chained"]["probabilities"],
            projections["V"]["ln1_chained"]["output"],
            params,
        )

        attention_trials[block_index].append({
            "name": softmax_trial["name"],
            "reference": score_trial["reference"],
            "local": local,
            "ln1_chained": chained,
        })

    print(f"Completed attention trials: block {block_index}", flush=True)

Completed attention trials: block 0
Completed attention trials: block 1
Completed attention trials: block 2
Completed attention trials: block 3
Completed attention trials: block 4
Completed attention trials: block 5
Completed attention trials: block 6
Completed attention trials: block 7


In [29]:
attention_metrics = {}

rows = [
    "| Block | Cases | Local output agreement | Max code diff | "
    "Chained output agreement | Max code diff | "
    "Chained context relative L2 | Saturations local / chained |",
    "|---:|---|---:|---:|---:|---:|---:|---|",
]

for block_index in range(8):
    params = attention_params[block_index]

    for group_name, names in (
        ("Benchmark", {"benchmark"}),
        ("Calibration", {case["name"] for case in calibration_cases}),
    ):
        selected = [
            trial for trial in attention_trials[block_index]
            if trial["name"] in names
        ]

        reference_output = np.concatenate([
            trial["reference"]["output"].reshape(-1)
            for trial in selected
        ]).astype(np.int32)

        reference_context = np.concatenate([
            trial["reference"]["context"].reshape(-1)
            for trial in selected
        ]).astype(np.float64)

        metrics = {}

        for path in ("local", "ln1_chained"):
            actual_output = np.concatenate([
                trial[path]["output"].reshape(-1)
                for trial in selected
            ]).astype(np.int32)

            difference = actual_output - reference_output

            represented_context = np.concatenate([
                trial[path]["context_heads"].reshape(-1)
                for trial in selected
            ]).astype(np.float64) * params["context_scale"]

            metrics[path] = {
                "agreement": 100.0 * np.mean(difference == 0),
                "max_code_difference": int(np.max(np.abs(difference))),
                "saturation_count": sum(
                    trial[path]["saturation_count"]
                    for trial in selected
                ),
                "context": error_metrics(
                    represented_context, reference_context
                ),
            }

        attention_metrics[(block_index, group_name)] = metrics
        local = metrics["local"]
        chained = metrics["ln1_chained"]

        rows.append(
            f"| {block_index} | {group_name} | "
            f"{local['agreement']:.3f}% | "
            f"{local['max_code_difference']} | "
            f"{chained['agreement']:.3f}% | "
            f"{chained['max_code_difference']} | "
            f"{chained['context']['relative_l2_percent']:.6g}% | "
            f"{local['saturation_count']} / "
            f"{chained['saturation_count']} |"
        )

display(Markdown("\n".join(rows)))
print("PASS: context trials completed without checked arithmetic overflow.")

| Block | Cases | Local output agreement | Max code diff | Chained output agreement | Max code diff | Chained context relative L2 | Saturations local / chained |
|---:|---|---:|---:|---:|---:|---:|---|
| 0 | Benchmark | 100.000% | 0 | 99.243% | 1 | 0.169022% | 0 / 0 |
| 0 | Calibration | 99.955% | 1 | 98.839% | 1 | 0.181437% | 1 / 1 |
| 1 | Benchmark | 99.976% | 1 | 94.360% | 4 | 0.764666% | 0 / 0 |
| 1 | Calibration | 99.969% | 1 | 93.638% | 14 | 1.40273% | 2 / 2 |
| 2 | Benchmark | 99.976% | 1 | 97.778% | 2 | 0.282877% | 0 / 0 |
| 2 | Calibration | 99.962% | 1 | 93.628% | 6 | 1.02325% | 1 / 1 |
| 3 | Benchmark | 99.951% | 1 | 95.288% | 3 | 0.570273% | 0 / 0 |
| 3 | Calibration | 99.934% | 1 | 94.821% | 5 | 0.848974% | 3 / 3 |
| 4 | Benchmark | 99.951% | 1 | 92.578% | 2 | 0.878981% | 1 / 1 |
| 4 | Calibration | 99.962% | 1 | 94.734% | 3 | 0.77523% | 1 / 1 |
| 5 | Benchmark | 99.976% | 1 | 95.044% | 2 | 0.399144% | 0 / 0 |
| 5 | Calibration | 99.965% | 1 | 92.425% | 3 | 0.656683% | 2 / 2 |
| 6 | Benchmark | 99.927% | 1 | 91.064% | 4 | 0.739165% | 0 / 0 |
| 6 | Calibration | 99.941% | 1 | 94.207% | 2 | 0.593557% | 1 / 1 |
| 7 | Benchmark | 99.951% | 1 | 95.020% | 3 | 0.548706% | 0 / 0 |
| 7 | Calibration | 99.941% | 1 | 95.396% | 3 | 0.472523% | 1 / 1 |

PASS: context trials completed without checked arithmetic overflow.


## Part 4 — Output projection and first residual, shared across blocks 0–7

### Input, output, and parameters

- Attention input: INT8 `[1,32,128]`.
- Original block input: INT8 `[1,32,128]`, scale `2^-F`, zero point `0`.
- Weights: INT8 `[128,128]`, stored as `[output,input]`, zero point `0`.
- Bias: saved INT32 `[128]`, at the projection accumulator scale.
- Final residual output: INT32 `[1,32,128]`, scale `2^-16`, zero point `0`.

Each block supplies its own weights, bias, quantization metadata, and multipliers.

### 1. Projection accumulation and bias

$$
A[t,o]
=
\sum_{k=0}^{127}(q_{\mathrm{attention}}[t,k]-z_{\mathrm{attention}})
q_W[o,k]
+b_q[o]
$$

The accumulator scale is:

$$
s_A=s_{\mathrm{attention}}s_W
$$

- XTILE computes the dot products.
- Software adds the bias before requantization.
- Products, partial sums, and bias addition must fit INT32.
- No requantization occurs inside the dot product.

### 2. Preserve the INT8 projection output

Prepare the multiplier offline:

$$
M_P=\operatorname{round}_{even}
\left(\frac{s_A}{s_P}2^{39}\right)
$$

**Requantization 1 — after accumulation and bias:**

$$
q_P=
\operatorname{clip}_{[-128,127]}
\left(RQ(A,M_P,39)+z_P\right)
$$

Here `s_P` and `z_P` are the saved TFLite projection-output quantization.

Use an INT64 multiplier and checked INT64 product/rounding arithmetic.
The C implementation uses XREQUANT mode 2.

### 3. Convert the projection branch to the residual scale

Center the INT8 projection output:

$$
c_P=q_P-z_P
$$

Prepare offline:

$$
M_R=\operatorname{round}_{even}
\left(s_P\,2^{16}\,2^{22}\right)
$$

**Requantization 2 — before residual addition:**

$$
q_{\mathrm{branch}}=RQ(c_P,M_R,22)
$$

Store as INT32 at scale `2^-16`, zero point `0`.

This conversion does not saturate to INT8.
The result must fit INT32.
The C implementation uses XREQUANT mode 3.

### 4. Align the original block input exactly

$$
q_{\mathrm{aligned}}=q_{\mathrm{input}}2^{16-F}
$$

| Blocks | Input scale | Exact alignment |
|---|---|---:|
| 0, 2–7 | `2^-8` | 256 |
| 1 | `2^-7` | 512 |

This multiplication is exact; it introduces no rounding.

### 5. Add the residual branches

Both branches now use scale `2^-16`:

$$
q_R=q_{\mathrm{branch}}+q_{\mathrm{aligned}}
$$

Check the sum before INT32 storage.

There is no additional requantization or INT8 saturation here.
Retain `q_R` for:
- Part 5: second LayerNorm.
- Part 7: final residual connection.

### Rounding rule

$$
RQ(a,M,r)=(aM+2^{r-1})\gg r
$$

Runtime rounding is nearest, with ties toward positive infinity.
Offline multiplier preparation uses nearest, ties to even.

### Validation

Local path:
reference attention output → integer Part 4.

Chained path:
integer LN1 → Q/K/V → attention → Part 4.

Both paths use the same encoded original block input for the residual.

Current results across eight blocks and eight cases per block:

- Local projection: 262,144 outputs, zero mismatches.
- Local residual RMSE: approximately 4.4e-6.
- Chained projection maximum difference: five INT8 codes.
- Chained residual relative L2: approximately 0.372%–1.446%.
- Parameter and checked arithmetic tests completed without overflow.

These are within-block stage comparisons.
Complete multi-block integer inference validation remains pending.

In [30]:
out_residual_params = {}

rows = [
    "| Block | Projection multiplier / shift | Projection output ZP | "
    "Residual-branch multiplier / shift | Input alignment |",
    "|---:|---|---:|---|---:|",
]

for block_index in range(8):
    model = block_models[block_index]
    interpreter = model["interpreter"]
    tensors = model["tensors"]
    op = model["ops"][40]

    assert op["op_name"] == "FULLY_CONNECTED"
    inputs = [int(v) for v in op["inputs"] if int(v) >= 0]
    assert len(inputs) == 3
    input_id, weight_id, bias_id = inputs
    output_id = int(op["outputs"][0])

    input_scale, input_zp = tensor_int8_quant(tensors[input_id])
    output_scale, output_zp = tensor_int8_quant(tensors[output_id])

    weights = interpreter.get_tensor(weight_id).copy()
    bias = interpreter.get_tensor(bias_id).copy()
    assert weights.dtype == np.int8 and weights.shape == (128, 128)
    assert bias.dtype == np.int32 and bias.shape == (128,)

    wq = tensors[weight_id]["quantization_parameters"]
    weight_scales = np.asarray(wq["scales"], dtype=np.float64)
    weight_zps = np.asarray(wq["zero_points"], dtype=np.int64)

    assert weight_scales.size in (1, 128)
    assert weight_zps.size in (1, 128)
    assert np.all(np.isfinite(weight_scales))
    assert np.all(weight_scales > 0)
    assert np.all(weight_scales == weight_scales[0])
    assert np.all(weight_zps == 0)
    if weight_scales.size > 1 or weight_zps.size > 1:
        assert int(wq["quantized_dimension"]) == 0

    weight_scale = float(weight_scales[0])

    bq = tensors[bias_id]["quantization_parameters"]
    assert len(bq["scales"]) in (1, 128)
    assert len(bq["zero_points"]) in (1, 128)
    assert np.all(np.asarray(bq["zero_points"]) == 0)
    assert np.allclose(
        np.asarray(bq["scales"], dtype=np.float64),
        input_scale * weight_scale,
        rtol=1e-5, atol=0,
    )
    if len(bq["scales"]) > 1 or len(bq["zero_points"]) > 1:
        assert int(bq["quantized_dimension"]) == 0

    multiplier = int(np.rint(
        input_scale * weight_scale / output_scale * (1 << 39)
    ))
    branch_multiplier = int(np.rint(
        output_scale * (1 << 16) * (1 << 22)
    ))

    boundary = block_boundary_specs[block_index]["input"]
    assert boundary["zero_point"] == 0
    input_frac_bits = -int(boundary["scale_exponent"])
    assert input_frac_bits in (7, 8)
    input_alignment = 1 << (16 - input_frac_bits)

    input_bound = max(abs(-128 - input_zp), abs(127 - input_zp))
    accumulator_bounds = (
        input_bound * np.abs(weights.astype(np.int64)).sum(axis=1)
        + np.abs(bias.astype(np.int64))
    )
    assert np.all(accumulator_bounds <= np.iinfo(np.int32).max)

    assert 0 < multiplier <= np.iinfo(np.int64).max
    assert (
        (1 << 31) * multiplier + (1 << 38)
        <= np.iinfo(np.int64).max
    )

    centered_output_bound = max(
        abs(-128 - output_zp), abs(127 - output_zp)
    )
    assert 0 < branch_multiplier <= np.iinfo(np.int64).max
    assert (
        centered_output_bound * branch_multiplier + (1 << 21)
        <= np.iinfo(np.int64).max
    )

    branch_bound = (
        centered_output_bound * branch_multiplier + (1 << 21)
    ) >> 22
    assert (
        branch_bound + 128 * input_alignment
        <= np.iinfo(np.int32).max
    )

    out_residual_params[block_index] = {
        "input_tensor": input_id,
        "output_tensor": output_id,
        "residual_tensor": int(model["ops"][42]["outputs"][0]),
        "input_zero_point": input_zp,
        "weights": weights,
        "bias": bias,
        "multiplier": multiplier,
        "shift": 39,
        "output_scale": output_scale,
        "output_zero_point": output_zp,
        "branch_multiplier": branch_multiplier,
        "branch_shift": 22,
        "input_alignment": input_alignment,
    }

    rows.append(
        f"| {block_index} | {multiplier} / 39 | {output_zp} | "
        f"{branch_multiplier} / 22 | {input_alignment} |"
    )

display(Markdown("\n".join(rows)))
print("PASS: Part 4 parameter and arithmetic-bound checks completed.")

| Block | Projection multiplier / shift | Projection output ZP | Residual-branch multiplier / shift | Input alignment |
|---:|---|---:|---|---:|
| 0 | 747156156 / 39 | -13 | 939062400 / 22 | 256 |
| 1 | 1418659633 / 39 | 31 | 893601216 / 22 | 512 |
| 2 | 1849292849 / 39 | 7 | 375301408 / 22 | 256 |
| 3 | 2204147194 / 39 | 2 | 315659392 / 22 | 256 |
| 4 | 2329300601 / 39 | 5 | 257761440 / 22 | 256 |
| 5 | 1441446675 / 39 | 9 | 231581552 / 22 | 256 |
| 6 | 1365701654 / 39 | -6 | 220068672 / 22 | 256 |
| 7 | 1390383334 / 39 | 45 | 535577664 / 22 | 256 |

PASS: Part 4 parameter and arithmetic-bound checks completed.


In [31]:
def integer_projection(input_codes, params):
    output_features, input_features = params["weights"].shape

    if (
        input_codes.dtype != np.int8
        or input_codes.shape != (1, 32, input_features)
    ):
        raise ValueError("Unexpected INT8 projection input")

    centered = (
        input_codes.reshape(32, input_features).astype(np.int64)
        - params["input_zero_point"]
    )

    accumulator = checked_integer(
        centered @ params["weights"].astype(np.int64).T,
        np.int32, "projection dot product",
    )

    if params.get("bias") is not None:
        accumulator = checked_integer(
            accumulator.astype(object) + params["bias"].astype(object),
            np.int32, "projection bias addition",
        )

    converted = rq_checked(
        accumulator, params["multiplier"], params["shift"],
        "projection requantization",
    )
    unbounded = checked_integer(
        converted.astype(object) + params["output_zero_point"],
        np.int64, "projection zero point",
    )

    return {
        "accumulator": accumulator.reshape(1, 32, output_features),
        "output": np.clip(unbounded, -128, 127).astype(np.int8).reshape(
            1, 32, output_features
        ),
        "saturation_count": int(np.count_nonzero(
            (unbounded < -128) | (unbounded > 127)
        )),
    }


def integer_out_residual(attention_output, input_codes, params):
    if input_codes.dtype != np.int8 or input_codes.shape != (1, 32, 128):
        raise ValueError("Residual input must be INT8 [1,32,128]")

    projection = integer_projection(attention_output, params)

    # Center the INT8 projection output, then convert into scale 2^-16.
    centered = (
        projection["output"].astype(object)
        - params["output_zero_point"]
    )
    branch = checked_integer(
        rq_checked(
            centered,
            params["branch_multiplier"],
            params["branch_shift"],
            "residual branch",
        ),
        np.int32, "stored residual branch",
    )

    # Exact alignment of the original block input.
    aligned_input = checked_integer(
        input_codes.astype(object) * params["input_alignment"],
        np.int32, "aligned block input",
    )

    # Both branches now use scale 2^-16.
    output = checked_integer(
        branch.astype(object) + aligned_input.astype(object),
        np.int32, "first residual sum",
    )

    return {
        "projection": projection,
        "branch": branch,
        "output": output,
    }

In [32]:
out_residual_trials = {i: [] for i in range(8)}

for block_index in range(8):
    model = block_models[block_index]
    interpreter = model["interpreter"]
    params = out_residual_params[block_index]
    input_scale = block_boundary_specs[block_index]["input"]["scale"]

    assert len(attention_trials[block_index]) == len(ln1_trials[block_index])

    for attention_trial, ln1_trial in zip(
        attention_trials[block_index], ln1_trials[block_index]
    ):
        assert attention_trial["name"] == ln1_trial["name"]

        input_codes = ln1_trial["input_codes"]
        represented = (
            input_codes.astype(np.float64) * input_scale
        ).astype(np.float32)

        run_tflite_block(block_index, represented)

        np.testing.assert_array_equal(
            interpreter.get_tensor(params["input_tensor"]),
            attention_trial["reference"]["output"],
        )

        reference_projection = interpreter.get_tensor(
            params["output_tensor"]
        ).copy()
        reference_residual = interpreter.get_tensor(
            params["residual_tensor"]
        ).copy()

        out_residual_trials[block_index].append({
            "name": attention_trial["name"],
            "reference_projection": reference_projection,
            "reference_residual": reference_residual,
            "local": integer_out_residual(
                attention_trial["reference"]["output"],
                input_codes,
                params,
            ),
            "ln1_chained": integer_out_residual(
                attention_trial["ln1_chained"]["output"],
                input_codes,
                params,
            ),
        })

    print(f"Completed Part 4 trials: block {block_index}", flush=True)

Completed Part 4 trials: block 0
Completed Part 4 trials: block 1
Completed Part 4 trials: block 2
Completed Part 4 trials: block 3
Completed Part 4 trials: block 4
Completed Part 4 trials: block 5
Completed Part 4 trials: block 6
Completed Part 4 trials: block 7


In [33]:
out_residual_metrics = {}

rows = [
    "| Block | Cases | Local projection agreement | Local residual RMSE | "
    "Chained projection agreement | Chained max code diff | "
    "Chained residual RMSE | Chained residual relative L2 |",
    "|---:|---|---:|---:|---:|---:|---:|---:|",
]

for block_index in range(8):
    for group_name, names in (
        ("Benchmark", {"benchmark"}),
        ("Calibration", {case["name"] for case in calibration_cases}),
    ):
        selected = [
            trial for trial in out_residual_trials[block_index]
            if trial["name"] in names
        ]

        reference_projection = np.concatenate([
            trial["reference_projection"].reshape(-1)
            for trial in selected
        ]).astype(np.int32)

        reference_residual = np.concatenate([
            trial["reference_residual"].reshape(-1)
            for trial in selected
        ]).astype(np.float64)

        metrics = {}

        for path in ("local", "ln1_chained"):
            projection = np.concatenate([
                trial[path]["projection"]["output"].reshape(-1)
                for trial in selected
            ]).astype(np.int32)

            difference = projection - reference_projection

            represented_residual = np.concatenate([
                trial[path]["output"].reshape(-1)
                for trial in selected
            ]).astype(np.float64) * (2.0 ** -16)

            metrics[path] = {
                "projection_agreement": 100.0 * np.mean(difference == 0),
                "max_code_difference": int(np.max(np.abs(difference))),
                "projection_saturations": sum(
                    trial[path]["projection"]["saturation_count"]
                    for trial in selected
                ),
                "residual": error_metrics(
                    represented_residual, reference_residual
                ),
            }

        out_residual_metrics[(block_index, group_name)] = metrics
        local = metrics["local"]
        chained = metrics["ln1_chained"]

        rows.append(
            f"| {block_index} | {group_name} | "
            f"{local['projection_agreement']:.3f}% | "
            f"{local['residual']['rmse']:.6g} | "
            f"{chained['projection_agreement']:.3f}% | "
            f"{chained['max_code_difference']} | "
            f"{chained['residual']['rmse']:.6g} | "
            f"{chained['residual']['relative_l2_percent']:.6g}% |"
        )

display(Markdown("\n".join(rows)))
print("PASS: Part 4 trials completed without checked arithmetic overflow.")

| Block | Cases | Local projection agreement | Local residual RMSE | Chained projection agreement | Chained max code diff | Chained residual RMSE | Chained residual relative L2 |
|---:|---|---:|---:|---:|---:|---:|---:|
| 0 | Benchmark | 100.000% | 4.35984e-06 | 98.364% | 1 | 0.000437057 | 0.372255% |
| 0 | Calibration | 100.000% | 4.43463e-06 | 97.625% | 1 | 0.00052653 | 0.456661% |
| 1 | Benchmark | 100.000% | 4.44524e-06 | 86.646% | 2 | 0.00120088 | 1.20197% |
| 1 | Calibration | 100.000% | 4.40967e-06 | 85.564% | 5 | 0.00141423 | 1.44628% |
| 2 | Benchmark | 100.000% | 4.39733e-06 | 92.212% | 1 | 0.000381074 | 0.440004% |
| 2 | Calibration | 100.000% | 4.40015e-06 | 81.822% | 4 | 0.000650886 | 0.767806% |
| 3 | Benchmark | 100.000% | 4.40546e-06 | 83.228% | 2 | 0.000472475 | 0.648525% |
| 3 | Calibration | 100.000% | 4.42277e-06 | 83.772% | 4 | 0.000496103 | 0.688827% |
| 4 | Benchmark | 100.000% | 4.36276e-06 | 74.341% | 2 | 0.000479136 | 0.700093% |
| 4 | Calibration | 100.000% | 4.39574e-06 | 81.079% | 3 | 0.000416015 | 0.61833% |
| 5 | Benchmark | 100.000% | 4.40103e-06 | 86.133% | 1 | 0.000313819 | 0.468834% |
| 5 | Calibration | 100.000% | 4.39437e-06 | 82.949% | 2 | 0.00034816 | 0.523128% |
| 6 | Benchmark | 100.000% | 4.3416e-06 | 81.787% | 2 | 0.000343017 | 0.52022% |
| 6 | Calibration | 100.000% | 4.40344e-06 | 85.812% | 2 | 0.000301777 | 0.457634% |
| 7 | Benchmark | 100.000% | 4.38399e-06 | 92.212% | 1 | 0.000543705 | 0.873333% |
| 7 | Calibration | 100.000% | 4.42076e-06 | 92.644% | 1 | 0.00052843 | 0.806768% |

PASS: Part 4 trials completed without checked arithmetic overflow.


## Part 5 — Integer LN2, shared across blocks 0–7

### Input, output, and parameters

- Input: first residual, INT32 `[1,32,128]`, scale `2^-16`, zero point `0`.
- Output: INT8 `[1,32,128]`, using the block’s saved LN2 output scale and zero point.
- Gamma/beta: independent LN2 parameters, encoded offline as INT32 at scale `2^-16`.
- Reuse the shared integer LayerNorm algorithm.

Common settings:

| Parameter | Value |
|---|---|
| Feature count | 128 |
| Square conversion shift | 23 |
| Normalization conversion shift | 23 |
| Variance scale | `2^-30` |
| Integer epsilon | 10737 |
| Reciprocal-square-root numerator | `2^62` |
| Inverse-standard-deviation scale | `2^-16` |
| Final conversion shift | 47 |

### 1. Mean and centered differences

For each token:

$$
m=\sum_{k=0}^{127}q_R[k]
$$

Store the sum as INT32 at scale `2^-23`, representing the mean.

Align and subtract:

$$
d[k]=128q_R[k]-m
$$

Compute using INT64 at scale `2^-23`.

### 2. Squared differences and variance

Compute `d²` using INT64 at scale `2^-46`.

**Requantization 1 — after squaring:**

$$
u[k]=RQ(d[k]^2,1,23)
$$

Store `u` as INT32 at scale `2^-23`.

Sum over features:

$$
V=\sum_k u[k]
$$

Store `V` as INT32 at scale `2^-30`, representing variance.

### 3. Inverse standard deviation

$$
I=\operatorname{isqrt}
\left(
\left\lfloor\frac{2^{62}}{V+10737}\right\rfloor
\right)
$$

The denominator must be positive.
Store `I` as INT32 at scale `2^-16`.

### 4. Normalize

Compute `d × I` using INT64 at scale `2^-39`.

**Requantization 2 — after normalization:**

$$
n[k]=RQ(d[k]I,1,23)
$$

Store `n` as INT32 at scale `2^-16`.

### 5. Apply gamma and beta

Offline encoding:

$$
\gamma_q=\operatorname{round}_{even}(\gamma2^{16}),
\qquad
\beta_q=\operatorname{round}_{even}(\beta2^{16})
$$

Runtime affine calculation:

$$
a[k]=n[k]\gamma_q[k]+\beta_q[k]2^{16}
$$

The affine result uses INT64 at scale `2^-32`.
Beta alignment is exact.

### 6. Convert to the INT8 MLP input

Read `s_out` and `z_out` from the block’s saved LN2 output metadata.

Prepare offline:

$$
M=\operatorname{round}_{even}
\left(\frac{2^{-32}}{s_{\mathrm{out}}}2^{47}\right)
$$

**Requantization 3 — after the affine calculation:**

$$
q_{\mathrm{out}}=
\operatorname{clip}_{[-128,127]}
\left(RQ(a,M,47)+z_{\mathrm{out}}\right)
$$

This output feeds Part 6’s MLP expansion.
Retain the original first residual separately for Part 7.

### Arithmetic rules

$$
RQ(a,M,r)=(aM+2^{r-1})\gg r
$$

- Runtime rounding: nearest, ties toward positive infinity.
- Offline parameter encoding: nearest, ties to even.
- Check all intended INT32/INT64 results before storing.
- No floating-point arithmetic is required at runtime.

### Validation

Local:
encode the reference FP32 residual at scale `2^-16`, then run integer LN2.
This includes residual-encoding error and LN2 approximation error.

Chained:
integer Parts 1–4 → integer LN2, without re-encoding the residual.

Current eight-case results across all blocks:

- Local agreement: 99.658%–99.927%.
- Local maximum difference: one INT8 code.
- Chained agreement: 83.698%–97.607%.
- Chained maximum difference: six INT8 codes.
- Saturation counts differ by one value in two calibration groups.
- Checked arithmetic completed without overflow.

Complete integer block and multi-block inference validation remain pending.

In [35]:
for block_index in range(8):
    for trial in ln1_trials[block_index]:
        repeated = integer_ln1(
            trial["input_codes"], ln1_params[block_index]
        )
        np.testing.assert_array_equal(
            repeated["output"], trial["integer"]["output"]
        )

print("PASS: shared LayerNorm preserves all previous LN1 outputs.")

ln2_params = {}

rows = [
    "| Block | Square / normalize shift | Integer epsilon | "
    "RSQRT numerator | Output multiplier / shift | Output ZP |",
    "|---:|---:|---:|---|---|---:|",
]

for block_index in range(8):
    model = block_models[block_index]
    interpreter = model["interpreter"]
    ops = model["ops"]

    assert ops[43]["op_name"] == "MEAN"
    assert ops[55]["op_name"] == "MUL"
    assert ops[56]["op_name"] == "ADD"
    assert ops[57]["op_name"] == "QUANTIZE"

    gamma_id = int(ops[55]["inputs"][1])
    beta_id = int(ops[56]["inputs"][1])
    epsilon_id = int(ops[47]["inputs"][1])
    input_id = int(ops[43]["inputs"][0])
    output_id = int(ops[57]["outputs"][0])

    assert input_id == out_residual_params[block_index]["residual_tensor"]

    gamma = interpreter.get_tensor(gamma_id).astype(np.float64).reshape(128)
    beta = interpreter.get_tensor(beta_id).astype(np.float64).reshape(128)
    epsilon = float(interpreter.get_tensor(epsilon_id).item())

    assert np.all(np.isfinite(gamma))
    assert np.all(np.isfinite(beta))
    assert math.isfinite(epsilon) and epsilon > 0

    output_scale, output_zp = tensor_int8_quant(
        model["tensors"][output_id]
    )

    params = {
        "input_frac_bits": 16,
        "square_shift": 23,
        "normalize_shift": 23,
        "epsilon_q": int(np.rint(epsilon * (1 << 30))),
        "rsqrt_numerator": 1 << 62,
        "gamma": checked_integer(
            np.rint(gamma * (1 << 16)), np.int32, "LN2 gamma"
        ),
        "beta": checked_integer(
            np.rint(beta * (1 << 16)), np.int32, "LN2 beta"
        ),
        "output_scale": output_scale,
        "output_zero_point": output_zp,
        "output_multiplier": int(np.rint(
            (2.0 ** -32 / output_scale) * (1 << 47)
        )),
        "output_shift": 47,
        "input_tensor": input_id,
        "output_tensor": output_id,
    }

    assert 0 < params["epsilon_q"] <= np.iinfo(np.int32).max
    assert 0 < params["output_multiplier"] <= np.iinfo(np.int64).max

    ln2_params[block_index] = params

    rows.append(
        f"| {block_index} | 23 | {params['epsilon_q']} | "
        f"2^62 | {params['output_multiplier']} / 47 | {output_zp} |"
    )

display(Markdown("\n".join(rows)))

PASS: shared LayerNorm preserves all previous LN1 outputs.


| Block | Square / normalize shift | Integer epsilon | RSQRT numerator | Output multiplier / shift | Output ZP |
|---:|---:|---:|---|---|---:|
| 0 | 23 | 10737 | 2^62 | 1009162 / 47 | 2 |
| 1 | 23 | 10737 | 2^62 | 862727 / 47 | -1 |
| 2 | 23 | 10737 | 2^62 | 885999 / 47 | 4 |
| 3 | 23 | 10737 | 2^62 | 903830 / 47 | -1 |
| 4 | 23 | 10737 | 2^62 | 815738 / 47 | 5 |
| 5 | 23 | 10737 | 2^62 | 849844 / 47 | -4 |
| 6 | 23 | 10737 | 2^62 | 860729 / 47 | -7 |
| 7 | 23 | 10737 | 2^62 | 772621 / 47 | 11 |

In [36]:
ln2_trials = {i: [] for i in range(8)}

for block_index in range(8):
    model = block_models[block_index]
    interpreter = model["interpreter"]
    params = ln2_params[block_index]
    input_scale = block_boundary_specs[block_index]["input"]["scale"]

    assert len(out_residual_trials[block_index]) == len(
        ln1_trials[block_index]
    )

    for residual_trial, ln1_trial in zip(
        out_residual_trials[block_index], ln1_trials[block_index]
    ):
        assert residual_trial["name"] == ln1_trial["name"]

        represented_input = (
            ln1_trial["input_codes"].astype(np.float64) * input_scale
        ).astype(np.float32)

        run_tflite_block(block_index, represented_input)

        reference_residual = interpreter.get_tensor(
            params["input_tensor"]
        ).copy()
        np.testing.assert_array_equal(
            reference_residual,
            residual_trial["reference_residual"],
        )

        reference_output = interpreter.get_tensor(
            params["output_tensor"]
        ).copy()

        # Reference encoding for the isolated LN2 test only.
        local_input = checked_integer(
            np.rint(reference_residual.astype(np.float64) * (1 << 16)),
            np.int32, "encoded reference residual",
        )
        encoding_error = float(np.max(np.abs(
            local_input.astype(np.float64) * (2.0 ** -16)
            - reference_residual.astype(np.float64)
        )))

        ln2_trials[block_index].append({
            "name": residual_trial["name"],
            "reference_output": reference_output,
            "local_input": local_input,
            "local_input_encoding_error": encoding_error,
            "local": integer_ln2(local_input, params),
            "ln1_chained": integer_ln2(
                residual_trial["ln1_chained"]["output"],
                params,
            ),
        })

    print(f"Completed LN2 trials: block {block_index}", flush=True)

Completed LN2 trials: block 0
Completed LN2 trials: block 1
Completed LN2 trials: block 2
Completed LN2 trials: block 3
Completed LN2 trials: block 4
Completed LN2 trials: block 5
Completed LN2 trials: block 6
Completed LN2 trials: block 7


In [37]:
ln2_metrics = {}

rows = [
    "| Block | Cases | Local agreement | Max code diff | "
    "Chained agreement | Max code diff | Saturations local / chained |",
    "|---:|---|---:|---:|---:|---:|---|",
]

for block_index in range(8):
    for group_name, names in (
        ("Benchmark", {"benchmark"}),
        ("Calibration", {case["name"] for case in calibration_cases}),
    ):
        selected = [
            trial for trial in ln2_trials[block_index]
            if trial["name"] in names
        ]

        reference = np.concatenate([
            trial["reference_output"].reshape(-1)
            for trial in selected
        ]).astype(np.int32)

        metrics = {}

        for path in ("local", "ln1_chained"):
            actual = np.concatenate([
                trial[path]["output"].reshape(-1)
                for trial in selected
            ]).astype(np.int32)

            difference = actual - reference

            metrics[path] = {
                "agreement": 100.0 * np.mean(difference == 0),
                "max_code_difference": int(np.max(np.abs(difference))),
                "saturation_count": sum(
                    trial[path]["saturation_count"]
                    for trial in selected
                ),
            }

        ln2_metrics[(block_index, group_name)] = metrics
        local = metrics["local"]
        chained = metrics["ln1_chained"]

        rows.append(
            f"| {block_index} | {group_name} | "
            f"{local['agreement']:.3f}% | "
            f"{local['max_code_difference']} | "
            f"{chained['agreement']:.3f}% | "
            f"{chained['max_code_difference']} | "
            f"{local['saturation_count']} / "
            f"{chained['saturation_count']} |"
        )

display(Markdown("\n".join(rows)))

print(
    "Maximum local residual encoding error:",
    max(
        trial["local_input_encoding_error"]
        for trials in ln2_trials.values()
        for trial in trials
    ),
)
print("PASS: LN2 trials completed without checked arithmetic overflow.")

| Block | Cases | Local agreement | Max code diff | Chained agreement | Max code diff | Saturations local / chained |
|---:|---|---:|---:|---:|---:|---|
| 0 | Benchmark | 99.902% | 1 | 97.607% | 1 | 0 / 0 |
| 0 | Calibration | 99.909% | 1 | 96.634% | 2 | 2 / 2 |
| 1 | Benchmark | 99.854% | 1 | 84.644% | 3 | 0 / 0 |
| 1 | Calibration | 99.878% | 1 | 83.698% | 6 | 1 / 1 |
| 2 | Benchmark | 99.854% | 1 | 94.556% | 1 | 0 / 0 |
| 2 | Calibration | 99.822% | 1 | 88.299% | 2 | 2 / 2 |
| 3 | Benchmark | 99.780% | 1 | 89.160% | 1 | 0 / 0 |
| 3 | Calibration | 99.819% | 1 | 89.394% | 3 | 1 / 1 |
| 4 | Benchmark | 99.927% | 1 | 86.816% | 1 | 1 / 1 |
| 4 | Calibration | 99.773% | 1 | 90.196% | 2 | 1 / 1 |
| 5 | Benchmark | 99.805% | 1 | 92.847% | 1 | 0 / 0 |
| 5 | Calibration | 99.822% | 1 | 91.253% | 1 | 0 / 1 |
| 6 | Benchmark | 99.927% | 1 | 91.309% | 1 | 1 / 1 |
| 6 | Calibration | 99.833% | 1 | 92.787% | 1 | 2 / 2 |
| 7 | Benchmark | 99.658% | 1 | 90.967% | 2 | 0 / 0 |
| 7 | Calibration | 99.822% | 1 | 91.643% | 2 | 4 / 3 |

Maximum local residual encoding error: 7.62939453125e-06
PASS: LN2 trials completed without checked arithmetic overflow.


## Part 6 — MLP expansion and GELU, shared across blocks 0–7

### Input and output

- Input: LN2 output, INT8 `[1,32,128]`.
- Expansion: INT8 weights `[512,128]`, INT32 bias.
- Expansion and GELU outputs: INT8 `[1,32,512]`.
- Read each block’s constants, scales, and zero points from its saved TFLite.

### Expansion

$$
A[t,o]=\sum_k(q_x[t,k]-z_x)q_W[o,k]+b_q[o]
$$

XTILE computes the dot products; software adds bias.

**Requantize after accumulation and bias:**

$$
q_U=\operatorname{clip}_{INT8}(RQ(A,M_U,39)+z_U)
$$

$$
M_U=\operatorname{round}_{even}
\left(\frac{s_xs_W}{s_U}2^{39}\right)
$$

### GELU sequence and conversion points

Use saved quantized coefficients, not replacements from the conceptual floating-point GELU formula.

| Stage | Computation | Conversion |
|---|---|---|
| Half branch | Centered expansion × centered half coefficient | INT8, shift 38 |
| Cube | Centered expansion cubed in INT32 | INT8, shift 45 |
| Weighted cube | Centered cube × centered cubic coefficient | INT8, shift 38 |
| Branch sum | Multiply centered linear/cubic branches by separate scale multipliers; add in INT64 | One combined rounding, shift 31, then INT8 |
| Tanh argument | Centered sum × centered argument coefficient | INT8, shift 38 |
| Tanh | Lookup indexed by argument code + 128 | INT32 at scale `2^-24` |
| One plus tanh | Add `2^24` exactly | Preserve scale `2^-24` |
| Factor | Convert one-plus-tanh | INT8, shift 47 |
| Final product | Centered half branch × centered factor | INT32, then two-round INT8 conversion |

Standard conversion multipliers approximate the source/destination scale ratio.
Products, rounding additions, and stored results must fit their intended types.

### Tanh table

For each INT8 argument code `q`:

$$
L[q+128]=\operatorname{round}_{even}
\left(\tanh((q-z_a)s_a)2^{24}\right)
$$

Each block has a 256-entry INT32 table based on its own argument quantization.

### Final two-round conversion

Approximate the final product scale ratio as:

$$
\frac{s_{\mathrm{half}}s_{\mathrm{factor}}}{s_{\mathrm{GELU}}}
\approx M_{31}2^{-31}2^{-r}
$$

Runtime:

1. Rounded Q31 multiplication, ties toward positive infinity.
2. Rounded division by `2^r`, ties away from zero.
3. Add the output zero point and saturate to INT8.

Use the existing XREQUANT mode 4.

### Validation

Across eight blocks and eight cases per block:

- Local expansion: 1,048,576 outputs, zero mismatches.
- All five local GELU front stages match TFLite exactly.
- Local factor and final GELU outputs match TFLite exactly.
- Chained final GELU agreement: 76.581%–95.123%.
- Chained maximum difference: 18 codes in block 1 calibration.
- Final saturation counts are identical between local and chained paths.

Chained results include preceding integer stages within each block.
Complete multi-block integer inference validation remains pending.

In [38]:
def read_projection_params(block_index, op_index, expected_weight_shape):
    model = block_models[block_index]
    interpreter = model["interpreter"]
    tensors = model["tensors"]
    op = model["ops"][op_index]

    assert op["op_name"] == "FULLY_CONNECTED"
    inputs = [int(v) for v in op["inputs"] if int(v) >= 0]
    assert len(inputs) in (2, 3)
    assert len(op["outputs"]) == 1

    input_id, weight_id = inputs[:2]
    output_id = int(op["outputs"][0])

    input_scale, input_zp = tensor_int8_quant(tensors[input_id])
    output_scale, output_zp = tensor_int8_quant(tensors[output_id])

    weights = interpreter.get_tensor(weight_id).copy()
    assert weights.dtype == np.int8
    assert weights.shape == expected_weight_shape

    output_features, input_features = weights.shape
    assert tuple(tensors[input_id]["shape"]) == (1, 32, input_features)
    assert tuple(tensors[output_id]["shape"]) == (1, 32, output_features)

    wq = tensors[weight_id]["quantization_parameters"]
    weight_scales = np.asarray(wq["scales"], dtype=np.float64)
    weight_zps = np.asarray(wq["zero_points"], dtype=np.int64)

    assert weight_scales.size in (1, output_features)
    assert weight_zps.size in (1, output_features)
    assert np.all(np.isfinite(weight_scales))
    assert np.all(weight_scales > 0)
    assert np.all(weight_scales == weight_scales[0])
    assert np.all(weight_zps == 0)

    if weight_scales.size > 1 or weight_zps.size > 1:
        assert int(wq["quantized_dimension"]) == 0

    weight_scale = float(weight_scales[0])
    bias = None

    if len(inputs) == 3:
        bias_id = inputs[2]
        bias = interpreter.get_tensor(bias_id).copy()
        assert bias.dtype == np.int32
        assert bias.shape == (output_features,)

        bq = tensors[bias_id]["quantization_parameters"]
        assert len(bq["scales"]) in (1, output_features)
        assert len(bq["zero_points"]) in (1, output_features)
        assert np.all(np.asarray(bq["zero_points"]) == 0)
        assert np.allclose(
            np.asarray(bq["scales"], dtype=np.float64),
            input_scale * weight_scale,
            rtol=1e-5, atol=0,
        )

        if len(bq["scales"]) > 1 or len(bq["zero_points"]) > 1:
            assert int(bq["quantized_dimension"]) == 0

    input_bound = max(abs(-128 - input_zp), abs(127 - input_zp))
    accumulator_bounds = (
        input_bound * np.abs(weights.astype(np.int64)).sum(axis=1)
    )
    if bias is not None:
        accumulator_bounds += np.abs(bias.astype(np.int64))

    assert np.all(accumulator_bounds <= np.iinfo(np.int32).max)

    multiplier = int(np.rint(
        input_scale * weight_scale / output_scale * (1 << 39)
    ))
    assert 0 < multiplier <= np.iinfo(np.int64).max
    assert (
        (1 << 31) * multiplier + (1 << 38)
        <= np.iinfo(np.int64).max
    ), "Projection product/rounding exceeds INT64"

    return {
        "input_tensor": input_id,
        "output_tensor": output_id,
        "input_scale": input_scale,
        "input_zero_point": input_zp,
        "weights": weights,
        "weight_scale": weight_scale,
        "bias": bias,
        "output_scale": output_scale,
        "output_zero_point": output_zp,
        "multiplier": multiplier,
        "shift": 39,
    }

In [39]:
mlp_expand_params = {}

rows = [
    "| Block | Input ZP | Weight scale | Output scale | "
    "Output ZP | Multiplier / shift |",
    "|---:|---:|---:|---:|---:|---|",
]

for block_index in range(8):
    params = read_projection_params(
        block_index,
        op_index=58,
        expected_weight_shape=(512, 128),
    )

    assert params["bias"] is not None
    assert (
        params["input_tensor"]
        == ln2_params[block_index]["output_tensor"]
    )

    mlp_expand_params[block_index] = params

    rows.append(
        f"| {block_index} | {params['input_zero_point']} | "
        f"{params['weight_scale']:.12g} | "
        f"{params['output_scale']:.12g} | "
        f"{params['output_zero_point']} | "
        f"{params['multiplier']} / 39 |"
    )

display(Markdown("\n".join(rows)))
print("PASS: expansion parameter and arithmetic-bound checks completed.")

| Block | Input ZP | Weight scale | Output scale | Output ZP | Multiplier / shift |
|---:|---:|---:|---:|---:|---|
| 0 | 2 | 0.000803247734439 | 0.0112525857985 | 37 | 1274254450 / 39 |
| 1 | -1 | 0.000919356767554 | 0.0104163484648 | 29 | 1842956424 / 39 |
| 2 | 4 | 0.000966044783127 | 0.0107453726232 | 38 | 1827940586 / 39 |
| 3 | -1 | 0.000833616475575 | 0.0128157725558 | 33 | 1296446300 / 39 |
| 4 | 5 | 0.000942005601246 | 0.0117985485122 | 30 | 1763168980 / 39 |
| 5 | -4 | 0.00111625995487 | 0.0150570431724 | 48 | 1571469889 / 39 |
| 6 | -7 | 0.000881322252098 | 0.0118841156363 | 33 | 1552105132 / 39 |
| 7 | 11 | 0.000968697713688 | 0.0186460912228 | 54 | 1211305507 / 39 |

PASS: expansion parameter and arithmetic-bound checks completed.


In [40]:
gelu_reference_ops = {
    "half": 60,
    "cube": 62,
    "weighted_cube": 63,
    "sum": 64,
    "argument": 65,
    "tanh": 67,
    "factor_fine": 68,
    "factor": 69,
    "output": 70,
}

mlp_expand_trials = {i: [] for i in range(8)}

for block_index in range(8):
    model = block_models[block_index]
    interpreter = model["interpreter"]
    params = mlp_expand_params[block_index]
    input_scale = block_boundary_specs[block_index]["input"]["scale"]

    assert len(ln2_trials[block_index]) == len(ln1_trials[block_index])

    for ln2_trial, ln1_trial in zip(
        ln2_trials[block_index], ln1_trials[block_index]
    ):
        assert ln2_trial["name"] == ln1_trial["name"]

        represented_input = (
            ln1_trial["input_codes"].astype(np.float64) * input_scale
        ).astype(np.float32)

        run_tflite_block(block_index, represented_input)

        reference_input = interpreter.get_tensor(
            params["input_tensor"]
        ).copy()
        np.testing.assert_array_equal(
            reference_input, ln2_trial["reference_output"]
        )

        reference_output = interpreter.get_tensor(
            params["output_tensor"]
        ).copy()

        gelu_references = {
            name: interpreter.get_tensor(
                int(model["ops"][op_index]["outputs"][0])
            ).copy()
            for name, op_index in gelu_reference_ops.items()
        }

        mlp_expand_trials[block_index].append({
            "name": ln2_trial["name"],
            "reference_output": reference_output,
            "gelu_references": gelu_references,
            "local": integer_projection(reference_input, params),
            "ln1_chained": integer_projection(
                ln2_trial["ln1_chained"]["output"], params
            ),
        })

    print(f"Completed MLP expansion: block {block_index}", flush=True)

Completed MLP expansion: block 0
Completed MLP expansion: block 1
Completed MLP expansion: block 2
Completed MLP expansion: block 3
Completed MLP expansion: block 4
Completed MLP expansion: block 5
Completed MLP expansion: block 6
Completed MLP expansion: block 7


In [41]:
mlp_expand_metrics = {}

rows = [
    "| Block | Cases | Local agreement | Max code diff | "
    "Chained agreement | Max code diff | Saturations local / chained |",
    "|---:|---|---:|---:|---:|---:|---|",
]

for block_index in range(8):
    for group_name, names in (
        ("Benchmark", {"benchmark"}),
        ("Calibration", {case["name"] for case in calibration_cases}),
    ):
        selected = [
            trial for trial in mlp_expand_trials[block_index]
            if trial["name"] in names
        ]

        reference = np.concatenate([
            trial["reference_output"].reshape(-1)
            for trial in selected
        ]).astype(np.int32)

        metrics = {}

        for path in ("local", "ln1_chained"):
            actual = np.concatenate([
                trial[path]["output"].reshape(-1)
                for trial in selected
            ]).astype(np.int32)

            difference = actual - reference

            metrics[path] = {
                "agreement": 100.0 * np.mean(difference == 0),
                "max_code_difference": int(np.max(np.abs(difference))),
                "saturation_count": sum(
                    trial[path]["saturation_count"]
                    for trial in selected
                ),
            }

        mlp_expand_metrics[(block_index, group_name)] = metrics
        local = metrics["local"]
        chained = metrics["ln1_chained"]

        rows.append(
            f"| {block_index} | {group_name} | "
            f"{local['agreement']:.3f}% | "
            f"{local['max_code_difference']} | "
            f"{chained['agreement']:.3f}% | "
            f"{chained['max_code_difference']} | "
            f"{local['saturation_count']} / "
            f"{chained['saturation_count']} |"
        )

display(Markdown("\n".join(rows)))
print("PASS: expansion trials completed without checked arithmetic overflow.")

| Block | Cases | Local agreement | Max code diff | Chained agreement | Max code diff | Saturations local / chained |
|---:|---|---:|---:|---:|---:|---|
| 0 | Benchmark | 100.000% | 0 | 94.611% | 1 | 1 / 1 |
| 0 | Calibration | 100.000% | 0 | 92.256% | 1 | 4 / 4 |
| 1 | Benchmark | 100.000% | 0 | 71.332% | 4 | 0 / 0 |
| 1 | Calibration | 100.000% | 0 | 73.248% | 9 | 5 / 5 |
| 2 | Benchmark | 100.000% | 0 | 87.000% | 2 | 1 / 1 |
| 2 | Calibration | 100.000% | 0 | 78.782% | 3 | 6 / 6 |
| 3 | Benchmark | 100.000% | 0 | 82.727% | 2 | 0 / 0 |
| 3 | Calibration | 100.000% | 0 | 82.925% | 3 | 4 / 4 |
| 4 | Benchmark | 100.000% | 0 | 75.031% | 2 | 2 / 2 |
| 4 | Calibration | 100.000% | 0 | 80.281% | 2 | 12 / 12 |
| 5 | Benchmark | 100.000% | 0 | 85.516% | 1 | 2 / 2 |
| 5 | Calibration | 100.000% | 0 | 84.474% | 2 | 3 / 3 |
| 6 | Benchmark | 100.000% | 0 | 80.566% | 2 | 1 / 1 |
| 6 | Calibration | 100.000% | 0 | 82.326% | 2 | 9 / 9 |
| 7 | Benchmark | 100.000% | 0 | 88.281% | 2 | 1 / 1 |
| 7 | Calibration | 100.000% | 0 | 86.610% | 2 | 6 / 6 |

PASS: expansion trials completed without checked arithmetic overflow.


In [42]:
def make_gelu_conversion(model, output_id, source_scale, source_bound, shift):
    output_scale, output_zp = tensor_int8_quant(
        model["tensors"][output_id]
    )

    assert math.isfinite(source_scale) and source_scale > 0
    assert 0 <= source_bound <= np.iinfo(np.int32).max

    multiplier = int(np.rint(
        source_scale / output_scale * (1 << shift)
    ))

    assert 0 < multiplier <= np.iinfo(np.int64).max
    assert (
        source_bound * multiplier + (1 << (shift - 1))
        <= np.iinfo(np.int64).max
    ), "GELU product/rounding exceeds INT64"

    return {
        "output_tensor": output_id,
        "output_scale": output_scale,
        "output_zero_point": output_zp,
        "multiplier": multiplier,
        "shift": shift,
    }


def make_gelu_scalar_mul(model, op_index):
    op = model["ops"][op_index]
    assert op["op_name"] == "MUL"

    input_id, constant_id = [int(v) for v in op["inputs"]]
    output_id = int(op["outputs"][0])

    input_scale, input_zp = tensor_int8_quant(
        model["tensors"][input_id]
    )
    constant_scale, constant_zp = tensor_int8_quant(
        model["tensors"][constant_id]
    )

    constant = model["interpreter"].get_tensor(constant_id)
    assert constant.dtype == np.int8 and constant.size == 1
    centered_constant = int(constant.item()) - constant_zp

    input_bound = max(abs(-128 - input_zp), abs(127 - input_zp))
    source_bound = input_bound * abs(centered_constant)

    params = make_gelu_conversion(
        model, output_id,
        input_scale * constant_scale,
        source_bound,
        shift=38,
    )
    params.update({
        "input_tensor": input_id,
        "input_zero_point": input_zp,
        "centered_constant": centered_constant,
    })
    return params

In [43]:
gelu_params = {}

rows = [
    "| Block | Half M / shift | Cube M / shift | "
    "Weighted cube M / shift | Sum multipliers / shift | Argument M / shift |",
    "|---:|---|---|---|---|---|",
]

for block_index in range(8):
    model = block_models[block_index]
    ops = model["ops"]
    tensors = model["tensors"]

    half = make_gelu_scalar_mul(model, 60)
    weighted = make_gelu_scalar_mul(model, 63)
    argument = make_gelu_scalar_mul(model, 65)

    assert (
        half["input_tensor"]
        == mlp_expand_params[block_index]["output_tensor"]
    )

    input_scale, input_zp = tensor_int8_quant(
        tensors[half["input_tensor"]]
    )
    input_bound = max(abs(-128 - input_zp), abs(127 - input_zp))

    assert ops[61]["op_name"] == "POW"
    exponent_id = int(ops[61]["inputs"][1])
    assert float(model["interpreter"].get_tensor(exponent_id).item()) == 3.0

    cube_id = int(ops[62]["outputs"][0])
    cube = make_gelu_conversion(
        model, cube_id, input_scale ** 3, input_bound ** 3, shift=45
    )
    assert weighted["input_tensor"] == cube_id

    assert ops[64]["op_name"] == "ADD"
    sum_id = int(ops[64]["outputs"][0])
    sum_scale, sum_zp = tensor_int8_quant(tensors[sum_id])
    assert argument["input_tensor"] == sum_id

    linear_multiplier = int(np.rint(
        input_scale / sum_scale * (1 << 31)
    ))
    cubic_multiplier = int(np.rint(
        weighted["output_scale"] / sum_scale * (1 << 31)
    ))

    assert 0 < linear_multiplier <= np.iinfo(np.int64).max
    assert 0 < cubic_multiplier <= np.iinfo(np.int64).max

    cubic_zp = weighted["output_zero_point"]
    cubic_bound = max(abs(-128 - cubic_zp), abs(127 - cubic_zp))
    combined_bound = (
        input_bound * linear_multiplier
        + cubic_bound * cubic_multiplier
    )
    assert combined_bound + (1 << 30) <= np.iinfo(np.int64).max

    sum_params = {
        "linear_zero_point": input_zp,
        "cubic_zero_point": cubic_zp,
        "linear_multiplier": linear_multiplier,
        "cubic_multiplier": cubic_multiplier,
        "multiplier": 1,
        "shift": 31,
        "output_tensor": sum_id,
        "output_scale": sum_scale,
        "output_zero_point": sum_zp,
    }

    gelu_params[block_index] = {
        "input_scale": input_scale,
        "input_zero_point": input_zp,
        "half": half,
        "cube": cube,
        "weighted_cube": weighted,
        "sum": sum_params,
        "argument": argument,
    }

    rows.append(
        f"| {block_index} | {half['multiplier']} / 38 | "
        f"{cube['multiplier']} / 45 | "
        f"{weighted['multiplier']} / 38 | "
        f"{linear_multiplier}, {cubic_multiplier} / 31 | "
        f"{argument['multiplier']} / 38 |"
    )

display(Markdown("\n".join(rows)))
print("PASS: GELU front-stage parameter bounds checked.")

| Block | Half M / shift | Cube M / shift | Weighted cube M / shift | Sum multipliers / shift | Argument M / shift |
|---:|---|---|---|---|---|
| 0 | 1077952640 / 38 | 1670962322 / 45 | 1500813243 / 38 | 1916390445, 228467824 / 31 | 1077952560 / 38 |
| 1 | 1077952640 / 38 | 1757105062 / 45 | 1500813294 / 38 | 1972153116, 191591803 / 31 | 1077952465 / 38 |
| 2 | 1077952640 / 38 | 1489441103 / 45 | 1500813221 / 38 | 1934513336, 235936102 / 31 | 1077952628 / 38 |
| 3 | 1077952640 / 38 | 1548357869 / 45 | 1500813269 / 38 | 1877067126, 313257208 / 31 | 1077952562 / 38 |
| 4 | 1077952640 / 38 | 1684978506 / 45 | 1500813295 / 38 | 1894705625, 246267765 / 31 | 1077952621 / 38 |
| 5 | 1077952640 / 38 | 1494530941 / 45 | 1500813297 / 38 | 1753706041, 418537429 / 31 | 1077952492 / 38 |
| 6 | 1077952640 / 38 | 1571066477 / 45 | 1500813330 / 38 | 1865933094, 263899312 / 31 | 1077952596 / 38 |
| 7 | 1077952640 / 38 | 1389045012 / 45 | 1500813147 / 38 | 1531716450, 603171389 / 31 | 1077952592 / 38 |

PASS: GELU front-stage parameter bounds checked.


In [44]:
def requantize_to_i8(values, params, label):
    shifted = rq_checked(
        values, params["multiplier"], params["shift"], label
    )
    unbounded = checked_integer(
        shifted.astype(object) + params["output_zero_point"],
        np.int64, label + " zero point",
    )

    return {
        "output": np.clip(unbounded, -128, 127).astype(np.int8),
        "saturation_count": int(np.count_nonzero(
            (unbounded < -128) | (unbounded > 127)
        )),
    }


def integer_scalar_mul_i8(codes, params, label):
    centered = codes.astype(object) - params["input_zero_point"]
    product = checked_integer(
        centered * params["centered_constant"],
        np.int32, label + " product",
    )
    return requantize_to_i8(product, params, label)


def integer_gelu_front(input_codes, params):
    if input_codes.dtype != np.int8 or input_codes.shape != (1, 32, 512):
        raise ValueError("GELU expects INT8 [1,32,512]")

    half = integer_scalar_mul_i8(
        input_codes, params["half"], "GELU half"
    )

    centered = input_codes.astype(object) - params["input_zero_point"]
    cube_raw = checked_integer(
        centered ** 3, np.int32, "GELU cube"
    )
    cube = requantize_to_i8(
        cube_raw, params["cube"], "GELU cube conversion"
    )

    weighted = integer_scalar_mul_i8(
        cube["output"], params["weighted_cube"], "GELU weighted cube"
    )

    spec = params["sum"]
    linear_product = checked_integer(
        (input_codes.astype(object) - spec["linear_zero_point"])
        * spec["linear_multiplier"],
        np.int64, "GELU linear branch",
    )
    cubic_product = checked_integer(
        (weighted["output"].astype(object) - spec["cubic_zero_point"])
        * spec["cubic_multiplier"],
        np.int64, "GELU cubic branch",
    )
    combined = checked_integer(
        linear_product.astype(object) + cubic_product.astype(object),
        np.int64, "GELU combined branches",
    )

    # One rounded conversion after adding both scaled branches.
    added = requantize_to_i8(combined, spec, "GELU branch sum")

    argument = integer_scalar_mul_i8(
        added["output"], params["argument"], "GELU argument"
    )

    stages = {
        "half": half,
        "cube": cube,
        "weighted_cube": weighted,
        "sum": added,
        "argument": argument,
    }

    return {
        **{name: result["output"] for name, result in stages.items()},
        "cube_raw": cube_raw,
        "saturations": {
            name: result["saturation_count"]
            for name, result in stages.items()
        },
    }

In [45]:
gelu_front_trials = {i: [] for i in range(8)}

for block_index in range(8):
    params = gelu_params[block_index]

    for expansion_trial in mlp_expand_trials[block_index]:
        gelu_front_trials[block_index].append({
            "name": expansion_trial["name"],
            "source": expansion_trial,
            "local": integer_gelu_front(
                expansion_trial["reference_output"], params
            ),
            "ln1_chained": integer_gelu_front(
                expansion_trial["ln1_chained"]["output"], params
            ),
        })

    print(f"Completed GELU front stages: block {block_index}", flush=True)


gelu_front_metrics = {}
rows = [
    "| Block | Stage | Local agreement | Max code diff | "
    "Chained agreement | Max code diff |",
    "|---:|---|---:|---:|---:|---:|",
]

for block_index in range(8):
    trials = gelu_front_trials[block_index]

    for stage in ("half", "cube", "weighted_cube", "sum", "argument"):
        reference = np.concatenate([
            trial["source"]["gelu_references"][stage].reshape(-1)
            for trial in trials
        ]).astype(np.int32)

        metrics = {}

        for path in ("local", "ln1_chained"):
            actual = np.concatenate([
                trial[path][stage].reshape(-1)
                for trial in trials
            ]).astype(np.int32)

            difference = actual - reference
            metrics[path] = {
                "agreement": 100.0 * np.mean(difference == 0),
                "max_code_difference": int(np.max(np.abs(difference))),
            }

        gelu_front_metrics[(block_index, stage)] = metrics
        local = metrics["local"]
        chained = metrics["ln1_chained"]

        rows.append(
            f"| {block_index} | {stage} | "
            f"{local['agreement']:.3f}% | "
            f"{local['max_code_difference']} | "
            f"{chained['agreement']:.3f}% | "
            f"{chained['max_code_difference']} |"
        )

display(Markdown("\n".join(rows)))
print("PASS: GELU front stages completed without checked arithmetic overflow.")

Completed GELU front stages: block 0
Completed GELU front stages: block 1
Completed GELU front stages: block 2
Completed GELU front stages: block 3
Completed GELU front stages: block 4
Completed GELU front stages: block 5
Completed GELU front stages: block 6
Completed GELU front stages: block 7


| Block | Stage | Local agreement | Max code diff | Chained agreement | Max code diff |
|---:|---|---:|---:|---:|---:|
| 0 | half | 100.000% | 0 | 92.550% | 1 |
| 0 | cube | 100.000% | 0 | 99.463% | 4 |
| 0 | weighted_cube | 100.000% | 0 | 99.463% | 4 |
| 0 | sum | 100.000% | 0 | 93.272% | 1 |
| 0 | argument | 100.000% | 0 | 93.272% | 1 |
| 1 | half | 100.000% | 0 | 73.009% | 9 |
| 1 | cube | 100.000% | 0 | 95.329% | 8 |
| 1 | weighted_cube | 100.000% | 0 | 95.329% | 8 |
| 1 | sum | 100.000% | 0 | 74.712% | 8 |
| 1 | argument | 100.000% | 0 | 74.712% | 8 |
| 2 | half | 100.000% | 0 | 79.810% | 3 |
| 2 | cube | 100.000% | 0 | 96.899% | 4 |
| 2 | weighted_cube | 100.000% | 0 | 96.899% | 4 |
| 2 | sum | 100.000% | 0 | 81.459% | 3 |
| 2 | argument | 100.000% | 0 | 81.459% | 3 |
| 3 | half | 100.000% | 0 | 82.900% | 3 |
| 3 | cube | 100.000% | 0 | 97.964% | 4 |
| 3 | weighted_cube | 100.000% | 0 | 97.964% | 4 |
| 3 | sum | 100.000% | 0 | 84.776% | 3 |
| 3 | argument | 100.000% | 0 | 84.776% | 3 |
| 4 | half | 100.000% | 0 | 79.625% | 2 |
| 4 | cube | 100.000% | 0 | 96.497% | 4 |
| 4 | weighted_cube | 100.000% | 0 | 96.497% | 4 |
| 4 | sum | 100.000% | 0 | 81.585% | 2 |
| 4 | argument | 100.000% | 0 | 81.585% | 2 |
| 5 | half | 100.000% | 0 | 84.605% | 2 |
| 5 | cube | 100.000% | 0 | 98.299% | 5 |
| 5 | weighted_cube | 100.000% | 0 | 98.299% | 5 |
| 5 | sum | 100.000% | 0 | 87.096% | 3 |
| 5 | argument | 100.000% | 0 | 87.096% | 3 |
| 6 | half | 100.000% | 0 | 82.106% | 2 |
| 6 | cube | 100.000% | 0 | 96.848% | 4 |
| 6 | weighted_cube | 100.000% | 0 | 96.848% | 4 |
| 6 | sum | 100.000% | 0 | 84.074% | 2 |
| 6 | argument | 100.000% | 0 | 84.074% | 2 |
| 7 | half | 100.000% | 0 | 86.819% | 2 |
| 7 | cube | 100.000% | 0 | 98.833% | 3 |
| 7 | weighted_cube | 100.000% | 0 | 98.833% | 3 |
| 7 | sum | 100.000% | 0 | 90.306% | 2 |
| 7 | argument | 100.000% | 0 | 90.306% | 2 |

PASS: GELU front stages completed without checked arithmetic overflow.


In [46]:
rows = [
    "| Block | Tanh argument scale / ZP | Factor M / shift | "
    "Final Q31 multiplier | Second shift | Final output ZP |",
    "|---:|---|---|---:|---:|---:|",
]

for block_index in range(8):
    model = block_models[block_index]
    params = gelu_params[block_index]
    ops = model["ops"]

    assert ops[67]["op_name"] == "TANH"
    assert int(ops[66]["inputs"][0]) == params["argument"]["output_tensor"]

    one_id = int(ops[68]["inputs"][1])
    assert float(model["interpreter"].get_tensor(one_id).item()) == 1.0

    argument_scale = params["argument"]["output_scale"]
    argument_zp = params["argument"]["output_zero_point"]

    argument_codes = np.arange(-128, 128, dtype=np.int32)
    argument_values = (
        argument_codes.astype(np.float64) - argument_zp
    ) * argument_scale

    tanh_lut = checked_integer(
        np.rint(np.tanh(argument_values) * (1 << 24)),
        np.int32,
        "tanh table",
    )
    assert np.all(
        (tanh_lut >= -(1 << 24)) & (tanh_lut <= (1 << 24))
    )

    factor_id = int(ops[69]["outputs"][0])
    factor = make_gelu_conversion(
        model,
        factor_id,
        source_scale=2.0 ** -24,
        source_bound=1 << 25,
        shift=47,
    )

    assert [int(v) for v in ops[70]["inputs"]] == [
        params["half"]["output_tensor"],
        factor_id,
    ]

    output_id = int(ops[70]["outputs"][0])
    output_scale, output_zp = tensor_int8_quant(
        model["tensors"][output_id]
    )

    ratio = (
        params["half"]["output_scale"]
        * factor["output_scale"]
        / output_scale
    )
    assert math.isfinite(ratio) and ratio > 0

    fraction, exponent = math.frexp(ratio)
    q31_multiplier = int(math.floor(fraction * (1 << 31) + 0.5))

    if q31_multiplier == (1 << 31):
        q31_multiplier //= 2
        exponent += 1

    second_shift = -exponent
    assert 0 < q31_multiplier <= np.iinfo(np.int32).max
    assert 0 <= second_shift <= 31

    half_zp = params["half"]["output_zero_point"]
    factor_zp = factor["output_zero_point"]
    half_bound = max(abs(-128 - half_zp), abs(127 - half_zp))
    factor_bound = max(abs(-128 - factor_zp), abs(127 - factor_zp))
    product_bound = half_bound * factor_bound

    assert product_bound <= np.iinfo(np.int32).max
    assert (
        product_bound * q31_multiplier + (1 << 30)
        <= np.iinfo(np.int64).max
    )

    params.update({
        "tanh_lut": tanh_lut,
        "factor": factor,
        "final": {
            "q31_multiplier": q31_multiplier,
            "second_shift": second_shift,
            "half_zero_point": half_zp,
            "factor_zero_point": factor_zp,
            "output_tensor": output_id,
            "output_scale": output_scale,
            "output_zero_point": output_zp,
        },
    })

    rows.append(
        f"| {block_index} | {argument_scale:.12g} / {argument_zp} | "
        f"{factor['multiplier']} / 47 | "
        f"{q31_multiplier} | {second_shift} | {output_zp} |"
    )

display(Markdown("\n".join(rows)))
print("PASS: remaining GELU parameter bounds checked.")

| Block | Tanh argument scale / ZP | Factor M / shift | Final Q31 multiplier | Second shift | Final output ZP |
|---:|---|---|---:|---:|---:|
| 0 | 0.0100609334186 / 37 | 1269488229 / 47 | 1195729761 | 6 | -88 |
| 1 | 0.00904992222786 / 33 | 1306554124 / 47 | 1226892139 | 6 | -83 |
| 2 | 0.00951742939651 / 42 | 1297158563 / 47 | 1256871958 | 6 | -83 |
| 3 | 0.0116986278445 / 40 | 1243575406 / 47 | 1260543699 | 6 | -92 |
| 4 | 0.0106698116288 / 33 | 1214011577 / 47 | 1197044174 | 6 | -92 |
| 5 | 0.0147113613784 / 61 | 1212907607 / 47 | 1427761311 | 6 | -94 |
| 6 | 0.0109129138291 / 36 | 1231216555 / 47 | 1184683637 | 6 | -92 |
| 7 | 0.0208583213389 / 68 | 1181384362 / 47 | 1638691267 | 6 | -97 |

PASS: remaining GELU parameter bounds checked.


In [47]:
def requantize_two_round_i8(values, params):
    if values.dtype != np.int32:
        raise ValueError("Two-round conversion expects INT32 input")

    multiplier = params["q31_multiplier"]
    shift = params["second_shift"]

    if not (
        0 < multiplier <= np.iinfo(np.int32).max
        and 0 <= shift <= 31
    ):
        raise ValueError("Invalid two-round parameters")

    high = rq_checked(
        values, multiplier, 31, "GELU Q31 output"
    )

    mask = (1 << shift) - 1
    remainder = high & mask
    threshold = (mask >> 1) + (high < 0).astype(np.int64)

    rounded = (
        (high >> shift)
        + (remainder > threshold).astype(np.int64)
    )

    unbounded = checked_integer(
        rounded.astype(object) + params["output_zero_point"],
        np.int64,
        "GELU final zero point",
    )

    return {
        "output": np.clip(unbounded, -128, 127).astype(np.int8),
        "saturation_count": int(np.count_nonzero(
            (unbounded < -128) | (unbounded > 127)
        )),
    }


def integer_gelu_back(front, params):
    indices = front["argument"].astype(np.int16) + 128
    tanh_codes = params["tanh_lut"][indices]

    # Tanh and one-plus-tanh both use scale 2^-24.
    factor_fine = checked_integer(
        tanh_codes.astype(object) + (1 << 24),
        np.int32,
        "one plus tanh",
    )

    factor = requantize_to_i8(
        factor_fine, params["factor"], "GELU factor"
    )

    final_params = params["final"]
    product = checked_integer(
        (front["half"].astype(object) - final_params["half_zero_point"])
        * (
            factor["output"].astype(object)
            - final_params["factor_zero_point"]
        ),
        np.int32,
        "GELU final product",
    )

    output = requantize_two_round_i8(product, final_params)

    return {
        "tanh": tanh_codes,
        "factor_fine": factor_fine,
        "factor": factor["output"],
        "product": product,
        "output": output["output"],
        "saturations": {
            "factor": factor["saturation_count"],
            "output": output["saturation_count"],
        },
    }


def integer_gelu(input_codes, params):
    front = integer_gelu_front(input_codes, params)
    back = integer_gelu_back(front, params)

    return {
        **front,
        **back,
        "saturations": {
            **front["saturations"],
            **back["saturations"],
        },
    }

In [48]:
gelu_trials = {i: [] for i in range(8)}

for block_index in range(8):
    params = gelu_params[block_index]

    for front_trial in gelu_front_trials[block_index]:
        gelu_trials[block_index].append({
            "name": front_trial["name"],
            "source": front_trial,
            "reference": front_trial["source"]["gelu_references"],
            "local": integer_gelu_back(front_trial["local"], params),
            "ln1_chained": integer_gelu_back(
                front_trial["ln1_chained"], params
            ),
        })

    print(f"Completed GELU: block {block_index}", flush=True)

Completed GELU: block 0
Completed GELU: block 1
Completed GELU: block 2
Completed GELU: block 3
Completed GELU: block 4
Completed GELU: block 5
Completed GELU: block 6
Completed GELU: block 7


In [49]:
gelu_metrics = {}
maximum_local_tanh_error = 0.0

rows = [
    "| Block | Cases | Local factor agreement | Local GELU agreement | "
    "Max code diff | Chained GELU agreement | Max code diff | "
    "Final saturations local / chained |",
    "|---:|---|---:|---:|---:|---:|---:|---|",
]

for block_index in range(8):
    for group_name, names in (
        ("Benchmark", {"benchmark"}),
        ("Calibration", {case["name"] for case in calibration_cases}),
    ):
        selected = [
            trial for trial in gelu_trials[block_index]
            if trial["name"] in names
        ]

        reference_output = np.concatenate([
            trial["reference"]["output"].reshape(-1)
            for trial in selected
        ]).astype(np.int32)

        reference_factor = np.concatenate([
            trial["reference"]["factor"].reshape(-1)
            for trial in selected
        ]).astype(np.int32)

        reference_tanh = np.concatenate([
            trial["reference"]["tanh"].reshape(-1)
            for trial in selected
        ]).astype(np.float64)

        metrics = {}

        for path in ("local", "ln1_chained"):
            actual_output = np.concatenate([
                trial[path]["output"].reshape(-1)
                for trial in selected
            ]).astype(np.int32)

            actual_factor = np.concatenate([
                trial[path]["factor"].reshape(-1)
                for trial in selected
            ]).astype(np.int32)

            represented_tanh = np.concatenate([
                trial[path]["tanh"].reshape(-1)
                for trial in selected
            ]).astype(np.float64) * (2.0 ** -24)

            difference = actual_output - reference_output

            metrics[path] = {
                "agreement": 100.0 * np.mean(difference == 0),
                "max_code_difference": int(np.max(np.abs(difference))),
                "factor_agreement": 100.0 * np.mean(
                    actual_factor == reference_factor
                ),
                "tanh_max_abs_error": float(np.max(np.abs(
                    represented_tanh - reference_tanh
                ))),
                "saturation_count": sum(
                    trial[path]["saturations"]["output"]
                    for trial in selected
                ),
            }

        gelu_metrics[(block_index, group_name)] = metrics
        local = metrics["local"]
        chained = metrics["ln1_chained"]

        maximum_local_tanh_error = max(
            maximum_local_tanh_error,
            local["tanh_max_abs_error"],
        )

        rows.append(
            f"| {block_index} | {group_name} | "
            f"{local['factor_agreement']:.3f}% | "
            f"{local['agreement']:.3f}% | "
            f"{local['max_code_difference']} | "
            f"{chained['agreement']:.3f}% | "
            f"{chained['max_code_difference']} | "
            f"{local['saturation_count']} / "
            f"{chained['saturation_count']} |"
        )

display(Markdown("\n".join(rows)))
print("Maximum local tanh absolute error:", maximum_local_tanh_error)
print("PASS: GELU trials completed without checked arithmetic overflow.")

| Block | Cases | Local factor agreement | Local GELU agreement | Max code diff | Chained GELU agreement | Max code diff | Final saturations local / chained |
|---:|---|---:|---:|---:|---:|---:|---|
| 0 | Benchmark | 100.000% | 100.000% | 0 | 95.123% | 3 | 0 / 0 |
| 0 | Calibration | 100.000% | 100.000% | 0 | 92.983% | 3 | 0 / 0 |
| 1 | Benchmark | 100.000% | 100.000% | 0 | 76.581% | 6 | 0 / 0 |
| 1 | Calibration | 100.000% | 100.000% | 0 | 77.798% | 18 | 4 / 4 |
| 2 | Benchmark | 100.000% | 100.000% | 0 | 89.734% | 4 | 0 / 0 |
| 2 | Calibration | 100.000% | 100.000% | 0 | 83.441% | 6 | 0 / 0 |
| 3 | Benchmark | 100.000% | 100.000% | 0 | 86.456% | 3 | 0 / 0 |
| 3 | Calibration | 100.000% | 100.000% | 0 | 86.719% | 6 | 2 / 2 |
| 4 | Benchmark | 100.000% | 100.000% | 0 | 82.397% | 4 | 0 / 0 |
| 4 | Calibration | 100.000% | 100.000% | 0 | 86.225% | 5 | 0 / 0 |
| 5 | Benchmark | 100.000% | 100.000% | 0 | 89.294% | 4 | 0 / 0 |
| 5 | Calibration | 100.000% | 100.000% | 0 | 88.411% | 4 | 0 / 0 |
| 6 | Benchmark | 100.000% | 100.000% | 0 | 86.487% | 3 | 0 / 0 |
| 6 | Calibration | 100.000% | 100.000% | 0 | 87.777% | 3 | 0 / 0 |
| 7 | Benchmark | 100.000% | 100.000% | 0 | 90.869% | 5 | 0 / 0 |
| 7 | Calibration | 100.000% | 100.000% | 0 | 89.767% | 4 | 0 / 0 |

Maximum local tanh absolute error: 5.960464477539063e-08
PASS: GELU trials completed without checked arithmetic overflow.


## Part 7 — MLP output, final residual, and INT8 block output

### Inputs and output

- GELU input to this part: INT8 `[1,32,512]`.
- Retained first residual: INT32 `[1,32,128]`, scale `2^-16`, zero point `0`.
- Projection weights: INT8 `[128,512]`, zero point `0`.
- Projection bias: saved INT32 `[128]`, at accumulator scale.
- Internal final residual: INT32 `[1,32,128]`, scale `2^-16`.
- Block output: INT8 `[1,32,128]`, with the selected boundary scale and zero point `0`.

### 1. Project 512 → 128

$$
A[t,o]
=
\sum_{k=0}^{511}(q_G[t,k]-z_G)q_W[o,k]+b_q[o]
$$

The accumulator scale is:

$$
s_A=s_Gs_W
$$

XTILE computes the dot products; software adds bias.
Products, partial sums, and bias addition must fit INT32.

### 2. Preserve the saved INT8 projection output

Prepare offline:

$$
M_P=\operatorname{round}_{even}
\left(\frac{s_A}{s_P}2^{39}\right)
$$

**Requantization 1 — after accumulation and bias:**

$$
q_P=\operatorname{clip}_{[-128,127]}
\left(RQ(A,M_P,39)+z_P\right)
$$

Use the block’s saved projection-output scale and zero point.
The C implementation uses XREQUANT mode 2.

### 3. Convert the projection branch to scale 2^-16

Prepare offline:

$$
M_R=\operatorname{round}_{even}(s_P2^{16}2^{22})
$$

**Requantization 2 — before residual addition:**

$$
q_B=RQ(q_P-z_P,M_R,22)
$$

Store `q_B` as INT32 at scale `2^-16`, zero point `0`.
No INT8 saturation is applied here.
The C implementation uses XREQUANT mode 3.

### 4. Add the retained first residual

$$
q_Y=q_R+q_B
$$

Both branches use scale `2^-16`.
Check the sum before INT32 storage.

No rounding is required for this addition.

### 5. Produce the INT8 block output

For selected boundary scale `2^-F_out`:

**Requantization 3 — after the final residual addition:**

$$
q_{\mathrm{block,out}}
=
\operatorname{clip}_{[-128,127]}
\left(RQ(q_Y,1,16-F_{\mathrm{out}})\right)
$$

| Blocks | Output scale | Rounded right shift |
|---|---|---:|
| 0, 7 | `2^-7` | 9 |
| 1–6 | `2^-8` | 8 |

The C implementation can use XREQUANT mode 2 with multiplier `1`.

The next block consumes these same INT8 codes using this output scale.
No second conversion is required at the connection.

### Arithmetic rules

$$
RQ(a,M,r)=(aM+2^{r-1})\gg r
$$

- Runtime rounding: nearest, ties toward positive infinity.
- Offline multiplier preparation: nearest, ties to even.
- Multiplier products and rounding additions use checked INT64.
- Stored accumulators, converted branches, and residual sums must fit INT32.

### Validation

Local:
reference GELU output and independently encoded reference first residual → Part 7.

Chained:
integer Parts 1–6 and the retained integer Part 4 residual → Part 7.

Current eight-case results across all blocks:

- Local projection: 262,144 outputs, zero mismatches.
- Local pre-boundary RMSE: approximately 6.2e-6.
- Local boundary agreement: 99.683%–99.937%.
- Within-block chained pre-boundary relative L2: 0.765%–2.381%.
- Chained boundary agreement: 77.808%–95.776%.
- Chained maximum boundary difference: five INT8 codes.
- No final-boundary clipping occurred.
- Checked arithmetic completed without overflow.

Complete multi-block integer inference validation remains pending.

In [50]:
mlp_out_params = {}

rows = [
    "| Block | Projection M / shift | Projection ZP | "
    "Residual-branch M / shift | Final INT8 boundary shift |",
    "|---:|---|---:|---|---:|",
]

for block_index in range(8):
    model = block_models[block_index]
    params = read_projection_params(
        block_index, 71, expected_weight_shape=(128, 512)
    )
    assert params["bias"] is not None
    assert (
        params["input_tensor"]
        == gelu_params[block_index]["final"]["output_tensor"]
    )

    branch_multiplier = int(np.rint(
        params["output_scale"] * (1 << 16) * (1 << 22)
    ))
    centered_bound = max(
        abs(-128 - params["output_zero_point"]),
        abs(127 - params["output_zero_point"]),
    )

    assert 0 < branch_multiplier <= np.iinfo(np.int64).max
    assert (
        centered_bound * branch_multiplier + (1 << 21)
        <= np.iinfo(np.int64).max
    )

    boundary = block_boundary_specs[block_index]["output"]
    assert boundary["zero_point"] == 0
    output_frac_bits = -int(boundary["scale_exponent"])
    assert output_frac_bits in (7, 8)

    params.update({
        "branch_multiplier": branch_multiplier,
        "branch_shift": 22,
        "reference_residual_tensor": int(model["ops"][42]["outputs"][0]),
        "reference_output_tensor": int(model["ops"][73]["outputs"][0]),
        "boundary": {
            "multiplier": 1,
            "shift": 16 - output_frac_bits,
            "output_zero_point": 0,
            "scale": boundary["scale"],
        },
    })
    mlp_out_params[block_index] = params

    rows.append(
        f"| {block_index} | {params['multiplier']} / 39 | "
        f"{params['output_zero_point']} | "
        f"{branch_multiplier} / 22 | "
        f"{params['boundary']['shift']} |"
    )

display(Markdown("\n".join(rows)))

| Block | Projection M / shift | Projection ZP | Residual-branch M / shift | Final INT8 boundary shift |
|---:|---|---:|---|---:|
| 0 | 641059315 / 39 | 10 | 720118272 / 22 | 9 |
| 1 | 591122097 / 39 | -11 | 581086080 / 22 | 8 |
| 2 | 694140757 / 39 | 6 | 440037408 / 22 | 8 |
| 3 | 983191189 / 39 | -10 | 387050784 / 22 | 8 |
| 4 | 1132406433 / 39 | -3 | 403415264 / 22 | 8 |
| 5 | 1092815675 / 39 | 5 | 418461824 / 22 | 8 |
| 6 | 895902594 / 39 | 0 | 464180864 / 22 | 8 |
| 7 | 1069260537 / 39 | 41 | 740551232 / 22 | 9 |

In [51]:
def integer_mlp_out_residual(gelu_output, residual_input, params):
    if (
        residual_input.dtype != np.int32
        or residual_input.shape != (1, 32, 128)
    ):
        raise ValueError("Retained residual must be INT32 [1,32,128]")

    projection = integer_projection(gelu_output, params)

    centered = (
        projection["output"].astype(object)
        - params["output_zero_point"]
    )
    branch = checked_integer(
        rq_checked(
            centered,
            params["branch_multiplier"],
            params["branch_shift"],
            "final residual branch",
        ),
        np.int32, "stored final branch",
    )

    output_i32 = checked_integer(
        residual_input.astype(object) + branch.astype(object),
        np.int32, "final residual sum",
    )

    boundary = requantize_to_i8(
        output_i32, params["boundary"], "block output boundary"
    )

    return {
        "projection": projection,
        "branch": branch,
        "output_i32": output_i32,
        "output": boundary["output"],
        "boundary_saturations": boundary["saturation_count"],
    }


def encode_reference_block_output(values, boundary):
    # Diagnostic reference encoding using the runtime tie rule.
    rounded = np.floor(
        values.astype(np.float64) / boundary["scale"] + 0.5
    )
    return {
        "output": np.clip(rounded, -128, 127).astype(np.int8),
        "saturation_count": int(np.count_nonzero(
            (rounded < -128) | (rounded > 127)
        )),
    }

In [52]:
mlp_out_trials = {i: [] for i in range(8)}

for block_index in range(8):
    model = block_models[block_index]
    interpreter = model["interpreter"]
    params = mlp_out_params[block_index]
    input_scale = block_boundary_specs[block_index]["input"]["scale"]

    assert (
        len(gelu_trials[block_index])
        == len(out_residual_trials[block_index])
        == len(ln1_trials[block_index])
    )

    for gelu_trial, residual_trial, ln1_trial in zip(
        gelu_trials[block_index],
        out_residual_trials[block_index],
        ln1_trials[block_index],
    ):
        assert (
            gelu_trial["name"]
            == residual_trial["name"]
            == ln1_trial["name"]
        )

        represented_input = (
            ln1_trial["input_codes"].astype(np.float64) * input_scale
        ).astype(np.float32)
        run_tflite_block(block_index, represented_input)

        reference_gelu = interpreter.get_tensor(
            params["input_tensor"]
        ).copy()
        np.testing.assert_array_equal(
            reference_gelu, gelu_trial["reference"]["output"]
        )

        reference_residual = interpreter.get_tensor(
            params["reference_residual_tensor"]
        ).copy()
        np.testing.assert_array_equal(
            reference_residual, residual_trial["reference_residual"]
        )

        reference_projection = interpreter.get_tensor(
            params["output_tensor"]
        ).copy()
        reference_output = interpreter.get_tensor(
            params["reference_output_tensor"]
        ).copy()

        local_residual = checked_integer(
            np.rint(reference_residual.astype(np.float64) * (1 << 16)),
            np.int32, "encoded reference first residual",
        )

        mlp_out_trials[block_index].append({
            "name": gelu_trial["name"],
            "reference_projection": reference_projection,
            "reference_output": reference_output,
            "reference_boundary": encode_reference_block_output(
                reference_output, params["boundary"]
            ),
            "local": integer_mlp_out_residual(
                reference_gelu, local_residual, params
            ),
            "ln1_chained": integer_mlp_out_residual(
                gelu_trial["ln1_chained"]["output"],
                residual_trial["ln1_chained"]["output"],
                params,
            ),
        })

    print(f"Completed Part 7: block {block_index}", flush=True)

Completed Part 7: block 0
Completed Part 7: block 1
Completed Part 7: block 2
Completed Part 7: block 3
Completed Part 7: block 4
Completed Part 7: block 5
Completed Part 7: block 6
Completed Part 7: block 7


In [53]:
mlp_out_metrics = {}

rows = [
    "| Block | Cases | Local projection agreement | Local pre-boundary RMSE | "
    "Chained pre-boundary relative L2 | Local boundary agreement | "
    "Chained boundary agreement | Chained max code diff | "
    "Boundary clips: reference / local / chained |",
    "|---:|---|---:|---:|---:|---:|---:|---:|---|",
]

for block_index in range(8):
    for group_name, names in (
        ("Benchmark", {"benchmark"}),
        ("Calibration", {case["name"] for case in calibration_cases}),
    ):
        selected = [
            trial for trial in mlp_out_trials[block_index]
            if trial["name"] in names
        ]

        reference_projection = np.concatenate([
            trial["reference_projection"].reshape(-1)
            for trial in selected
        ]).astype(np.int32)

        reference_output = np.concatenate([
            trial["reference_output"].reshape(-1)
            for trial in selected
        ]).astype(np.float64)

        reference_boundary = np.concatenate([
            trial["reference_boundary"]["output"].reshape(-1)
            for trial in selected
        ]).astype(np.int32)

        metrics = {}

        for path in ("local", "ln1_chained"):
            projection = np.concatenate([
                trial[path]["projection"]["output"].reshape(-1)
                for trial in selected
            ]).astype(np.int32)

            represented_output = np.concatenate([
                trial[path]["output_i32"].reshape(-1)
                for trial in selected
            ]).astype(np.float64) * (2.0 ** -16)

            output_codes = np.concatenate([
                trial[path]["output"].reshape(-1)
                for trial in selected
            ]).astype(np.int32)

            difference = output_codes - reference_boundary

            metrics[path] = {
                "projection_agreement": 100.0 * np.mean(
                    projection == reference_projection
                ),
                "pre_boundary": error_metrics(
                    represented_output, reference_output
                ),
                "boundary_agreement": 100.0 * np.mean(difference == 0),
                "max_boundary_difference": int(np.max(np.abs(difference))),
                "boundary_clips": sum(
                    trial[path]["boundary_saturations"]
                    for trial in selected
                ),
            }

        reference_clips = sum(
            trial["reference_boundary"]["saturation_count"]
            for trial in selected
        )
        mlp_out_metrics[(block_index, group_name)] = metrics
        local = metrics["local"]
        chained = metrics["ln1_chained"]

        rows.append(
            f"| {block_index} | {group_name} | "
            f"{local['projection_agreement']:.3f}% | "
            f"{local['pre_boundary']['rmse']:.6g} | "
            f"{chained['pre_boundary']['relative_l2_percent']:.6g}% | "
            f"{local['boundary_agreement']:.3f}% | "
            f"{chained['boundary_agreement']:.3f}% | "
            f"{chained['max_boundary_difference']} | "
            f"{reference_clips} / {local['boundary_clips']} / "
            f"{chained['boundary_clips']} |"
        )

display(Markdown("\n".join(rows)))
print("PASS: Part 7 trials completed without checked arithmetic overflow.")

| Block | Cases | Local projection agreement | Local pre-boundary RMSE | Chained pre-boundary relative L2 | Local boundary agreement | Chained boundary agreement | Chained max code diff | Boundary clips: reference / local / chained |
|---:|---|---:|---:|---:|---:|---:|---:|---|
| 0 | Benchmark | 100.000% | 6.23468e-06 | 0.765101% | 99.902% | 95.776% | 1 | 0 / 0 / 0 |
| 0 | Calibration | 100.000% | 6.25692e-06 | 0.939292% | 99.913% | 94.078% | 1 | 0 / 0 / 0 |
| 1 | Benchmark | 100.000% | 6.34191e-06 | 1.49409% | 99.683% | 77.808% | 2 | 0 / 0 / 0 |
| 1 | Calibration | 100.000% | 6.24436e-06 | 1.58931% | 99.787% | 78.805% | 5 | 0 / 0 / 0 |
| 2 | Benchmark | 100.000% | 6.25707e-06 | 1.03624% | 99.829% | 88.794% | 2 | 0 / 0 / 0 |
| 2 | Calibration | 100.000% | 6.21433e-06 | 1.27866% | 99.843% | 83.625% | 2 | 0 / 0 / 0 |
| 3 | Benchmark | 100.000% | 6.30084e-06 | 1.46164% | 99.805% | 82.642% | 2 | 0 / 0 / 0 |
| 3 | Calibration | 100.000% | 6.2563e-06 | 1.43118% | 99.885% | 83.475% | 2 | 0 / 0 / 0 |
| 4 | Benchmark | 100.000% | 6.25597e-06 | 1.62087% | 99.902% | 79.517% | 1 | 0 / 0 / 0 |
| 4 | Calibration | 100.000% | 6.21779e-06 | 1.4377% | 99.874% | 82.649% | 2 | 0 / 0 / 0 |
| 5 | Benchmark | 100.000% | 6.24925e-06 | 1.70016% | 99.756% | 79.736% | 1 | 0 / 0 / 0 |
| 5 | Calibration | 100.000% | 6.19753e-06 | 1.72042% | 99.791% | 79.356% | 2 | 0 / 0 / 0 |
| 6 | Benchmark | 100.000% | 6.13165e-06 | 1.64886% | 99.902% | 79.492% | 1 | 0 / 0 / 0 |
| 6 | Calibration | 100.000% | 6.17176e-06 | 1.524% | 99.826% | 81.717% | 2 | 0 / 0 / 0 |
| 7 | Benchmark | 100.000% | 6.18457e-06 | 2.37248% | 99.927% | 87.964% | 1 | 0 / 0 / 0 |
| 7 | Calibration | 100.000% | 6.25547e-06 | 2.38087% | 99.937% | 85.986% | 1 | 0 / 0 / 0 |

PASS: Part 7 trials completed without checked arithmetic overflow.


# FULL TESTING

In [54]:
def integer_block_forward(input_codes, block_index):
    if block_index not in range(8):
        raise ValueError("Block index must be 0..7")

    ln1 = integer_ln1(input_codes, ln1_params[block_index])

    projections = {
        name: integer_projection(
            ln1["output"], qkv_params[block_index][name]
        )
        for name in ("Q", "K", "V")
    }

    params = attention_params[block_index]
    scores = integer_attention_scores(
        projections["Q"]["output"],
        projections["K"]["output"],
        params,
    )
    softmax = integer_causal_softmax(scores, params)
    attention = integer_attention_context(
        softmax["probabilities"],
        projections["V"]["output"],
        params,
    )

    residual = integer_out_residual(
        attention["output"],
        input_codes,
        out_residual_params[block_index],
    )
    ln2 = integer_ln2(residual["output"], ln2_params[block_index])

    expansion = integer_projection(
        ln2["output"], mlp_expand_params[block_index]
    )
    gelu = integer_gelu(expansion["output"], gelu_params[block_index])

    final = integer_mlp_out_residual(
        gelu["output"],
        residual["output"],
        mlp_out_params[block_index],
    )

    return {
        "output": final["output"],
        "output_i32": final["output_i32"],
        "saturations": {
            "ln1": ln1["saturation_count"],
            **{
                name: projections[name]["saturation_count"]
                for name in ("Q", "K", "V")
            },
            "attention": attention["saturation_count"],
            "out_projection": residual["projection"]["saturation_count"],
            "ln2": ln2["saturation_count"],
            "mlp_expand": expansion["saturation_count"],
            **{
                "gelu_" + name: count
                for name, count in gelu["saturations"].items()
            },
            "mlp_out_projection": final["projection"]["saturation_count"],
            "block_boundary": final["boundary_saturations"],
        },
        "max_softmax_row_sum_error": softmax["max_row_sum_error"],
    }

In [55]:
for block_index in range(7):
    output_spec = block_boundary_specs[block_index]["output"]
    next_input_spec = block_boundary_specs[block_index + 1]["input"]

    assert output_spec["scale"] == next_input_spec["scale"]
    assert output_spec["zero_point"] == next_input_spec["zero_point"]

for block_index in range(8):
    assert len(ln1_trials[block_index]) == len(
        mlp_out_trials[block_index]
    )

    for input_trial, final_trial in zip(
        ln1_trials[block_index], mlp_out_trials[block_index]
    ):
        assert input_trial["name"] == final_trial["name"]

        actual = integer_block_forward(
            input_trial["input_codes"], block_index
        )
        expected = final_trial["ln1_chained"]

        np.testing.assert_array_equal(
            actual["output_i32"],
            expected["output_i32"],
            err_msg=f"Block {block_index}: internal final output mismatch",
        )
        np.testing.assert_array_equal(
            actual["output"],
            expected["output"],
            err_msg=f"Block {block_index}: INT8 boundary mismatch",
        )

    print(
        f"PASS: block {block_index} reproduces all saved stage results.",
        flush=True,
    )

print("PASS: block-forward wiring verified for all 64 block/case combinations.")

PASS: block 0 reproduces all saved stage results.
PASS: block 1 reproduces all saved stage results.
PASS: block 2 reproduces all saved stage results.
PASS: block 3 reproduces all saved stage results.
PASS: block 4 reproduces all saved stage results.
PASS: block 5 reproduces all saved stage results.
PASS: block 6 reproduces all saved stage results.
PASS: block 7 reproduces all saved stage results.
PASS: block-forward wiring verified for all 64 block/case combinations.


In [56]:
integer_chain_results = {}
mixed_boundary_results = {}

for case in reference_cases:
    name = case["name"]

    initial_codes, initial_represented, initial_clips = encode_boundary(
        case["blocks"][0]["input_fp32"], 0
    )

    integer_input = initial_codes.copy()
    mixed_input = initial_represented.copy()

    integer_blocks = []
    mixed_outputs = []
    mixed_clips = []

    for block_index in range(8):
        try:
            integer_result = integer_block_forward(
                integer_input, block_index
            )
        except (OverflowError, ValueError, AssertionError) as error:
            raise RuntimeError(
                f"{name}, block {block_index}: integer chain failed"
            ) from error

        mixed_raw = run_tflite_block(block_index, mixed_input)
        boundary = mlp_out_params[block_index]["boundary"]
        mixed_encoded = encode_reference_block_output(
            mixed_raw, boundary
        )

        integer_blocks.append(integer_result)
        mixed_outputs.append(mixed_encoded["output"].copy())
        mixed_clips.append(mixed_encoded["saturation_count"])

        # Actual INT8 codes pass directly to the next integer block.
        integer_input = integer_result["output"]

        # FP32 bridge for the mixed TFLite diagnostic path only.
        mixed_input = (
            mixed_encoded["output"].astype(np.float64)
            * boundary["scale"]
        ).astype(np.float32)

    integer_chain_results[name] = {
        "blocks": integer_blocks,
        "initial_input_clips": initial_clips,
    }
    mixed_boundary_results[name] = {
        "outputs": mixed_outputs,
        "boundary_clips": mixed_clips,
        "initial_input_clips": initial_clips,
    }

    print(f"Completed both eight-block chains: {name}", flush=True)

Completed both eight-block chains: benchmark
Completed both eight-block chains: calibration_0
Completed both eight-block chains: calibration_1
Completed both eight-block chains: calibration_2
Completed both eight-block chains: calibration_3
Completed both eight-block chains: calibration_4
Completed both eight-block chains: calibration_5
Completed both eight-block chains: calibration_6


In [57]:
integer_chain_metrics = {}

rows = [
    "| Cases | After block | Direct TFLite relative L2 | "
    "TFLite + INT8 boundaries relative L2 | "
    "Integer relative L2 | Integer RMSE | "
    "Boundary clips: TFLite / integer |",
    "|---|---:|---:|---:|---:|---:|---|",
]

for group_name, cases in (
    ("Benchmark", benchmark_cases),
    ("Calibration", calibration_cases),
):
    for block_index in range(8):
        scale = block_boundary_specs[block_index]["output"]["scale"]

        reference = np.concatenate([
            case["blocks"][block_index]["output_fp32"].reshape(-1)
            for case in cases
        ]).astype(np.float64)

        direct = np.concatenate([
            boundary_chain_results[case["name"]]
            ["direct_outputs"][block_index].reshape(-1)
            for case in cases
        ]).astype(np.float64)

        mixed = np.concatenate([
            mixed_boundary_results[case["name"]]
            ["outputs"][block_index].reshape(-1)
            for case in cases
        ]).astype(np.float64) * scale

        integer = np.concatenate([
            integer_chain_results[case["name"]]
            ["blocks"][block_index]["output"].reshape(-1)
            for case in cases
        ]).astype(np.float64) * scale

        metrics = {
            "direct": error_metrics(direct, reference),
            "mixed_boundary": error_metrics(mixed, reference),
            "integer": error_metrics(integer, reference),
            "integer_vs_mixed": error_metrics(integer, mixed),
            "mixed_clips": sum(
                mixed_boundary_results[case["name"]]
                ["boundary_clips"][block_index]
                for case in cases
            ),
            "integer_clips": sum(
                integer_chain_results[case["name"]]
                ["blocks"][block_index]["saturations"]["block_boundary"]
                for case in cases
            ),
        }
        integer_chain_metrics[(group_name, block_index)] = metrics

        rows.append(
            f"| {group_name} | {block_index} | "
            f"{metrics['direct']['relative_l2_percent']:.4f}% | "
            f"{metrics['mixed_boundary']['relative_l2_percent']:.4f}% | "
            f"{metrics['integer']['relative_l2_percent']:.4f}% | "
            f"{metrics['integer']['rmse']:.6g} | "
            f"{metrics['mixed_clips']} / {metrics['integer_clips']} |"
        )

display(Markdown("\n".join(rows)))

print(
    "Initial input clipping:",
    sum(
        result["initial_input_clips"]
        for result in integer_chain_results.values()
    ),
)

| Cases | After block | Direct TFLite relative L2 | TFLite + INT8 boundaries relative L2 | Integer relative L2 | Integer RMSE | Boundary clips: TFLite / integer |
|---|---:|---:|---:|---:|---:|---|
| Benchmark | 0 | 1.5867% | 2.6121% | 2.6155% | 0.00310175 | 0 / 0 |
| Benchmark | 1 | 4.9660% | 4.9625% | 5.7633% | 0.00561712 | 0 / 0 |
| Benchmark | 2 | 5.6571% | 6.1704% | 6.6521% | 0.005575 | 0 / 0 |
| Benchmark | 3 | 6.6737% | 7.3480% | 7.7609% | 0.00568596 | 0 / 0 |
| Benchmark | 4 | 7.4423% | 8.3046% | 8.8714% | 0.00650304 | 0 / 0 |
| Benchmark | 5 | 8.2660% | 9.2021% | 9.6706% | 0.0066589 | 0 / 0 |
| Benchmark | 6 | 8.0716% | 9.0282% | 9.5275% | 0.00681119 | 0 / 0 |
| Benchmark | 7 | 12.5281% | 14.4581% | 15.6708% | 0.0112653 | 0 / 0 |
| Calibration | 0 | 1.5328% | 2.5653% | 2.5591% | 0.00303902 | 0 / 0 |
| Calibration | 1 | 4.8188% | 5.9591% | 5.8317% | 0.00565627 | 0 / 0 |
| Calibration | 2 | 5.6921% | 7.1847% | 7.1093% | 0.00594682 | 0 / 0 |
| Calibration | 3 | 6.6239% | 8.3514% | 8.5052% | 0.0062372 | 0 / 0 |
| Calibration | 4 | 7.3664% | 9.3294% | 9.4657% | 0.00702631 | 0 / 0 |
| Calibration | 5 | 8.0631% | 10.1583% | 10.3660% | 0.00721359 | 0 / 0 |
| Calibration | 6 | 7.7805% | 9.8271% | 10.0789% | 0.00738887 | 0 / 0 |
| Calibration | 7 | 11.4409% | 14.1973% | 14.6341% | 0.011368 | 0 / 0 |

Initial input clipping: 0


In [58]:
integer_prediction_results = {}
prediction_totals = {
    "direct": 0,
    "mixed_boundary": 0,
    "integer": 0,
}
position_count = 0

rows = [
    "| Case | Direct matches FP32 / 32 | "
    "TFLite + INT8 matches FP32 / 32 | Integer matches FP32 / 32 | "
    "Final-position IDs: FP32 / direct / TFLite+INT8 / integer |",
    "|---|---:|---:|---:|---|",
]

final_scale = block_boundary_specs[7]["output"]["scale"]

for case in reference_cases:
    name = case["name"]

    fp32_hidden = case["blocks"][7]["output_fp32"]
    direct_hidden = boundary_chain_results[name]["direct_outputs"][7]

    mixed_hidden = (
        mixed_boundary_results[name]["outputs"][7].astype(np.float64)
        * final_scale
    )
    integer_hidden = (
        integer_chain_results[name]["blocks"][7]["output"]
        .astype(np.float64) * final_scale
    )

    predictions = {
        "fp32": predict_from_final_hidden(fp32_hidden),
        "direct": predict_from_final_hidden(direct_hidden),
        "mixed_boundary": predict_from_final_hidden(mixed_hidden),
        "integer": predict_from_final_hidden(integer_hidden),
    }
    integer_prediction_results[name] = predictions

    matches = {
        path: int(np.count_nonzero(
            predictions[path] == predictions["fp32"]
        ))
        for path in prediction_totals
    }

    for path in prediction_totals:
        prediction_totals[path] += matches[path]
    position_count += predictions["fp32"].size

    last_ids = " / ".join(
        str(predictions[path][-1])
        for path in ("fp32", "direct", "mixed_boundary", "integer")
    )

    rows.append(
        f"| {name} | {matches['direct']}/32 | "
        f"{matches['mixed_boundary']}/32 | "
        f"{matches['integer']}/32 | {last_ids} |"
    )

display(Markdown("\n".join(rows)))

for path, matches in prediction_totals.items():
    print(
        f"{path}: {matches}/{position_count} positions agree with FP32 "
        f"({100.0 * matches / position_count:.2f}%)"
    )

| Case | Direct matches FP32 / 32 | TFLite + INT8 matches FP32 / 32 | Integer matches FP32 / 32 | Final-position IDs: FP32 / direct / TFLite+INT8 / integer |
|---|---:|---:|---:|---|
| benchmark | 30/32 | 31/32 | 29/32 | 1965 / 1965 / 1965 / 1965 |
| calibration_0 | 30/32 | 30/32 | 29/32 | 13 / 13 / 13 / 13 |
| calibration_1 | 30/32 | 28/32 | 29/32 | 2933 / 2933 / 2933 / 2576 |
| calibration_2 | 30/32 | 30/32 | 27/32 | 287 / 287 / 287 / 287 |
| calibration_3 | 27/32 | 26/32 | 26/32 | 673 / 262 / 262 / 262 |
| calibration_4 | 28/32 | 30/32 | 31/32 | 290 / 290 / 290 / 290 |
| calibration_5 | 32/32 | 28/32 | 29/32 | 351 / 351 / 351 / 351 |
| calibration_6 | 24/32 | 26/32 | 26/32 | 198 / 198 / 198 / 198 |

direct: 231/256 positions agree with FP32 (90.23%)
mixed_boundary: 229/256 positions agree with FP32 (89.45%)
integer: 226/256 positions agree with FP32 (88.28%)


In [61]:
import torch.nn.functional as torch_functional

loss_input_ids = {
    "benchmark": validation_inputs["benchmark_input_ids"].copy(),
    **{
        f"calibration_{i}": ids.reshape(1, 32).copy()
        for i, ids in enumerate(validation_inputs["calibration_input_ids"])
    },
}


def logits_from_final_hidden(hidden_states):
    hidden = torch.from_numpy(
        np.ascontiguousarray(hidden_states, dtype=np.float32)
    )

    with torch.no_grad():
        normalized = reference_model.transformer.ln_f(hidden)
        return reference_model.get_output_embeddings()(normalized)


# Confirm the captured FP32 hidden states reproduce full-model logits.
benchmark_case = next(
    case for case in reference_cases if case["name"] == "benchmark"
)

with torch.no_grad():
    full_model_logits = reference_model(
        input_ids=torch.as_tensor(
            loss_input_ids["benchmark"], dtype=torch.long
        ),
        use_cache=False,
        return_dict=True,
    ).logits

reconstructed_logits = logits_from_final_hidden(
    benchmark_case["blocks"][7]["output_fp32"]
)

np.testing.assert_allclose(
    reconstructed_logits.cpu().numpy(),
    full_model_logits.cpu().numpy(),
    atol=1e-5,
    rtol=1e-5,
)

print("PASS: captured FP32 states reproduce the full-model logits.")

PASS: captured FP32 states reproduce the full-model logits.


In [62]:
def next_token_losses(hidden_states, input_ids):
    ids = np.asarray(input_ids, dtype=np.int64)
    assert ids.shape == (1, 32)

    logits = logits_from_final_hidden(hidden_states)

    # Positions 0..30 predict the actual tokens at positions 1..31.
    prediction_logits = logits[:, :-1, :].double()
    targets = torch.from_numpy(ids[:, 1:].copy()).reshape(-1)

    losses = torch_functional.cross_entropy(
        prediction_logits.reshape(-1, prediction_logits.shape[-1]),
        targets,
        reduction="none",
    ).cpu().numpy().copy()

    assert losses.size == 31
    assert np.all(np.isfinite(losses))
    return losses


next_token_results = {}
final_scale = block_boundary_specs[7]["output"]["scale"]

for case in reference_cases:
    name = case["name"]

    hidden_states = {
        "fp32": case["blocks"][7]["output_fp32"],
        "direct": boundary_chain_results[name]["direct_outputs"][7],
        "mixed_boundary": (
            mixed_boundary_results[name]["outputs"][7]
            .astype(np.float64) * final_scale
        ),
        "integer": (
            integer_chain_results[name]["blocks"][7]["output"]
            .astype(np.float64) * final_scale
        ),
    }

    next_token_results[name] = {
        path: next_token_losses(hidden, loss_input_ids[name])
        for path, hidden in hidden_states.items()
    }

    print(f"Scored 31 next-token targets: {name}", flush=True)

Scored 31 next-token targets: benchmark
Scored 31 next-token targets: calibration_0
Scored 31 next-token targets: calibration_1
Scored 31 next-token targets: calibration_2
Scored 31 next-token targets: calibration_3
Scored 31 next-token targets: calibration_4
Scored 31 next-token targets: calibration_5
Scored 31 next-token targets: calibration_6


In [63]:
path_labels = {
    "fp32": "Original FP32",
    "direct": "Direct TFLite",
    "mixed_boundary": "TFLite + INT8 boundaries",
    "integer": "Integer blocks + INT8 boundaries",
}

loss_summary = {}

rows = [
    "| Cases | Path | Targets | Cross-entropy, nats/token | "
    "Perplexity | Perplexity / FP32 |",
    "|---|---|---:|---:|---:|---:|",
]

groups = [
    ("Benchmark", benchmark_cases),
    ("Calibration", calibration_cases),
    ("All", reference_cases),
]

for group_name, cases in groups:
    fp32_losses = np.concatenate([
        next_token_results[case["name"]]["fp32"]
        for case in cases
    ])
    fp32_mean = float(fp32_losses.mean())

    for path, label in path_labels.items():
        losses = np.concatenate([
            next_token_results[case["name"]][path]
            for case in cases
        ])

        mean_loss = float(losses.mean())
        perplexity = math.exp(mean_loss)
        relative_perplexity = math.exp(mean_loss - fp32_mean)

        loss_summary[(group_name, path)] = {
            "targets": losses.size,
            "cross_entropy": mean_loss,
            "perplexity": perplexity,
            "perplexity_ratio_vs_fp32": relative_perplexity,
        }

        rows.append(
            f"| {group_name} | {label} | {losses.size} | "
            f"{mean_loss:.6f} | {perplexity:.6f} | "
            f"{relative_perplexity:.6f} |"
        )

display(Markdown("\n".join(rows)))

| Cases | Path | Targets | Cross-entropy, nats/token | Perplexity | Perplexity / FP32 |
|---|---|---:|---:|---:|---:|
| Benchmark | Original FP32 | 31 | 2.876513 | 17.752270 | 1.000000 |
| Benchmark | Direct TFLite | 31 | 2.900814 | 18.188938 | 1.024598 |
| Benchmark | TFLite + INT8 boundaries | 31 | 2.928304 | 18.695890 | 1.053155 |
| Benchmark | Integer blocks + INT8 boundaries | 31 | 2.937672 | 18.871869 | 1.063068 |
| Calibration | Original FP32 | 217 | 4.022318 | 55.830351 | 1.000000 |
| Calibration | Direct TFLite | 217 | 4.059128 | 57.923794 | 1.037496 |
| Calibration | TFLite + INT8 boundaries | 217 | 4.035523 | 56.572476 | 1.013292 |
| Calibration | Integer blocks + INT8 boundaries | 217 | 4.050910 | 57.449735 | 1.029005 |
| All | Original FP32 | 248 | 3.879092 | 48.380271 | 1.000000 |
| All | Direct TFLite | 248 | 3.914339 | 50.115930 | 1.035875 |
| All | TFLite + INT8 boundaries | 248 | 3.897120 | 49.260385 | 1.018192 |
| All | Integer blocks + INT8 boundaries | 248 | 3.911756 | 49.986634 | 1.033203 |

In [64]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    FP32_DIR,
    local_files_only=True,
)
assert tokenizer.eos_token_id is not None
reference_model.eval()

GENERATION_PROMPT = "Once upon a time, a small fox"
MAX_NEW_TOKENS = 20

generation_prompt_ids = tokenizer.encode(
    GENERATION_PROMPT,
    add_special_tokens=False,
)

print("Prompt:", GENERATION_PROMPT)
print("Prompt tokens:", len(generation_prompt_ids))
print("Requested new tokens:", MAX_NEW_TOKENS)


def prepare_generation_input(token_ids):
    length = len(token_ids)
    if not 1 <= length <= 32:
        raise ValueError("The generation prefix must contain 1..32 tokens")

    padded_ids = np.full(
        (1, 32), tokenizer.eos_token_id, dtype=np.int64
    )
    padded_ids[0, :length] = token_ids

    ids = torch.from_numpy(padded_ids)
    positions = torch.arange(32, dtype=torch.long).unsqueeze(0)

    with torch.no_grad():
        embeddings = reference_model.transformer.drop(
            reference_model.transformer.wte(ids)
            + reference_model.transformer.wpe(positions)
        ).cpu().numpy().copy()

    return {
        "input_ids": padded_ids,
        "embeddings": embeddings,
        "length": length,
        "prediction_position": length - 1,
    }

Prompt: Once upon a time, a small fox
Prompt tokens: 8
Requested new tokens: 20


In [65]:
generation_path_labels = {
    "fp32": "Original FP32",
    "direct": "Direct TFLite",
    "mixed_boundary": "TFLite + INT8 boundaries",
    "integer": "Integer blocks + INT8 boundaries",
}


def generation_next_logits(path, frame):
    position = frame["prediction_position"]
    diagnostics = {}

    if path == "fp32":
        with torch.no_grad():
            logits = reference_model(
                input_ids=torch.from_numpy(frame["input_ids"]),
                position_ids=torch.arange(
                    32, dtype=torch.long
                ).unsqueeze(0),
                use_cache=False,
                return_dict=True,
            ).logits

        return logits[:, position, :], diagnostics

    if path == "direct":
        hidden = frame["embeddings"].copy()
        for block_index in range(8):
            hidden = run_tflite_block(block_index, hidden)

    elif path == "mixed_boundary":
        _, hidden, input_clips = encode_boundary(
            frame["embeddings"], 0
        )
        boundary_clips = []

        for block_index in range(8):
            raw = run_tflite_block(block_index, hidden)
            boundary = mlp_out_params[block_index]["boundary"]
            encoded = encode_reference_block_output(raw, boundary)

            boundary_clips.append(encoded["saturation_count"])
            hidden = (
                encoded["output"].astype(np.float64)
                * boundary["scale"]
            ).astype(np.float32)

        diagnostics = {
            "initial_input_clips": input_clips,
            "boundary_clips": boundary_clips,
        }

    elif path == "integer":
        codes, _, input_clips = encode_boundary(
            frame["embeddings"], 0
        )
        boundary_clips = []

        for block_index in range(8):
            result = integer_block_forward(codes, block_index)
            codes = result["output"]
            boundary_clips.append(
                result["saturations"]["block_boundary"]
            )

        diagnostics = {
            "initial_input_clips": input_clips,
            "boundary_clips": boundary_clips,
            "final_codes": codes.copy(),
        }

        hidden = (
            codes.astype(np.float64)
            * block_boundary_specs[7]["output"]["scale"]
        ).astype(np.float32)

    else:
        raise ValueError(f"Unknown generation path: {path}")

    logits = logits_from_final_hidden(hidden)
    return logits[:, position, :], diagnostics

In [66]:
benchmark_frame = prepare_generation_input(
    loss_input_ids["benchmark"].reshape(-1).tolist()
)

np.testing.assert_allclose(
    benchmark_frame["embeddings"],
    validation_inputs["block0_input_fp32"],
    atol=1e-6,
    rtol=1e-6,
)

_, benchmark_diagnostics = generation_next_logits(
    "integer", benchmark_frame
)

np.testing.assert_array_equal(
    benchmark_diagnostics["final_codes"],
    integer_chain_results["benchmark"]["blocks"][7]["output"],
)

print("PASS: generation wrapper reproduces the integer benchmark output.")

prompt_frame = prepare_generation_input(generation_prompt_ids)
padded_logits, _ = generation_next_logits("fp32", prompt_frame)

with torch.no_grad():
    prefix_logits = reference_model(
        input_ids=torch.tensor(
            [generation_prompt_ids], dtype=torch.long
        ),
        use_cache=False,
        return_dict=True,
    ).logits[:, -1, :]

np.testing.assert_allclose(
    padded_logits.cpu().numpy(),
    prefix_logits.cpu().numpy(),
    atol=1e-4,
    rtol=1e-4,
)

print("PASS: right-padding preserves the FP32 prefix prediction.")

PASS: generation wrapper reproduces the integer benchmark output.
PASS: right-padding preserves the FP32 prefix prediction.


In [67]:
def generate_with_path(path, prompt_ids, max_new_tokens=20):
    token_ids = list(prompt_ids)
    if not 1 <= len(token_ids) < 32:
        raise ValueError("Choose a prompt containing 1..31 tokens")

    budget = min(max_new_tokens, 32 - len(token_ids))
    steps = []
    stop_reason = (
        "context_limit" if budget < max_new_tokens else "token_limit"
    )

    for step_index in range(budget):
        frame = prepare_generation_input(token_ids)
        logits, diagnostics = generation_next_logits(path, frame)

        next_id = int(logits.argmax(dim=-1).item())
        top_two = torch.topk(logits, k=2, dim=-1)
        margin = float(
            (top_two.values[0, 0] - top_two.values[0, 1]).item()
        )

        token_ids.append(next_id)
        piece = tokenizer.decode(
            [next_id], clean_up_tokenization_spaces=False
        )

        steps.append({
            "token_id": next_id,
            "piece": piece,
            "top_two_logit_margin": margin,
            "diagnostics": diagnostics,
        })

        print(
            f"{path}, step {step_index + 1:02d}: {piece!r}",
            flush=True,
        )

        if next_id == tokenizer.eos_token_id:
            stop_reason = "eos"
            break

    return {
        "token_ids": token_ids,
        "generated_token_ids": token_ids[len(prompt_ids):],
        "steps": steps,
        "stop_reason": stop_reason,
        "text": tokenizer.decode(
            token_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        ),
    }


generation_results = {}

for path, label in generation_path_labels.items():
    print(f"\nRunning: {label}", flush=True)
    generation_results[path] = generate_with_path(
        path,
        generation_prompt_ids,
        max_new_tokens=MAX_NEW_TOKENS,
    )

for path, label in generation_path_labels.items():
    result = generation_results[path]
    print(f"\n{label}")
    print(result["text"])
    print(
        f"Generated {len(result['generated_token_ids'])} tokens; "
        f"stopped: {result['stop_reason']}"
    )


Running: Original FP32
fp32, step 01: ' lived'
fp32, step 02: ' in'
fp32, step 03: ' a'
fp32, step 04: ' big'
fp32, step 05: ' forest'
fp32, step 06: '.'
fp32, step 07: ' The'
fp32, step 08: ' fox'
fp32, step 09: ' was'
fp32, step 10: ' very'
fp32, step 11: ' happy'
fp32, step 12: ' and'
fp32, step 13: ' loved'
fp32, step 14: ' to'
fp32, step 15: ' play'
fp32, step 16: ' with'
fp32, step 17: ' his'
fp32, step 18: ' friends'
fp32, step 19: '.'
fp32, step 20: ' One'

Running: Direct TFLite
direct, step 01: ' lived'
direct, step 02: ' in'
direct, step 03: ' a'
direct, step 04: ' big'
direct, step 05: ' forest'
direct, step 06: '.'
direct, step 07: ' He'
direct, step 08: ' was'
direct, step 09: ' very'
direct, step 10: ' happy'
direct, step 11: ' and'
direct, step 12: ' loved'
direct, step 13: ' to'
direct, step 14: ' play'
direct, step 15: ' with'
direct, step 16: ' his'
direct, step 17: ' friends'
direct, step 18: '.'
direct, step 19: ' One'
direct, step 20: ' day'

Running: TFLite + IN

In [68]:
additional_prompts = [
    "A little girl found a tiny bird",
    "Tom wanted to help his friend",
    "The rabbit opened the old box",
]

additional_generation_results = {}

for prompt in additional_prompts:
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
    results = {}

    print(f"\nPROMPT: {prompt}", flush=True)

    for path, label in generation_path_labels.items():
        print(f"\nRunning: {label}", flush=True)
        results[path] = generate_with_path(
            path, prompt_ids, max_new_tokens=20
        )

    additional_generation_results[prompt] = results

    for path, label in generation_path_labels.items():
        print(f"\n{label}:")
        print(results[path]["text"])


PROMPT: A little girl found a tiny bird

Running: Original FP32
fp32, step 01: ' that'
fp32, step 02: ' was'
fp32, step 03: ' stuck'
fp32, step 04: ' in'
fp32, step 05: ' a'
fp32, step 06: ' tree'
fp32, step 07: '.'
fp32, step 08: ' The'
fp32, step 09: ' bird'
fp32, step 10: ' was'
fp32, step 11: ' stuck'
fp32, step 12: ' in'
fp32, step 13: ' a'
fp32, step 14: ' tree'
fp32, step 15: ' and'
fp32, step 16: ' couldn'
fp32, step 17: "'t"
fp32, step 18: ' get'
fp32, step 19: ' out'
fp32, step 20: '.'

Running: Direct TFLite
direct, step 01: ' that'
direct, step 02: ' was'
direct, step 03: ' stuck'
direct, step 04: ' in'
direct, step 05: ' a'
direct, step 06: ' tree'
direct, step 07: '.'
direct, step 08: ' She'
direct, step 09: ' tried'
direct, step 10: ' to'
direct, step 11: ' pull'
direct, step 12: ' it'
direct, step 13: ' out'
direct, step 14: ','
direct, step 15: ' but'
direct, step 16: ' it'
direct, step 17: ' was'
direct, step 18: ' too'
direct, step 19: ' hard'
direct, step 20: '.'



In [69]:
full_window_prompt = (
    "On a rainy afternoon, a little hedgehog found a red umbrella "
    "beside the garden gate. He wanted to return it to its owner, "
    "so he asked a friendly duck to help him search."
)

full_window_ids = tokenizer.encode(
    full_window_prompt,
    add_special_tokens=False,
)
assert len(full_window_ids) >= 32

full_window_ids = full_window_ids[:32]
assert len(full_window_ids) == 32

print("Actual 32-token prompt:")
print(tokenizer.decode(
    full_window_ids,
    clean_up_tokenization_spaces=False,
))
print("Input token count:", len(full_window_ids))

Actual 32-token prompt:
On a rainy afternoon, a little hedgehog found a red umbrella beside the garden gate. He wanted to return it to its owner, so he asked a friendly
Input token count: 32


In [70]:
def generate_with_32_token_window(path, prompt_ids, max_new_tokens=20):
    if len(prompt_ids) != 32:
        raise ValueError("This test requires exactly 32 prompt tokens")

    token_ids = list(prompt_ids)
    steps = []
    stop_reason = "token_limit"

    for step_index in range(max_new_tokens):
        context_ids = token_ids[-32:]
        frame = prepare_generation_input(context_ids)
        assert frame["length"] == 32

        logits, diagnostics = generation_next_logits(path, frame)
        next_id = int(logits.argmax(dim=-1).item())

        top_two = torch.topk(logits, k=2, dim=-1)
        margin = float(
            (top_two.values[0, 0] - top_two.values[0, 1]).item()
        )

        piece = tokenizer.decode(
            [next_id], clean_up_tokenization_spaces=False
        )
        steps.append({
            "context_ids": context_ids.copy(),
            "token_id": next_id,
            "piece": piece,
            "top_two_logit_margin": margin,
            "diagnostics": diagnostics,
        })
        token_ids.append(next_id)

        print(
            f"{path}, step {step_index + 1:02d}, "
            f"context=32: {piece!r}",
            flush=True,
        )

        if next_id == tokenizer.eos_token_id:
            stop_reason = "eos"
            break

    return {
        "token_ids": token_ids,
        "generated_token_ids": token_ids[32:],
        "steps": steps,
        "stop_reason": stop_reason,
        "text": tokenizer.decode(
            token_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        ),
    }

In [71]:
full_window_results = {}

for path, label in generation_path_labels.items():
    print(f"\nRunning: {label}", flush=True)
    full_window_results[path] = generate_with_32_token_window(
        path, full_window_ids, max_new_tokens=20
    )

rows = [
    "| Path | Generated tokens | Stop reason | "
    "Input clip events | Boundary clip events |",
    "|---|---:|---|---:|---:|",
]

for path, label in generation_path_labels.items():
    result = full_window_results[path]

    if path in ("mixed_boundary", "integer"):
        input_clips = sum(
            step["diagnostics"]["initial_input_clips"]
            for step in result["steps"]
        )
        boundary_clips = sum(
            sum(step["diagnostics"]["boundary_clips"])
            for step in result["steps"]
        )
    else:
        input_clips = "—"
        boundary_clips = "—"

    rows.append(
        f"| {label} | {len(result['generated_token_ids'])} | "
        f"{result['stop_reason']} | {input_clips} | {boundary_clips} |"
    )

    print(f"\n{label}:")
    print(result["text"])

display(Markdown("\n".join(rows)))


Running: Original FP32
fp32, step 01, context=32: ' voice'
fp32, step 02, context=32: ','
fp32, step 03, context=32: ' "'
fp32, step 04, context=32: 'Please'
fp32, step 05, context=32: ','
fp32, step 06, context=32: ' can'
fp32, step 07, context=32: ' I'
fp32, step 08, context=32: ' return'
fp32, step 09, context=32: ' it'
fp32, step 10, context=32: '?"'
fp32, step 11, context=32: '\n'
fp32, step 12, context=32: '\n'
fp32, step 13, context=32: 'The'
fp32, step 14, context=32: ' voice'
fp32, step 15, context=32: ' replied'
fp32, step 16, context=32: ','
fp32, step 17, context=32: ' "'
fp32, step 18, context=32: 'Yes'
fp32, step 19, context=32: ','
fp32, step 20, context=32: ' you'

Running: Direct TFLite
direct, step 01, context=32: ' voice'
direct, step 02, context=32: ','
direct, step 03, context=32: ' "'
direct, step 04, context=32: 'Please'
direct, step 05, context=32: ','
direct, step 06, context=32: ' please'
direct, step 07, context=32: ','
direct, step 08, context=32: ' please'

| Path | Generated tokens | Stop reason | Input clip events | Boundary clip events |
|---|---:|---|---:|---:|
| Original FP32 | 20 | token_limit | — | — |
| Direct TFLite | 20 | token_limit | — | — |
| TFLite + INT8 boundaries | 20 | token_limit | 0 | 0 |
| Integer blocks + INT8 boundaries | 20 | token_limit | 0 | 0 |